In [1]:
# Batch 0 / Cell 2 - Setup, paths, and helpers
from __future__ import annotations

import json
import math
import os
import platform
import shutil
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

NOTEBOOK_ID = "27_multi_model_comparison"
NOTEBOOK_TITLE = "Multi-Model Comparison"

STRICT_METRIC_COLUMNS = [
    "mse_improvement",
    "mae_improvement",
    "psnr_improvement",
    "ssim_improvement",
    "lpips_improvement",
    "clip_similarity_improvement",
    "dinov2_similarity_improvement",
    "mean_similarity_improvement",
]

METRIC_FAMILY_BY_COLUMN = {
    "mse_improvement": "classical",
    "mae_improvement": "classical",
    "psnr_improvement": "classical",
    "ssim_improvement": "classical",
    "lpips_improvement": "lpips",
    "clip_similarity_improvement": "feature_similarity",
    "dinov2_similarity_improvement": "feature_similarity",
    "mean_similarity_improvement": "feature_similarity",
}

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "outputs").exists() and (
            (candidate / "tools" / "build_project_inventory.py").exists()
            or (candidate / "outputs" / "inventory" / "project_file_inventory.csv").exists()
        ):
            return candidate
    raise RuntimeError("Could not locate project root. Run from inside the thesis repository.")

PROJECT_ROOT = find_project_root()
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / NOTEBOOK_ID

INVENTORY_DIR = OUTPUT_ROOT / "inventory"
VALIDATION_DIR = OUTPUT_ROOT / "validation"
METRICS_DIR = OUTPUT_ROOT / "metrics"
ANALYSIS_DIR = OUTPUT_ROOT / "analysis"
FIGURES_DIR = OUTPUT_ROOT / "figures"
REPORTS_DIR = OUTPUT_ROOT / "reports"
MANIFESTS_DIR = OUTPUT_ROOT / "manifests"

for directory in [
    INVENTORY_DIR,
    VALIDATION_DIR,
    METRICS_DIR,
    ANALYSIS_DIR,
    FIGURES_DIR,
    REPORTS_DIR,
    MANIFESTS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

PROJECT_INVENTORY_PATH = PROJECT_ROOT / "outputs" / "inventory" / "project_file_inventory.csv"
PROJECT_INVENTORY_MANIFEST_PATH = PROJECT_ROOT / "outputs" / "inventory" / "project_file_inventory_manifest.json"

BATCH0_INVENTORY_SNAPSHOT_PATH = INVENTORY_DIR / "project_inventory_snapshot.csv"
BATCH0_SOURCE_PLAN_PATH = VALIDATION_DIR / "batch0_source_plan.csv"
BATCH0_VALIDATION_PATH = VALIDATION_DIR / "batch0_validation.csv"
STAGE_MANIFEST_PATH = MANIFESTS_DIR / "multi_model_comparison_stage_manifest.json"

def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="seconds")

def clean_text(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, float) and math.isnan(value):
        return ""
    return str(value).strip()

def rel(path: Path | str) -> str:
    path = Path(path)
    try:
        return path.resolve().relative_to(PROJECT_ROOT).as_posix()
    except Exception:
        return path.as_posix().replace("\\", "/")

def norm_rel_path(value: Any) -> str:
    return clean_text(value).replace("\\", "/").lstrip("./")

def read_json_if_exists(path: Path) -> dict:
    if path.is_file():
        with path.open("r", encoding="utf-8") as handle:
            return json.load(handle)
    return {}

def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2, ensure_ascii=False)

def csv_shape(path: Path) -> tuple[int | None, int | None, str]:
    if not path.is_file():
        return None, None, "file_missing"
    try:
        sample_df = pd.read_csv(path, nrows=5)
        row_count = sum(1 for _ in path.open("r", encoding="utf-8", errors="replace")) - 1
        return max(row_count, 0), int(sample_df.shape[1]), ""
    except Exception as exc:
        return None, None, repr(exc)

def validation_row(check_name: str, actual: Any, expected: Any, passed: bool, failure_message: str) -> dict:
    return {
        "check_name": check_name,
        "actual": actual,
        "expected": expected,
        "passed": bool(passed),
        "failure_message": "" if passed else failure_message,
    }

print(f"Project root: {PROJECT_ROOT}")
print(f"Notebook output root: {rel(OUTPUT_ROOT)}")
print(f"Strict metric columns: {STRICT_METRIC_COLUMNS}")

Project root: D:\Masters\FH\Thesis\painting-restoration-eval
Notebook output root: outputs/27_multi_model_comparison
Strict metric columns: ['mse_improvement', 'mae_improvement', 'psnr_improvement', 'ssim_improvement', 'lpips_improvement', 'clip_similarity_improvement', 'dinov2_similarity_improvement', 'mean_similarity_improvement']


In [2]:
# Batch 0 / Cell 3 - Load inventory and save notebook-local snapshot
if not PROJECT_INVENTORY_PATH.is_file():
    raise RuntimeError(
        "Project inventory is missing. Run this first from the repository root:\n"
        "python .\\tools\\build_project_inventory.py --root . --out-dir .\\outputs\\inventory"
    )

project_inventory_df = pd.read_csv(PROJECT_INVENTORY_PATH)
project_inventory_df.columns = [clean_text(column) for column in project_inventory_df.columns]

if "relative_path" not in project_inventory_df.columns:
    raise RuntimeError("Inventory must contain a relative_path column.")

project_inventory_df["relative_path"] = project_inventory_df["relative_path"].map(norm_rel_path)
project_inventory_df = project_inventory_df.sort_values("relative_path", kind="stable").reset_index(drop=True)

project_inventory_df.to_csv(BATCH0_INVENTORY_SNAPSHOT_PATH, index=False)

inventory_manifest = read_json_if_exists(PROJECT_INVENTORY_MANIFEST_PATH)

print(f"Loaded inventory rows: {len(project_inventory_df):,}")
print(f"Inventory columns: {len(project_inventory_df.columns):,}")
print(f"Saved notebook inventory snapshot: {rel(BATCH0_INVENTORY_SNAPSHOT_PATH)}")

display(project_inventory_df.head(20))

Loaded inventory rows: 10,939
Inventory columns: 17
Saved notebook inventory snapshot: outputs/27_multi_model_comparison/inventory/project_inventory_snapshot.csv


,relative_path,file_name,parent_dir,extension,file_kind,size_bytes,last_modified_iso,depth,csv_row_count,csv_column_count,csv_columns,csv_error,image_width,image_height,image_mode,image_error,sha256_first_1mb
0,README.md,README.md,NaN,.md,md,33691,2026-07-22T20:05:56.952035+00:00,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,aafb59b7555699c61c5b72deae401fb55466ece8872236...
1,config/experiment_50_config.yaml,experiment_50_config.yaml,config,.yaml,other,7111,2026-07-23T12:34:35.693189+00:00,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7af22deb75434907302e3e460b4c28a73f25533f3274e4...
2,config/pilot_config.yaml,pilot_config.yaml,config,.yaml,other,677,2026-06-29T14:26:59.322671+00:00,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9c81e4ee3aec9dc1b86aa6bc28f5ba442b694810d83eed...
3,data/model_audit/model_candidates.csv,model_candidates.csv,data/model_audit,.csv,csv,5383,2026-07-03T09:14:19.349960+00:00,2,5.0,17.0,model_name | model_family | open_or_closed | d...,NaN,NaN,NaN,NaN,NaN,4ec7ee8d89e5a9d122ea5e9dd37efa6346a6f686263d66...
4,data/processed/clean/p001_clean.png,p001_clean.png,data/processed/clean,.png,clean_image,723881,2026-07-24T16:38:17.470158+00:00,3,NaN,NaN,NaN,NaN,768.0,768.0,RGB,NaN,b5852f0337c8342a1a70268f9e9f682534177581d23871...
5,data/processed/clean/p002_clean.png,p002_clean.png,data/processed/clean,.png,clean_image,339472,2026-07-24T16:38:18.292719+00:00,3,NaN,NaN,NaN,NaN,768.0,768.0,RGB,NaN,3f173a947e4c2be2ebdc267f0c852f490981bc81da6c4e...
6,data/processed/clean/p003_clean.png,p003_clean.png,data/processed/clean,.png,clean_image,648764,2026-07-24T16:38:19.476394+00:00,3,NaN,NaN,NaN,NaN,768.0,768.0,RGB,NaN,1e2ef6b59b4383d7d44ce2f9bb016c126633d2c3e4f39b...
7,data/processed/clean/p004_clean.png,p004_clean.png,data/processed/clean,.png,clean_image,709746,2026-07-24T16:38:20.522805+00:00,3,NaN,NaN,NaN,NaN,768.0,768.0,RGB,NaN,ae8cfa01d84bd2e8c2f41cfdfe1cb25b6f2abf34301351...
8,data/processed/clean/p005_clean.png,p005_clean.png,data/processed/clean,.png,clean_image,745221,2026-07-24T16:38:21.522086+00:00,3,NaN,NaN,NaN,NaN,768.0,768.0,RGB,NaN,da4c53324f40a8a662edb554ee6215db2a53bcb94fcdb7...
9,data/processed/clean/p006_clean.png,p006_clean.png,data/processed/clean,.png,clean_image,458402,2026-07-24T16:38:22.648624+00:00,3,NaN,NaN,NaN,NaN,768.0,768.0,RGB,NaN,fabead74b973d925a3692b0c10f8c1ddb90e140172f620...


In [3]:
# Batch 0 / Cell 4 - Define and write source plan
inventory_by_path = {
    norm_rel_path(row["relative_path"]): row
    for _, row in project_inventory_df.iterrows()
}

def inventory_row_for(path_value: str) -> pd.Series | None:
    return inventory_by_path.get(norm_rel_path(path_value))

def choose_first_available(candidate_paths: list[str]) -> tuple[str, str]:
    normalized = [norm_rel_path(path) for path in candidate_paths if clean_text(path)]
    for candidate in normalized:
        if candidate in inventory_by_path:
            return candidate, "present_in_inventory"
    for candidate in normalized:
        if (PROJECT_ROOT / candidate).exists():
            return candidate, "exists_on_disk"
    return normalized[0] if normalized else "", "missing"

def columns_from_inventory_row(row: pd.Series | None) -> list[str]:
    if row is None:
        return []
    raw_columns = clean_text(row.get("csv_columns", ""))
    if not raw_columns:
        return []
    try:
        parsed = json.loads(raw_columns)
        if isinstance(parsed, list):
            return [clean_text(column) for column in parsed]
    except Exception:
        pass
    return [part.strip().strip("'\"") for part in raw_columns.strip("[]").split(",") if part.strip()]

source_specs = [
    {
        "source_id": "opencv_classical_metrics",
        "model_name": "opencv_telea",
        "source_type": "metric",
        "metric_family": "classical",
        "required": True,
        "candidate_paths": ["data/processed/metrics/metrics_opencv_telea_classical.csv"],
        "required_columns": ["case_id", "model_name", "evaluation_region", "mse_improvement", "mae_improvement", "psnr_improvement", "ssim_improvement", "status"],
        "notes": "OpenCV classical paired metric source.",
    },
    {
        "source_id": "opencv_lpips_metrics",
        "model_name": "opencv_telea",
        "source_type": "metric",
        "metric_family": "lpips",
        "required": True,
        "candidate_paths": ["data/processed/metrics/metrics_opencv_telea_lpips.csv"],
        "required_columns": ["case_id", "model_name", "evaluation_region", "lpips_improvement", "status"],
        "notes": "OpenCV LPIPS metric source.",
    },
    {
        "source_id": "opencv_feature_metrics",
        "model_name": "opencv_telea",
        "source_type": "metric",
        "metric_family": "feature_similarity",
        "required": True,
        "candidate_paths": ["data/processed/metrics/metrics_opencv_telea_feature_similarity.csv"],
        "required_columns": ["case_id", "model_name", "evaluation_region", "clip_similarity_improvement", "dinov2_similarity_improvement", "mean_similarity_improvement", "status"],
        "notes": "OpenCV CLIP/DINO feature metric source.",
    },
    {
        "source_id": "opencv_report_cases",
        "model_name": "opencv_telea",
        "source_type": "report_visual",
        "metric_family": "report",
        "required": True,
        "candidate_paths": [
            "outputs/reports/opencv_report_assets/opencv_telea_report_case_metrics.csv",
            "outputs/13_opencv_report_generation/metrics/opencv_telea_report_case_metrics.csv",
        ],
        "required_columns": ["case_id"],
        "notes": "OpenCV report-facing paths/status source. Used for visuals and case metadata where available.",
    },
    {
        "source_id": "lama_classical_metrics",
        "model_name": "lama",
        "source_type": "metric",
        "metric_family": "classical",
        "required": True,
        "candidate_paths": ["outputs/metrics/classical_metrics_lama.csv"],
        "required_columns": ["case_id", "model_name", "evaluation_region", "mse_improvement", "mae_improvement", "psnr_improvement", "ssim_improvement", "status"],
        "notes": "LaMa classical paired metric source.",
    },
    {
        "source_id": "lama_lpips_metrics",
        "model_name": "lama",
        "source_type": "metric",
        "metric_family": "lpips",
        "required": True,
        "candidate_paths": ["outputs/metrics/lpips_metrics_lama.csv"],
        "required_columns": ["case_id", "model_name", "evaluation_region", "lpips_improvement", "status"],
        "notes": "LaMa LPIPS metric source.",
    },
    {
        "source_id": "lama_feature_metrics",
        "model_name": "lama",
        "source_type": "metric",
        "metric_family": "feature_similarity",
        "required": True,
        "candidate_paths": ["outputs/metrics/feature_similarity_metrics_lama.csv"],
        "required_columns": ["case_id", "model_name", "evaluation_region", "clip_similarity_improvement", "dinov2_similarity_improvement", "mean_similarity_improvement", "status"],
        "notes": "LaMa CLIP/DINO feature metric source.",
    },
    {
        "source_id": "lama_report_cases",
        "model_name": "lama",
        "source_type": "report_visual",
        "metric_family": "report",
        "required": True,
        "candidate_paths": [
            "outputs/19_lama_report_generation/metrics/lama_report_dataframe.csv",
            "outputs/19_lama_report_generation/metrics/lama_report_case_metrics.csv",
        ],
        "required_columns": ["case_id"],
        "notes": "LaMa report-facing paths/status source.",
    },
    {
        "source_id": "stable_diffusion_classical_metrics",
        "model_name": "stable_diffusion",
        "source_type": "metric",
        "metric_family": "classical",
        "required": True,
        "candidate_paths": ["outputs/22_stable_diffusion_classical_metrics/metrics/stable_diffusion_classical_metrics.csv"],
        "required_columns": ["case_id", "evaluation_region", "mse_improvement", "mae_improvement", "psnr_improvement", "ssim_improvement", "status"],
        "notes": "Stable Diffusion classical metrics. Later batches must choose a fair primary candidate per case.",
    },
    {
        "source_id": "stable_diffusion_lpips_metrics",
        "model_name": "stable_diffusion",
        "source_type": "metric",
        "metric_family": "lpips",
        "required": True,
        "candidate_paths": ["outputs/24_stable_diffusion_lpips_metrics/metrics/stable_diffusion_lpips_metrics.csv"],
        "required_columns": ["case_id", "evaluation_region", "lpips_improvement", "status"],
        "notes": "Stable Diffusion LPIPS metrics.",
    },
    {
        "source_id": "stable_diffusion_feature_metrics",
        "model_name": "stable_diffusion",
        "source_type": "metric",
        "metric_family": "feature_similarity",
        "required": True,
        "candidate_paths": ["outputs/25_stable_diffusion_feature_similarity/metrics/stable_diffusion_feature_similarity_metrics.csv"],
        "required_columns": ["case_id", "evaluation_region", "clip_similarity_improvement", "dinov2_similarity_improvement", "mean_similarity_improvement", "status"],
        "notes": "Stable Diffusion CLIP/DINO feature metrics.",
    },
    {
        "source_id": "stable_diffusion_report_audit",
        "model_name": "stable_diffusion",
        "source_type": "report_visual",
        "metric_family": "report",
        "required": True,
        "candidate_paths": ["outputs/26_stable_diffusion_report_generation/metrics/stable_diffusion_report_audit.csv"],
        "required_columns": ["report_candidate_id"],
        "notes": "Stable Diffusion report-facing audit. Do not auto-treat all numeric columns as metrics.",
    },
    {
        "source_id": "sdxl_optional_metrics_or_report",
        "model_name": "sdxl",
        "source_type": "optional_scan",
        "metric_family": "optional",
        "required": False,
        "candidate_paths": [
            "outputs/metrics/classical_metrics_sdxl.csv",
            "outputs/metrics/lpips_metrics_sdxl.csv",
            "outputs/metrics/feature_similarity_metrics_sdxl.csv",
            "outputs/26_sdxl_report_generation/metrics/sdxl_report_audit.csv",
        ],
        "required_columns": [],
        "notes": "Optional. If unavailable, Notebook 27 records SDXL as unavailable rather than failing.",
    },
]

source_plan_rows = []

for spec in source_specs:
    resolved_relative_path, availability_status = choose_first_available(spec["candidate_paths"])
    row = inventory_row_for(resolved_relative_path)
    inventory_columns = columns_from_inventory_row(row)
    missing_required_columns = [
        column for column in spec["required_columns"]
        if column not in inventory_columns
    ]

    file_exists = bool((PROJECT_ROOT / resolved_relative_path).exists()) if resolved_relative_path else False
    present_in_inventory = row is not None

    source_plan_rows.append(
        {
            "source_id": spec["source_id"],
            "model_name": spec["model_name"],
            "source_type": spec["source_type"],
            "metric_family": spec["metric_family"],
            "required": bool(spec["required"]),
            "resolved_relative_path": resolved_relative_path,
            "availability_status": availability_status,
            "present_in_inventory": bool(present_in_inventory),
            "file_exists": bool(file_exists),
            "inventory_csv_rows": row.get("csv_row_count", np.nan) if row is not None else np.nan,
            "inventory_csv_columns": row.get("csv_column_count", np.nan) if row is not None else np.nan,
            "inventory_csv_error": row.get("csv_read_error", "") if row is not None else "",
            "required_columns": json.dumps(spec["required_columns"]),
            "missing_required_columns_from_inventory": json.dumps(missing_required_columns),
            "candidate_paths": json.dumps([norm_rel_path(path) for path in spec["candidate_paths"]]),
            "strict_metric_columns_allowed": json.dumps([
                column for column in STRICT_METRIC_COLUMNS
                if METRIC_FAMILY_BY_COLUMN.get(column) == spec["metric_family"]
            ]),
            "notes": spec["notes"],
        }
    )

batch0_source_plan_df = pd.DataFrame(source_plan_rows)
batch0_source_plan_df.to_csv(BATCH0_SOURCE_PLAN_PATH, index=False)

required_source_df = batch0_source_plan_df.loc[batch0_source_plan_df["required"]].copy()
missing_required_source_df = required_source_df.loc[
    ~required_source_df["present_in_inventory"].astype(bool)
].copy()

print(f"Saved source plan: {rel(BATCH0_SOURCE_PLAN_PATH)}")
print(f"Required sources: {len(required_source_df):,}")
print(f"Missing required sources in inventory: {len(missing_required_source_df):,}")

display(batch0_source_plan_df)

Saved source plan: outputs/27_multi_model_comparison/validation/batch0_source_plan.csv
Required sources: 12
Missing required sources in inventory: 0


,source_id,model_name,source_type,metric_family,required,resolved_relative_path,availability_status,present_in_inventory,file_exists,inventory_csv_rows,inventory_csv_columns,inventory_csv_error,required_columns,missing_required_columns_from_inventory,candidate_paths,strict_metric_columns_allowed,notes
0,opencv_classical_metrics,opencv_telea,metric,classical,True,data/processed/metrics/metrics_opencv_telea_cl...,present_in_inventory,True,True,2295.0,48.0,,"[""case_id"", ""model_name"", ""evaluation_region"",...","[""case_id"", ""model_name"", ""evaluation_region"",...","[""data/processed/metrics/metrics_opencv_telea_...","[""mse_improvement"", ""mae_improvement"", ""psnr_i...",OpenCV classical paired metric source.
1,opencv_lpips_metrics,opencv_telea,metric,lpips,True,data/processed/metrics/metrics_opencv_telea_lp...,present_in_inventory,True,True,1175.0,47.0,,"[""case_id"", ""model_name"", ""evaluation_region"",...","[""case_id"", ""model_name"", ""evaluation_region"",...","[""data/processed/metrics/metrics_opencv_telea_...","[""lpips_improvement""]",OpenCV LPIPS metric source.
2,opencv_feature_metrics,opencv_telea,metric,feature_similarity,True,data/processed/metrics/metrics_opencv_telea_fe...,present_in_inventory,True,True,1175.0,59.0,,"[""case_id"", ""model_name"", ""evaluation_region"",...","[""case_id"", ""model_name"", ""evaluation_region"",...","[""data/processed/metrics/metrics_opencv_telea_...","[""clip_similarity_improvement"", ""dinov2_simila...",OpenCV CLIP/DINO feature metric source.
3,opencv_report_cases,opencv_telea,report_visual,report,True,outputs/reports/opencv_report_assets/opencv_te...,present_in_inventory,True,True,410.0,69.0,,"[""case_id""]","[""case_id""]","[""outputs/reports/opencv_report_assets/opencv_...",[],OpenCV report-facing paths/status source. Used...
4,lama_classical_metrics,lama,metric,classical,True,outputs/metrics/classical_metrics_lama.csv,present_in_inventory,True,True,2260.0,51.0,,"[""case_id"", ""model_name"", ""evaluation_region"",...","[""case_id"", ""model_name"", ""evaluation_region"",...","[""outputs/metrics/classical_metrics_lama.csv""]","[""mse_improvement"", ""mae_improvement"", ""psnr_i...",LaMa classical paired metric source.
5,lama_lpips_metrics,lama,metric,lpips,True,outputs/metrics/lpips_metrics_lama.csv,present_in_inventory,True,True,1175.0,47.0,,"[""case_id"", ""model_name"", ""evaluation_region"",...","[""case_id"", ""model_name"", ""evaluation_region"",...","[""outputs/metrics/lpips_metrics_lama.csv""]","[""lpips_improvement""]",LaMa LPIPS metric source.
6,lama_feature_metrics,lama,metric,feature_similarity,True,outputs/metrics/feature_similarity_metrics_lam...,present_in_inventory,True,True,1175.0,59.0,,"[""case_id"", ""model_name"", ""evaluation_region"",...","[""case_id"", ""model_name"", ""evaluation_region"",...","[""outputs/metrics/feature_similarity_metrics_l...","[""clip_similarity_improvement"", ""dinov2_simila...",LaMa CLIP/DINO feature metric source.
7,lama_report_cases,lama,report_visual,report,True,outputs/19_lama_report_generation/metrics/lama...,present_in_inventory,True,True,355.0,264.0,,"[""case_id""]","[""case_id""]","[""outputs/19_lama_report_generation/metrics/la...",[],LaMa report-facing paths/status source.
8,stable_diffusion_classical_metrics,stable_diffusion,metric,classical,True,outputs/22_stable_diffusion_classical_metrics/...,present_in_inventory,True,True,5294.0,58.0,,"[""case_id"", ""evaluation_region"", ""mse_improvem...","[""case_id"", ""evaluation_region"", ""mse_improvem...","[""outputs/22_stable_diffusion_classical_metric...","[""mse_improvement"", ""mae_improvement"", ""psnr_i...",Stable Diffusion classical metrics. Later batc...
9,stable_diffusion_lpips_metrics,stable_diffusion,metric,lpips,True,outputs/24_stable_diffusion_lpips_metrics/metr...,present_in_inventory,True,True,2724.0,66.0,,"[""case_id"", ""evaluation_region"", ""lpips_improv...","[""case_id"", ""evaluation_region"", ""lpips_improv...","[""outputs/24_stable_diffusion_lpip

In [4]:
# Batch 0 / Cell 5 - Batch 0 validation and stage manifest
snapshot_rows, snapshot_columns, snapshot_error = csv_shape(BATCH0_INVENTORY_SNAPSHOT_PATH)
source_plan_rows, source_plan_columns, source_plan_error = csv_shape(BATCH0_SOURCE_PLAN_PATH)

required_source_df = batch0_source_plan_df.loc[batch0_source_plan_df["required"]].copy()
available_required_count = int(required_source_df["present_in_inventory"].astype(bool).sum())
missing_required_count = int((~required_source_df["present_in_inventory"].astype(bool)).sum())

strict_metric_pollution_tokens = [
    "rank__",
    "file_size",
    "filename_length",
    "image_width",
    "image_height",
    "is_zero_control",
    "prompt_ablation_subset",
    "lpips_input_size",
]

allowed_metric_set = set(STRICT_METRIC_COLUMNS)
polluted_allowed_metrics = [
    column for column in STRICT_METRIC_COLUMNS
    if any(token in column for token in strict_metric_pollution_tokens)
]

batch0_validation_rows = [
    validation_row(
        "project_inventory_loaded",
        len(project_inventory_df),
        "> 0 rows",
        len(project_inventory_df) > 0,
        "Project inventory did not load any rows.",
    ),
    validation_row(
        "inventory_snapshot_written",
        rel(BATCH0_INVENTORY_SNAPSHOT_PATH),
        "file exists",
        BATCH0_INVENTORY_SNAPSHOT_PATH.is_file(),
        "Notebook-local inventory snapshot was not written.",
    ),
    validation_row(
        "inventory_snapshot_has_rows",
        snapshot_rows,
        "> 0 rows",
        bool(snapshot_rows and snapshot_rows > 0),
        f"Inventory snapshot is empty or unreadable: {snapshot_error}",
    ),
    validation_row(
        "source_plan_written",
        rel(BATCH0_SOURCE_PLAN_PATH),
        "file exists",
        BATCH0_SOURCE_PLAN_PATH.is_file(),
        "Batch 0 source plan was not written.",
    ),
    validation_row(
        "source_plan_has_rows",
        source_plan_rows,
        ">= 1 row",
        bool(source_plan_rows and source_plan_rows >= 1),
        f"Source plan is empty or unreadable: {source_plan_error}",
    ),
    validation_row(
        "strict_metric_allowlist_defined",
        STRICT_METRIC_COLUMNS,
        "exact report-facing improvement metrics only",
        set(STRICT_METRIC_COLUMNS) == allowed_metric_set and len(STRICT_METRIC_COLUMNS) == len(allowed_metric_set),
        "Strict metric allowlist is missing or contains duplicates.",
    ),
    validation_row(
        "strict_metric_allowlist_excludes_known_pollution",
        polluted_allowed_metrics,
        "[]",
        len(polluted_allowed_metrics) == 0,
        "Strict metric allowlist contains metadata/rank/config pollution.",
    ),
    validation_row(
        "required_sources_available_in_inventory",
        available_required_count,
        len(required_source_df),
        missing_required_count == 0,
        "One or more required sources are missing from the inventory. Inspect batch0_source_plan.csv before continuing.",
    ),
]

batch0_validation_df = pd.DataFrame(batch0_validation_rows)
batch0_validation_df.to_csv(BATCH0_VALIDATION_PATH, index=False)

batch0_passed = bool(batch0_validation_df["passed"].all())

stage_manifest = read_json_if_exists(STAGE_MANIFEST_PATH)
stage_manifest.update(
    {
        "notebook_id": NOTEBOOK_ID,
        "notebook_title": NOTEBOOK_TITLE,
        "stage": "batch0_setup_inventory_source_plan",
        "stage_status": "passed" if batch0_passed else "failed",
        "updated_at_utc": utc_now_iso(),
        "project_root": str(PROJECT_ROOT),
        "outputs": {
            "inventory_snapshot_csv": rel(BATCH0_INVENTORY_SNAPSHOT_PATH),
            "source_plan_csv": rel(BATCH0_SOURCE_PLAN_PATH),
            "batch0_validation_csv": rel(BATCH0_VALIDATION_PATH),
            "stage_manifest_json": rel(STAGE_MANIFEST_PATH),
        },
        "strict_metric_columns": STRICT_METRIC_COLUMNS,
        "source_counts": {
            "planned_sources": int(len(batch0_source_plan_df)),
            "required_sources": int(len(required_source_df)),
            "available_required_sources": available_required_count,
            "missing_required_sources": missing_required_count,
        },
        "runtime_environment": {
            "python": platform.python_version(),
            "platform": platform.platform(),
            "pandas": pd.__version__,
            "numpy": np.__version__,
        },
    }
)

write_json(STAGE_MANIFEST_PATH, stage_manifest)

print(f"Saved validation: {rel(BATCH0_VALIDATION_PATH)}")
print(f"Saved stage manifest: {rel(STAGE_MANIFEST_PATH)}")
print(f"Batch 0 checks passed: {int(batch0_validation_df['passed'].sum())} / {len(batch0_validation_df)}")

display(batch0_validation_df)

if not batch0_passed:
    display(batch0_source_plan_df.loc[batch0_source_plan_df["required"] & ~batch0_source_plan_df["present_in_inventory"].astype(bool)])
    raise RuntimeError("Batch 0 validation failed. Fix missing required sources or inventory before Batch 1.")

print("Batch 0 passed. Inventory snapshot and source plan are ready.")

Saved validation: outputs/27_multi_model_comparison/validation/batch0_validation.csv
Saved stage manifest: outputs/27_multi_model_comparison/manifests/multi_model_comparison_stage_manifest.json
Batch 0 checks passed: 8 / 8


,check_name,actual,expected,passed,failure_message
0,project_inventory_loaded,10939,> 0 rows,True,
1,inventory_snapshot_written,outputs/27_multi_model_comparison/inventory/pr...,file exists,True,
2,inventory_snapshot_has_rows,10939,> 0 rows,True,
3,source_plan_written,outputs/27_multi_model_comparison/validation/b...,file exists,True,
4,source_plan_has_rows,13,>= 1 row,True,
5,strict_metric_allowlist_defined,"[mse_improvement, mae_improvement, psnr_improv...",exact report-facing improvement metrics only,True,
6,strict_metric_allowlist_excludes_known_pollution,[],[],True,
7,required_sources_available_in_inventory,12,12,True,


Batch 0 passed. Inventory snapshot and source plan are ready.


In [5]:
# Batch 1 / Cell 2 - Load Batch 0 source plan and define validation helpers
from __future__ import annotations

import json
import math
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

required_batch0_globals = [
    "PROJECT_ROOT",
    "OUTPUT_ROOT",
    "VALIDATION_DIR",
    "MANIFESTS_DIR",
    "BATCH0_SOURCE_PLAN_PATH",
    "STRICT_METRIC_COLUMNS",
    "METRIC_FAMILY_BY_COLUMN",
    "rel",
    "clean_text",
    "norm_rel_path",
    "utc_now_iso",
    "read_json_if_exists",
    "write_json",
    "validation_row",
]

missing_batch0_globals = [name for name in required_batch0_globals if name not in globals()]
if missing_batch0_globals:
    raise RuntimeError(
        "Batch 1 requires Batch 0 to be run first. Missing globals: "
        + ", ".join(missing_batch0_globals)
    )

BATCH1_VALIDATION_PATH = VALIDATION_DIR / "batch1_input_validation.csv"
STAGE_MANIFEST_PATH = MANIFESTS_DIR / "multi_model_comparison_stage_manifest.json"

if "batch0_source_plan_df" not in globals():
    if not BATCH0_SOURCE_PLAN_PATH.is_file():
        raise RuntimeError(f"Missing Batch 0 source plan: {rel(BATCH0_SOURCE_PLAN_PATH)}")
    batch0_source_plan_df = pd.read_csv(BATCH0_SOURCE_PLAN_PATH)

batch0_source_plan_df = batch0_source_plan_df.copy()
batch0_source_plan_df["resolved_relative_path"] = batch0_source_plan_df["resolved_relative_path"].map(norm_rel_path)

def batch1_parse_json_list(value: Any) -> list[str]:
    text = clean_text(value)
    if not text:
        return []
    try:
        parsed = json.loads(text)
        if isinstance(parsed, list):
            return [clean_text(item) for item in parsed if clean_text(item)]
    except Exception:
        pass
    return [part.strip().strip("'\"") for part in text.strip("[]").split(",") if part.strip()]

def batch1_bool(value: Any) -> bool:
    if isinstance(value, bool):
        return value
    text = clean_text(value).lower()
    return text in {"true", "1", "yes", "y"}

def batch1_path_from_relative(relative_path: Any) -> Path:
    return PROJECT_ROOT / norm_rel_path(relative_path)

def batch1_column_lookup(columns: list[str]) -> dict[str, str]:
    return {clean_text(column).lower(): clean_text(column) for column in columns}

def batch1_has_any_column(columns: list[str], candidates: list[str]) -> bool:
    lookup = batch1_column_lookup(columns)
    return any(candidate.lower() in lookup for candidate in candidates)

def batch1_present_columns(columns: list[str], candidates: list[str]) -> list[str]:
    lookup = batch1_column_lookup(columns)
    return [lookup[candidate.lower()] for candidate in candidates if candidate.lower() in lookup]

def batch1_family_metrics(metric_family: str) -> list[str]:
    return [
        column for column in STRICT_METRIC_COLUMNS
        if METRIC_FAMILY_BY_COLUMN.get(column) == metric_family
    ]

# Report tables are allowed to use report-facing identifiers.
# OpenCV Notebook 13 writes report_case_id/report_source_case_id, not plain case_id.
BATCH1_CASE_ID_COLUMNS = [
    "case_id",
    "canonical_case_id",
    "canonical_id",
    "sample_id",
    "image_id",
    "report_case_id",
    "report_source_case_id",
    "source_case_id",
    "restoration_case_id",
]

BATCH1_CANDIDATE_ID_COLUMNS = [
    "candidate_id",
    "report_candidate_id",
    "stable_diffusion_candidate_id",
    "restoration_candidate_id",
]

BATCH1_STATUS_COLUMNS = [
    "status",
    "candidate_status",
    "batch3_candidate_status",
    "restoration_status",
    "metric_status",
]

BATCH1_PATH_COLUMNS_HINTS = [
    "clean_path",
    "damaged_path",
    "mask_path",
    "restored_path",
    "restored_image_path",
    "primary_restored_candidate_path",
    "difference_map",
]

# Correct Batch 0's conservative OpenCV report requirement in-memory and on disk.
# The actual table is report-facing and uses report_case_id.
opencv_report_mask = batch0_source_plan_df["source_id"].astype(str).eq("opencv_report_cases")
batch0_source_plan_df.loc[opencv_report_mask, "required_columns"] = json.dumps(["report_case_id"])
batch0_source_plan_df.loc[opencv_report_mask, "notes"] = (
    "OpenCV report-facing paths/status source. Uses report_case_id/report_source_case_id rather than plain case_id."
)

batch0_source_plan_df.to_csv(BATCH0_SOURCE_PLAN_PATH, index=False)

print(f"Batch 1 validation output: {rel(BATCH1_VALIDATION_PATH)}")
print(f"Source plan rows: {len(batch0_source_plan_df):,}")
print("Patched OpenCV report schema expectation to report_case_id.")
display(batch0_source_plan_df[["source_id", "model_name", "source_type", "metric_family", "required", "resolved_relative_path", "required_columns"]])

Batch 1 validation output: outputs/27_multi_model_comparison/validation/batch1_input_validation.csv
Source plan rows: 13
Patched OpenCV report schema expectation to report_case_id.


,source_id,model_name,source_type,metric_family,required,resolved_relative_path,required_columns
0,opencv_classical_metrics,opencv_telea,metric,classical,True,data/processed/metrics/metrics_opencv_telea_cl...,"[""case_id"", ""model_name"", ""evaluation_region"",..."
1,opencv_lpips_metrics,opencv_telea,metric,lpips,True,data/processed/metrics/metrics_opencv_telea_lp...,"[""case_id"", ""model_name"", ""evaluation_region"",..."
2,opencv_feature_metrics,opencv_telea,metric,feature_similarity,True,data/processed/metrics/metrics_opencv_telea_fe...,"[""case_id"", ""model_name"", ""evaluation_region"",..."
3,opencv_report_cases,opencv_telea,report_visual,report,True,outputs/reports/opencv_report_assets/opencv_te...,"[""report_case_id""]"
4,lama_classical_metrics,lama,metric,classical,True,outputs/metrics/classical_metrics_lama.csv,"[""case_id"", ""model_name"", ""evaluation_region"",..."
5,lama_lpips_metrics,lama,metric,lpips,True,outputs/metrics/lpips_metrics_lama.csv,"[""case_id"", ""model_name"", ""evaluation_region"",..."
6,lama_feature_metrics,lama,metric,feature_similarity,True,outputs/metrics/feature_similarity_metrics_lam...,"[""case_id"", ""model_name"", ""evaluation_region"",..."
7,lama_report_cases,lama,report_visual,report,True,outputs/19_lama_report_generation/metrics/lama...,"[""case_id""]"
8,stable_diffusion_classical_metrics,stable_diffusion,metric,classical,True,outputs/22_stable_diffusion_classical_metrics/...,"[""case_id"", ""evaluation_region"", ""mse_improvem..."
9,stable_diffusion_lpips_metrics,stable_diffusion,metric,lpips,True,outputs/24_stable_diffusion_lpips_metrics/metr...,"[""case_id"", ""evaluation_region"", ""lpips_improv..."


In [6]:
# Batch 1 / Cell 3 - Load available input files into a source registry
batch1_source_frames: dict[str, pd.DataFrame] = {}
batch1_source_load_rows: list[dict[str, Any]] = []

for _, source_row in batch0_source_plan_df.iterrows():
    source_id = clean_text(source_row["source_id"])
    source_type = clean_text(source_row["source_type"])
    model_name = clean_text(source_row["model_name"])
    metric_family = clean_text(source_row["metric_family"])
    required = batch1_bool(source_row["required"])
    resolved_relative_path = norm_rel_path(source_row["resolved_relative_path"])

    candidate_paths = batch1_parse_json_list(source_row.get("candidate_paths", "[]"))

    paths_to_try = []
    if resolved_relative_path:
        paths_to_try.append(resolved_relative_path)
    paths_to_try.extend([norm_rel_path(path) for path in candidate_paths])
    paths_to_try = list(dict.fromkeys([path for path in paths_to_try if path]))

    load_path = ""
    load_status = "missing"
    load_error = ""
    frame = pd.DataFrame()

    for candidate_path in paths_to_try:
        absolute_path = batch1_path_from_relative(candidate_path)
        if not absolute_path.is_file():
            continue

        try:
            frame = pd.read_csv(absolute_path, low_memory=False)
            load_path = candidate_path
            load_status = "loaded"
            batch1_source_frames[source_id] = frame
            break
        except Exception as exc:
            load_path = candidate_path
            load_status = "read_error"
            load_error = repr(exc)
            break

    batch1_source_load_rows.append(
        {
            "source_id": source_id,
            "model_name": model_name,
            "source_type": source_type,
            "metric_family": metric_family,
            "required": required,
            "load_status": load_status,
            "loaded_relative_path": load_path,
            "row_count": int(frame.shape[0]) if load_status == "loaded" else 0,
            "column_count": int(frame.shape[1]) if load_status == "loaded" else 0,
            "load_error": load_error,
        }
    )

batch1_source_load_df = pd.DataFrame(batch1_source_load_rows)

print(f"Loaded sources: {(batch1_source_load_df['load_status'] == 'loaded').sum()} / {len(batch1_source_load_df)}")
display(batch1_source_load_df)

Loaded sources: 12 / 13


,source_id,model_name,source_type,metric_family,required,load_status,loaded_relative_path,row_count,column_count,load_error
0,opencv_classical_metrics,opencv_telea,metric,classical,True,loaded,data/processed/metrics/metrics_opencv_telea_cl...,2295,48,
1,opencv_lpips_metrics,opencv_telea,metric,lpips,True,loaded,data/processed/metrics/metrics_opencv_telea_lp...,1175,47,
2,opencv_feature_metrics,opencv_telea,metric,feature_similarity,True,loaded,data/processed/metrics/metrics_opencv_telea_fe...,1175,59,
3,opencv_report_cases,opencv_telea,report_visual,report,True,loaded,outputs/reports/opencv_report_assets/opencv_te...,410,69,
4,lama_classical_metrics,lama,metric,classical,True,loaded,outputs/metrics/classical_metrics_lama.csv,2260,51,
5,lama_lpips_metrics,lama,metric,lpips,True,loaded,outputs/metrics/lpips_metrics_lama.csv,1175,47,
6,lama_feature_metrics,lama,metric,feature_similarity,True,loaded,outputs/metrics/feature_similarity_metrics_lam...,1175,59,
7,lama_report_cases,lama,report_visual,report,True,loaded,outputs/19_lama_report_generation/metrics/lama...,355,264,
8,stable_diffusion_classical_metrics,stable_diffusion,metric,classical,True,loaded,outputs/22_stable_diffusion_classical_metrics/...,5294,58,
9,stable_diffusion_lpips_metrics,stable_diffusion,metric,lpips,True,loaded,outputs/24_stable_diffusion_lpips_metrics/metr...,2724,66,


In [7]:
# Batch 1 / Cell 3 - Load available input files into a source registry
batch1_source_frames: dict[str, pd.DataFrame] = {}
batch1_source_load_rows: list[dict[str, Any]] = []

for _, source_row in batch0_source_plan_df.iterrows():
    source_id = clean_text(source_row["source_id"])
    source_type = clean_text(source_row["source_type"])
    model_name = clean_text(source_row["model_name"])
    metric_family = clean_text(source_row["metric_family"])
    required = batch1_bool(source_row["required"])
    resolved_relative_path = norm_rel_path(source_row["resolved_relative_path"])

    candidate_paths = batch1_parse_json_list(source_row.get("candidate_paths", "[]"))

    paths_to_try = []
    if resolved_relative_path:
        paths_to_try.append(resolved_relative_path)
    paths_to_try.extend([norm_rel_path(path) for path in candidate_paths])
    paths_to_try = list(dict.fromkeys([path for path in paths_to_try if path]))

    load_path = ""
    load_status = "missing"
    load_error = ""
    frame = pd.DataFrame()

    for candidate_path in paths_to_try:
        absolute_path = batch1_path_from_relative(candidate_path)
        if not absolute_path.is_file():
            continue

        try:
            frame = pd.read_csv(absolute_path, low_memory=False)
            load_path = candidate_path
            load_status = "loaded"
            batch1_source_frames[source_id] = frame
            break
        except Exception as exc:
            load_path = candidate_path
            load_status = "read_error"
            load_error = repr(exc)
            break

    batch1_source_load_rows.append(
        {
            "source_id": source_id,
            "model_name": model_name,
            "source_type": source_type,
            "metric_family": metric_family,
            "required": required,
            "load_status": load_status,
            "loaded_relative_path": load_path,
            "row_count": int(frame.shape[0]) if load_status == "loaded" else 0,
            "column_count": int(frame.shape[1]) if load_status == "loaded" else 0,
            "load_error": load_error,
        }
    )

batch1_source_load_df = pd.DataFrame(batch1_source_load_rows)

print(f"Loaded sources: {(batch1_source_load_df['load_status'] == 'loaded').sum()} / {len(batch1_source_load_df)}")
display(batch1_source_load_df)

Loaded sources: 12 / 13


,source_id,model_name,source_type,metric_family,required,load_status,loaded_relative_path,row_count,column_count,load_error
0,opencv_classical_metrics,opencv_telea,metric,classical,True,loaded,data/processed/metrics/metrics_opencv_telea_cl...,2295,48,
1,opencv_lpips_metrics,opencv_telea,metric,lpips,True,loaded,data/processed/metrics/metrics_opencv_telea_lp...,1175,47,
2,opencv_feature_metrics,opencv_telea,metric,feature_similarity,True,loaded,data/processed/metrics/metrics_opencv_telea_fe...,1175,59,
3,opencv_report_cases,opencv_telea,report_visual,report,True,loaded,outputs/reports/opencv_report_assets/opencv_te...,410,69,
4,lama_classical_metrics,lama,metric,classical,True,loaded,outputs/metrics/classical_metrics_lama.csv,2260,51,
5,lama_lpips_metrics,lama,metric,lpips,True,loaded,outputs/metrics/lpips_metrics_lama.csv,1175,47,
6,lama_feature_metrics,lama,metric,feature_similarity,True,loaded,outputs/metrics/feature_similarity_metrics_lam...,1175,59,
7,lama_report_cases,lama,report_visual,report,True,loaded,outputs/19_lama_report_generation/metrics/lama...,355,264,
8,stable_diffusion_classical_metrics,stable_diffusion,metric,classical,True,loaded,outputs/22_stable_diffusion_classical_metrics/...,5294,58,
9,stable_diffusion_lpips_metrics,stable_diffusion,metric,lpips,True,loaded,outputs/24_stable_diffusion_lpips_metrics/metr...,2724,66,


In [8]:
# Batch 1 / Cell 4 - Validate schemas and strict metric availability
batch1_schema_rows: list[dict[str, Any]] = []

for _, source_row in batch0_source_plan_df.iterrows():
    source_id = clean_text(source_row["source_id"])
    source_type = clean_text(source_row["source_type"])
    metric_family = clean_text(source_row["metric_family"])
    model_name = clean_text(source_row["model_name"])
    required = batch1_bool(source_row["required"])

    frame = batch1_source_frames.get(source_id, pd.DataFrame())
    loaded = source_id in batch1_source_frames
    columns = [clean_text(column) for column in frame.columns] if loaded else []
    column_lookup = batch1_column_lookup(columns)

    planned_required_columns = batch1_parse_json_list(source_row.get("required_columns", "[]"))
    planned_missing_columns = [
        column for column in planned_required_columns
        if column.lower() not in column_lookup
    ]

    strict_family_metrics = batch1_family_metrics(metric_family)
    present_strict_metrics = batch1_present_columns(columns, strict_family_metrics)

    has_case_id = batch1_has_any_column(columns, BATCH1_CASE_ID_COLUMNS)
    has_candidate_id = batch1_has_any_column(columns, BATCH1_CANDIDATE_ID_COLUMNS)
    has_status = batch1_has_any_column(columns, BATCH1_STATUS_COLUMNS)
    path_like_columns = [
        column for column in columns
        if any(token in column.lower() for token in BATCH1_PATH_COLUMNS_HINTS)
    ]

    if source_type == "metric":
        schema_passed = bool(
            loaded
            and frame.shape[0] > 0
            and has_case_id
            and len(present_strict_metrics) > 0
        )
        failure_message = (
            "" if schema_passed else
            "Metric source must load, have rows, expose a case id, and contain at least one strict metric for its metric family."
        )
    elif source_type == "report_visual":
        schema_passed = bool(
            loaded
            and frame.shape[0] > 0
            and (has_case_id or has_candidate_id)
        )
        failure_message = (
            "" if schema_passed else
            "Report/visual source must load, have rows, and expose either a case id or candidate id."
        )
    elif source_type == "optional_scan":
        schema_passed = True
        failure_message = ""
    else:
        schema_passed = bool(loaded and frame.shape[0] > 0)
        failure_message = "" if schema_passed else "Unknown source type did not load with rows."

    batch1_schema_rows.append(
        {
            "source_id": source_id,
            "model_name": model_name,
            "source_type": source_type,
            "metric_family": metric_family,
            "required": required,
            "loaded": bool(loaded),
            "row_count": int(frame.shape[0]) if loaded else 0,
            "column_count": int(frame.shape[1]) if loaded else 0,
            "has_case_id": bool(has_case_id),
            "has_candidate_id": bool(has_candidate_id),
            "has_status_column": bool(has_status),
            "strict_family_metrics_expected": json.dumps(strict_family_metrics),
            "strict_family_metrics_present": json.dumps(present_strict_metrics),
            "planned_required_columns": json.dumps(planned_required_columns),
            "planned_missing_columns": json.dumps(planned_missing_columns),
            "path_like_columns_detected": json.dumps(path_like_columns[:30]),
            "schema_passed": bool(schema_passed),
            "failure_message": failure_message,
        }
    )

batch1_schema_df = pd.DataFrame(batch1_schema_rows)

required_schema_failures_df = batch1_schema_df.loc[
    batch1_schema_df["required"].astype(bool) & ~batch1_schema_df["schema_passed"].astype(bool)
].copy()

print(f"Required schema failures: {len(required_schema_failures_df):,}")
display(batch1_schema_df)

if len(required_schema_failures_df):
    display(required_schema_failures_df)

Required schema failures: 0


,source_id,model_name,source_type,metric_family,required,loaded,row_count,column_count,has_case_id,has_candidate_id,has_status_column,strict_family_metrics_expected,strict_family_metrics_present,planned_required_columns,planned_missing_columns,path_like_columns_detected,schema_passed,failure_message
0,opencv_classical_metrics,opencv_telea,metric,classical,True,True,2295,48,True,False,True,"[""mse_improvement"", ""mae_improvement"", ""psnr_i...","[""mse_improvement"", ""mae_improvement"", ""psnr_i...","[""case_id"", ""model_name"", ""evaluation_region"",...",[],[],True,
1,opencv_lpips_metrics,opencv_telea,metric,lpips,True,True,1175,47,True,False,True,"[""lpips_improvement""]","[""lpips_improvement""]","[""case_id"", ""model_name"", ""evaluation_region"",...",[],[],True,
2,opencv_feature_metrics,opencv_telea,metric,feature_similarity,True,True,1175,59,True,False,True,"[""clip_similarity_improvement"", ""dinov2_simila...","[""clip_similarity_improvement"", ""dinov2_simila...","[""case_id"", ""model_name"", ""evaluation_region"",...",[],[],True,
3,opencv_report_cases,opencv_telea,report_visual,report,True,True,410,69,True,False,True,[],[],"[""report_case_id""]",[],"[""clean_path"", ""damaged_path"", ""mask_path"", ""r...",True,
4,lama_classical_metrics,lama,metric,classical,True,True,2260,51,True,False,True,"[""mse_improvement"", ""mae_improvement"", ""psnr_i...","[""mse_improvement"", ""mae_improvement"", ""psnr_i...","[""case_id"", ""model_name"", ""evaluation_region"",...",[],"[""clean_path"", ""damaged_path"", ""mask_path"", ""r...",True,
5,lama_lpips_metrics,lama,metric,lpips,True,True,1175,47,True,False,True,"[""lpips_improvement""]","[""lpips_improvement""]","[""case_id"", ""model_name"", ""evaluation_region"",...",[],[],True,
6,lama_feature_metrics,lama,metric,feature_similarity,True,True,1175,59,True,False,True,"[""clip_similarity_improvement"", ""dinov2_simila...","[""clip_similarity_improvement"", ""dinov2_simila...","[""case_id"", ""model_name"", ""evaluation_region"",...",[],[],True,
7,lama_report_cases,lama,report_visual,report,True,True,355,264,True,False,True,[],[],"[""case_id""]",[],"[""clean_path"", ""damaged_path"", ""mask_path"", ""r...",True,
8,stable_diffusion_classical_metrics,stable_diffusion,metric,classical,True,True,5294,58,True,True,True,"[""mse_improvement"", ""mae_improvement"", ""psnr_i...","[""mse_improvement"", ""mae_improvement"", ""psnr_i...","[""case_id"", ""evaluation_region"", ""mse_improvem...",[],"[""restored_path"", ""damaged_path"", ""mask_path"",...",True,
9,stable_diffusion_lpips_metrics,stable_diffusion,metric,lpips,True,True,2724,66,True,True,True,"[""lpips_improvement""]","[""lpips_improvement""]","[""case_id"", ""evaluation_region"", ""lpips_improv...",[],[],True,


In [9]:
# Batch 1 / Cell 5 - Detect optional SDXL availability
sdxl_source_plan_df = batch0_source_plan_df.loc[
    batch0_source_plan_df["model_name"].astype(str).str.lower().eq("sdxl")
].copy()

sdxl_candidate_paths = []
for _, row in sdxl_source_plan_df.iterrows():
    sdxl_candidate_paths.extend(batch1_parse_json_list(row.get("candidate_paths", "[]")))
    resolved_path = norm_rel_path(row.get("resolved_relative_path", ""))
    if resolved_path:
        sdxl_candidate_paths.append(resolved_path)

sdxl_candidate_paths = list(dict.fromkeys([norm_rel_path(path) for path in sdxl_candidate_paths if clean_text(path)]))

inventory_sdxl_paths = []
if "project_inventory_df" in globals() and "relative_path" in project_inventory_df.columns:
    inventory_sdxl_paths = sorted(
        project_inventory_df.loc[
            project_inventory_df["relative_path"].astype(str).str.lower().str.contains("sdxl", na=False),
            "relative_path",
        ].map(norm_rel_path).unique().tolist()
    )

detected_sdxl_paths = []
for candidate_path in sdxl_candidate_paths + inventory_sdxl_paths:
    absolute_path = batch1_path_from_relative(candidate_path)
    if absolute_path.is_file():
        detected_sdxl_paths.append(candidate_path)

detected_sdxl_paths = list(dict.fromkeys(detected_sdxl_paths))

batch1_sdxl_detection_df = pd.DataFrame(
    [
        {
            "detection_type": "planned_candidate",
            "relative_path": path,
            "file_exists": batch1_path_from_relative(path).is_file(),
        }
        for path in sdxl_candidate_paths
    ]
    + [
        {
            "detection_type": "inventory_scan",
            "relative_path": path,
            "file_exists": batch1_path_from_relative(path).is_file(),
        }
        for path in inventory_sdxl_paths
        if path not in sdxl_candidate_paths
    ]
)

batch1_sdxl_available = len(detected_sdxl_paths) > 0

print(f"SDXL available: {batch1_sdxl_available}")
print(f"Detected SDXL files: {len(detected_sdxl_paths):,}")
display(batch1_sdxl_detection_df.head(50))

SDXL available: True
Detected SDXL files: 2


,detection_type,relative_path,file_exists
0,planned_candidate,outputs/metrics/classical_metrics_sdxl.csv,False
1,planned_candidate,outputs/metrics/lpips_metrics_sdxl.csv,False
2,planned_candidate,outputs/metrics/feature_similarity_metrics_sdx...,False
3,planned_candidate,outputs/26_sdxl_report_generation/metrics/sdxl...,False
4,inventory_scan,notebooks/25_sdxl_feasibility_audit_cleaned.ipynb,True
5,inventory_scan,src/restoration_eval/restoration_sdxl.py,True


In [10]:
# Batch 1 / Cell 6 - Write validation CSV and update stage manifest
loaded_required_sources = int(
    batch1_source_load_df.loc[batch1_source_load_df["required"].astype(bool), "load_status"]
    .eq("loaded")
    .sum()
)
required_source_count = int(batch1_source_load_df["required"].astype(bool).sum())

required_schema_pass_count = int(
    batch1_schema_df.loc[batch1_schema_df["required"].astype(bool), "schema_passed"]
    .astype(bool)
    .sum()
)

batch1_validation_rows = [
    validation_row(
        "batch0_source_plan_available",
        rel(BATCH0_SOURCE_PLAN_PATH),
        "file exists",
        BATCH0_SOURCE_PLAN_PATH.is_file(),
        "Batch 0 source plan is missing.",
    ),
    validation_row(
        "required_sources_loaded",
        loaded_required_sources,
        required_source_count,
        loaded_required_sources == required_source_count,
        "One or more required input sources did not load.",
    ),
    validation_row(
        "required_schemas_valid",
        required_schema_pass_count,
        required_source_count,
        required_schema_pass_count == required_source_count,
        "One or more required input schemas failed validation.",
    ),
    validation_row(
        "strict_metric_columns_preserved",
        STRICT_METRIC_COLUMNS,
        "exact strict allowlist from Batch 0",
        STRICT_METRIC_COLUMNS == [
            "mse_improvement",
            "mae_improvement",
            "psnr_improvement",
            "ssim_improvement",
            "lpips_improvement",
            "clip_similarity_improvement",
            "dinov2_similarity_improvement",
            "mean_similarity_improvement",
        ],
        "Strict metric allowlist changed unexpectedly.",
    ),
    validation_row(
        "sdxl_optional_detection_completed",
        detected_sdxl_paths if batch1_sdxl_available else "no SDXL files detected",
        "available or explicitly unavailable",
        True,
        "",
    ),
]

batch1_validation_df = pd.concat(
    [
        pd.DataFrame(batch1_validation_rows),
        batch1_source_load_df.assign(
            check_name=lambda df: "source_load__" + df["source_id"],
            actual=lambda df: df["load_status"],
            expected=lambda df: np.where(df["required"], "loaded", "loaded or missing"),
            passed=lambda df: np.where(df["required"], df["load_status"].eq("loaded"), True),
            failure_message=lambda df: np.where(
                df["required"] & ~df["load_status"].eq("loaded"),
                "Required source failed to load: " + df["load_error"].fillna(""),
                "",
            ),
        )[["check_name", "actual", "expected", "passed", "failure_message"]],
        batch1_schema_df.assign(
            check_name=lambda df: "schema__" + df["source_id"],
            actual=lambda df: np.where(
                df["loaded"],
                "rows=" + df["row_count"].astype(str) + ", columns=" + df["column_count"].astype(str),
                "not loaded",
            ),
            expected=lambda df: np.where(df["required"], "valid required schema", "valid or optional"),
            passed=lambda df: np.where(df["required"], df["schema_passed"], True),
        )[["check_name", "actual", "expected", "passed", "failure_message"]],
    ],
    ignore_index=True,
)

batch1_validation_df["passed"] = batch1_validation_df["passed"].astype(bool)
batch1_validation_df.to_csv(BATCH1_VALIDATION_PATH, index=False)

batch1_passed = bool(batch1_validation_df["passed"].all())

stage_manifest = read_json_if_exists(STAGE_MANIFEST_PATH)
stage_manifest.update(
    {
        "notebook_id": NOTEBOOK_ID,
        "notebook_title": NOTEBOOK_TITLE,
        "stage": "batch1_load_inputs_validate_schemas_detect_sdxl",
        "stage_status": "passed" if batch1_passed else "failed",
        "updated_at_utc": utc_now_iso(),
        "outputs": {
            **stage_manifest.get("outputs", {}),
            "batch1_input_validation_csv": rel(BATCH1_VALIDATION_PATH),
        },
        "batch1": {
            "required_sources": required_source_count,
            "loaded_required_sources": loaded_required_sources,
            "required_schema_pass_count": required_schema_pass_count,
            "sdxl_available": bool(batch1_sdxl_available),
            "detected_sdxl_paths": detected_sdxl_paths,
            "loaded_source_ids": sorted(batch1_source_frames.keys()),
        },
    }
)

write_json(STAGE_MANIFEST_PATH, stage_manifest)

print(f"Saved Batch 1 validation: {rel(BATCH1_VALIDATION_PATH)}")
print(f"Batch 1 checks passed: {int(batch1_validation_df['passed'].sum())} / {len(batch1_validation_df)}")

display(batch1_validation_df)

if not batch1_passed:
    display(batch1_validation_df.loc[~batch1_validation_df["passed"], ["check_name", "actual", "expected", "failure_message"]])
    raise RuntimeError("Batch 1 validation failed. Fix missing inputs or schema mapping before Batch 2.")

print("Batch 1 passed. Inputs are loaded and validated for normalization.")

Saved Batch 1 validation: outputs/27_multi_model_comparison/validation/batch1_input_validation.csv
Batch 1 checks passed: 31 / 31


,check_name,actual,expected,passed,failure_message
0,batch0_source_plan_available,outputs/27_multi_model_comparison/validation/b...,file exists,True,
1,required_sources_loaded,12,12,True,
2,required_schemas_valid,12,12,True,
3,strict_metric_columns_preserved,"[mse_improvement, mae_improvement, psnr_improv...",exact strict allowlist from Batch 0,True,
4,sdxl_optional_detection_completed,[notebooks/25_sdxl_feasibility_audit_cleaned.i...,available or explicitly unavailable,True,
5,source_load__opencv_classical_metrics,loaded,loaded,True,
6,source_load__opencv_lpips_metrics,loaded,loaded,True,
7,source_load__opencv_feature_metrics,loaded,loaded,True,
8,source_load__opencv_report_cases,loaded,loaded,True,
9,source_load__lama_classical_metrics,loaded,loaded,True,


Batch 1 passed. Inputs are loaded and validated for normalization.


In [11]:
# Batch 2 / Cell 2 - Setup paths and helpers
from __future__ import annotations

import json
from typing import Any

import numpy as np
import pandas as pd

required_batch2_globals = [
    "PROJECT_ROOT",
    "OUTPUT_ROOT",
    "VALIDATION_DIR",
    "MANIFESTS_DIR",
    "STRICT_METRIC_COLUMNS",
    "METRIC_FAMILY_BY_COLUMN",
    "batch0_source_plan_df",
    "batch1_source_frames",
    "rel",
    "clean_text",
    "norm_rel_path",
    "utc_now_iso",
    "read_json_if_exists",
    "write_json",
    "validation_row",
]

missing_batch2_globals = [name for name in required_batch2_globals if name not in globals()]
if missing_batch2_globals:
    raise RuntimeError(
        "Batch 2 requires Batches 0 and 1 to be run first. Missing globals: "
        + ", ".join(missing_batch2_globals)
    )

METRICS_DIR = OUTPUT_ROOT / "metrics"
METRICS_DIR.mkdir(parents=True, exist_ok=True)

BATCH2_METRIC_LONG_PATH = METRICS_DIR / "multi_model_metric_long.csv"
BATCH2_VALIDATION_PATH = VALIDATION_DIR / "batch2_metric_long_validation.csv"
STAGE_MANIFEST_PATH = MANIFESTS_DIR / "multi_model_comparison_stage_manifest.json"

def batch2_parse_json_list(value: Any) -> list[str]:
    text = clean_text(value)
    if not text:
        return []
    try:
        parsed = json.loads(text)
        if isinstance(parsed, list):
            return [clean_text(item) for item in parsed if clean_text(item)]
    except Exception:
        pass
    return [part.strip().strip("'\"") for part in text.strip("[]").split(",") if part.strip()]

def batch2_bool(value: Any) -> bool:
    if isinstance(value, bool):
        return value
    text = clean_text(value).lower()
    return text in {"true", "1", "yes", "y"}

def batch2_first_present_column(columns: list[str], candidates: list[str]) -> str:
    lookup = {clean_text(column).lower(): clean_text(column) for column in columns}
    for candidate in candidates:
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]
    return ""

def batch2_present_strict_metrics(columns: list[str], metric_family: str) -> list[str]:
    expected = [
        column for column in STRICT_METRIC_COLUMNS
        if METRIC_FAMILY_BY_COLUMN.get(column) == metric_family
    ]
    lookup = {clean_text(column).lower(): clean_text(column) for column in columns}
    return [lookup[column.lower()] for column in expected if column.lower() in lookup]

BATCH2_CASE_ID_COLUMNS = [
    "case_id",
    "canonical_case_id",
    "canonical_id",
    "sample_id",
    "image_id",
]

BATCH2_CANDIDATE_ID_COLUMNS = [
    "candidate_id",
    "report_candidate_id",
    "stable_diffusion_candidate_id",
    "restoration_candidate_id",
]

BATCH2_STATUS_COLUMNS = [
    "status",
    "candidate_status",
    "batch3_candidate_status",
    "restoration_status",
    "metric_status",
]

BATCH2_VARIANT_COLUMNS = [
    "variant_type",
    "damage_variant",
    "mask_variant",
    "degradation_type",
    "prompt_ablation_group",
    "prompt_ablation_subset",
]

BATCH2_ALLOWED_SOURCE_TYPES = {"metric"}

metric_source_plan_df = batch0_source_plan_df.loc[
    batch0_source_plan_df["source_type"].astype(str).str.lower().isin(BATCH2_ALLOWED_SOURCE_TYPES)
].copy()

print(f"Batch 2 output: {rel(BATCH2_METRIC_LONG_PATH)}")
print(f"Metric sources planned: {len(metric_source_plan_df):,}")
display(metric_source_plan_df[["source_id", "model_name", "metric_family", "required", "resolved_relative_path"]])

Batch 2 output: outputs/27_multi_model_comparison/metrics/multi_model_metric_long.csv
Metric sources planned: 9


,source_id,model_name,metric_family,required,resolved_relative_path
0,opencv_classical_metrics,opencv_telea,classical,True,data/processed/metrics/metrics_opencv_telea_cl...
1,opencv_lpips_metrics,opencv_telea,lpips,True,data/processed/metrics/metrics_opencv_telea_lp...
2,opencv_feature_metrics,opencv_telea,feature_similarity,True,data/processed/metrics/metrics_opencv_telea_fe...
4,lama_classical_metrics,lama,classical,True,outputs/metrics/classical_metrics_lama.csv
5,lama_lpips_metrics,lama,lpips,True,outputs/metrics/lpips_metrics_lama.csv
6,lama_feature_metrics,lama,feature_similarity,True,outputs/metrics/feature_similarity_metrics_lam...
8,stable_diffusion_classical_metrics,stable_diffusion,classical,True,outputs/22_stable_diffusion_classical_metrics/...
9,stable_diffusion_lpips_metrics,stable_diffusion,lpips,True,outputs/24_stable_diffusion_lpips_metrics/metr...
10,stable_diffusion_feature_metrics,stable_diffusion,feature_similarity,True,outputs/25_stable_diffusion_feature_similarity...


In [12]:
# Batch 2 / Cell 3 - Normalize strict metrics
batch2_long_frames: list[pd.DataFrame] = []
batch2_source_summary_rows: list[dict[str, Any]] = []

for _, source_row in metric_source_plan_df.iterrows():
    source_id = clean_text(source_row["source_id"])
    model_name = clean_text(source_row["model_name"])
    metric_family = clean_text(source_row["metric_family"])
    required = batch2_bool(source_row["required"])
    source_relative_path = norm_rel_path(source_row.get("resolved_relative_path", ""))

    source_df = batch1_source_frames.get(source_id)
    loaded = source_df is not None and len(source_df) > 0

    if not loaded:
        batch2_source_summary_rows.append(
            {
                "source_id": source_id,
                "model_name": model_name,
                "metric_family": metric_family,
                "required": required,
                "loaded": False,
                "input_rows": 0,
                "strict_metric_columns": json.dumps([]),
                "normalized_rows": 0,
                "finite_metric_rows": 0,
                "status": "missing",
                "message": "Source was not loaded in Batch 1.",
            }
        )
        continue

    source_df = source_df.copy()
    source_df.columns = [clean_text(column) for column in source_df.columns]
    columns = list(source_df.columns)

    case_id_column = batch2_first_present_column(columns, BATCH2_CASE_ID_COLUMNS)
    candidate_id_column = batch2_first_present_column(columns, BATCH2_CANDIDATE_ID_COLUMNS)
    status_column = batch2_first_present_column(columns, BATCH2_STATUS_COLUMNS)
    evaluation_region_column = batch2_first_present_column(columns, ["evaluation_region", "region", "metric_region"])

    strict_metric_columns = batch2_present_strict_metrics(columns, metric_family)

    if not case_id_column or not strict_metric_columns:
        batch2_source_summary_rows.append(
            {
                "source_id": source_id,
                "model_name": model_name,
                "metric_family": metric_family,
                "required": required,
                "loaded": True,
                "input_rows": int(len(source_df)),
                "strict_metric_columns": json.dumps(strict_metric_columns),
                "normalized_rows": 0,
                "finite_metric_rows": 0,
                "status": "schema_failed",
                "message": "Missing case id column or strict metric columns.",
            }
        )
        continue

    working_df = pd.DataFrame(
        {
            "source_row_index": source_df.index.astype(int),
            "source_id": source_id,
            "source_relative_path": source_relative_path,
            "model_name": model_name,
            "metric_family": metric_family,
            "case_id": source_df[case_id_column].astype(str).map(clean_text),
            "case_id_column": case_id_column,
            "candidate_id": (
                source_df[candidate_id_column].astype(str).map(clean_text)
                if candidate_id_column
                else "single_candidate"
            ),
            "candidate_id_column": candidate_id_column if candidate_id_column else "",
            "evaluation_region": (
                source_df[evaluation_region_column].astype(str).map(clean_text)
                if evaluation_region_column
                else "unknown"
            ),
            "status": (
                source_df[status_column].astype(str).map(clean_text)
                if status_column
                else ""
            ),
        }
    )

    for variant_column in BATCH2_VARIANT_COLUMNS:
        if variant_column in source_df.columns:
            working_df[variant_column] = source_df[variant_column].astype(str).map(clean_text)

    for metric_column in strict_metric_columns:
        working_df[metric_column] = pd.to_numeric(source_df[metric_column], errors="coerce")

    normalized_df = working_df.melt(
        id_vars=[column for column in working_df.columns if column not in strict_metric_columns],
        value_vars=strict_metric_columns,
        var_name="metric_name",
        value_name="metric_value",
    )

    normalized_df["metric_value"] = pd.to_numeric(normalized_df["metric_value"], errors="coerce")
    normalized_df["metric_value_is_finite"] = np.isfinite(normalized_df["metric_value"].to_numpy(dtype=float))
    normalized_df["metric_direction"] = "higher_is_better"
    normalized_df["metric_kind"] = "improvement_delta"
    normalized_df["metric_record_id"] = (
        normalized_df["source_id"].astype(str)
        + "::"
        + normalized_df["source_row_index"].astype(str)
        + "::"
        + normalized_df["metric_name"].astype(str)
    )

    batch2_long_frames.append(normalized_df)

    batch2_source_summary_rows.append(
        {
            "source_id": source_id,
            "model_name": model_name,
            "metric_family": metric_family,
            "required": required,
            "loaded": True,
            "input_rows": int(len(source_df)),
            "strict_metric_columns": json.dumps(strict_metric_columns),
            "normalized_rows": int(len(normalized_df)),
            "finite_metric_rows": int(normalized_df["metric_value_is_finite"].sum()),
            "status": "normalized",
            "message": "",
        }
    )

batch2_metric_long_df = (
    pd.concat(batch2_long_frames, ignore_index=True)
    if batch2_long_frames
    else pd.DataFrame()
)

batch2_source_summary_df = pd.DataFrame(batch2_source_summary_rows)

if len(batch2_metric_long_df):
    batch2_metric_long_df = batch2_metric_long_df.sort_values(
        ["model_name", "metric_family", "case_id", "candidate_id", "evaluation_region", "metric_name", "source_row_index"],
        kind="mergesort",
    ).reset_index(drop=True)

print(f"Normalized metric rows: {len(batch2_metric_long_df):,}")
display(batch2_source_summary_df)
display(batch2_metric_long_df.head(20))

Normalized metric rows: 59,692


,source_id,model_name,metric_family,required,loaded,input_rows,strict_metric_columns,normalized_rows,finite_metric_rows,status,message
0,opencv_classical_metrics,opencv_telea,classical,True,True,2295,"[""mse_improvement"", ""mae_improvement"", ""psnr_i...",9180,8060,normalized,
1,opencv_lpips_metrics,opencv_telea,lpips,True,True,1175,"[""lpips_improvement""]",1175,1175,normalized,
2,opencv_feature_metrics,opencv_telea,feature_similarity,True,True,1175,"[""clip_similarity_improvement"", ""dinov2_simila...",3525,3525,normalized,
3,lama_classical_metrics,lama,classical,True,True,2260,"[""mse_improvement"", ""mae_improvement"", ""psnr_i...",9040,7960,normalized,
4,lama_lpips_metrics,lama,lpips,True,True,1175,"[""lpips_improvement""]",1175,1175,normalized,
5,lama_feature_metrics,lama,feature_similarity,True,True,1175,"[""clip_similarity_improvement"", ""dinov2_simila...",3525,3525,normalized,
6,stable_diffusion_classical_metrics,stable_diffusion,classical,True,True,5294,"[""mse_improvement"", ""mae_improvement"", ""psnr_i...",21176,18623,normalized,
7,stable_diffusion_lpips_metrics,stable_diffusion,lpips,True,True,2724,"[""lpips_improvement""]",2724,2724,normalized,
8,stable_diffusion_feature_metrics,stable_diffusion,feature_similarity,True,True,2724,"[""clip_similarity_improvement"", ""dinov2_simila...",8172,8172,normalized,


,source_row_index,source_id,source_relative_path,model_name,metric_family,case_id,case_id_column,candidate_id,candidate_id_column,evaluation_region,status,metric_name,metric_value,metric_value_is_finite,metric_direction,metric_kind,metric_record_id,prompt_ablation_subset
0,3,lama_classical_metrics,outputs/metrics/classical_metrics_lama.csv,lama,classical,canonical__p001_loss_large,case_id,single_candidate,,boundary_region,ok,mae_improvement,110.674597,True,higher_is_better,improvement_delta,lama_classical_metrics::3::mae_improvement,NaN
1,3,lama_classical_metrics,outputs/metrics/classical_metrics_lama.csv,lama,classical,canonical__p001_loss_large,case_id,single_candidate,,boundary_region,ok,mse_improvement,25396.024811,True,higher_is_better,improvement_delta,lama_classical_metrics::3::mse_improvement,NaN
2,3,lama_classical_metrics,outputs/metrics/classical_metrics_lama.csv,lama,classical,canonical__p001_loss_large,case_id,single_candidate,,boundary_region,ok,psnr_improvement,35.140351,True,higher_is_better,improvement_delta,lama_classical_metrics::3::psnr_improvement,NaN
3,3,lama_classical_metrics,outputs/metrics/classical_metrics_lama.csv,lama,classical,canonical__p001_loss_large,case_id,single_candidate,,boundary_region,ok,ssim_improvement,NaN,False,higher_is_better,improvement_delta,lama_classical_metrics::3::ssim_improvement,NaN
4,1,lama_classical_metrics,outputs/metrics/classical_metrics_lama.csv,lama,classical,canonical__p001_loss_large,case_id,single_candidate,,content_region,ok,mae_improvement,28.347510,True,higher_is_better,improvement_delta,lama_classical_metrics::1::mae_improvement,NaN
5,1,lama_classical_metrics,outputs/metrics/classical_metrics_lama.csv,lama,classical,canonical__p001_loss_large,case_id,single_candidate,,content_region,ok,mse_improvement,6556.833107,True,higher_is_better,improvement_delta,lama_classical_metrics::1::mse_improvement,NaN
6,1,lama_classical_metrics,outputs/metrics/classical_metrics_lama.csv,lama,classical,canonical__p001_loss_large,case_id,single_candidate,,content_region,ok,psnr_improvement,19.357619,True,higher_is_better,improvement_delta,lama_classical_metrics::1::psnr_improvement,NaN
7,1,lama_classical_metrics,outputs/metrics/classical_metrics_lama.csv,lama,classical,canonical__p001_loss_large,case_id,single_candidate,,content_region,ok,ssim_improvement,0.084635,True,higher_is_better,improvement_delta,lama_classical_metrics::1::ssim_improvement,NaN
8,0,lama_classical_metrics,outputs/metrics/classical_metrics_lama.csv,lama,classical,canonical__p001_loss_large,case_id,single_candidate,,full_image,ok,mae_improvement,24.471869,True,higher_is_better,improvement_delta,lama_classical_metrics::0::mae_improvement,NaN
9,0,lama_classical_metrics,outputs/metrics/classical_metrics_lama.csv,lama,classical,canonical__p001_loss_large,case_id,single_candidate,,full_image,ok,mse_improvement,5660.389977,True,higher_is_better,improvement_delta,lama_classical_metrics::0::mse_improvement,NaN


In [13]:
# Batch 2 / Cell 4 - Validate normalized metric table
observed_metric_names = (
    sorted(batch2_metric_long_df["metric_name"].dropna().astype(str).unique().tolist())
    if len(batch2_metric_long_df)
    else []
)

unexpected_metric_names = [
    metric_name for metric_name in observed_metric_names
    if metric_name not in STRICT_METRIC_COLUMNS
]

required_metric_sources = batch2_source_summary_df.loc[
    batch2_source_summary_df["required"].astype(bool)
].copy()

failed_required_sources = required_metric_sources.loc[
    ~required_metric_sources["status"].eq("normalized")
].copy()

empty_required_sources = required_metric_sources.loc[
    required_metric_sources["normalized_rows"].fillna(0).astype(int).le(0)
].copy()

finite_metric_rows = (
    int(batch2_metric_long_df["metric_value_is_finite"].sum())
    if len(batch2_metric_long_df)
    else 0
)

duplicate_metric_record_count = (
    int(batch2_metric_long_df["metric_record_id"].duplicated().sum())
    if len(batch2_metric_long_df)
    else 0
)

missing_case_id_count = (
    int(batch2_metric_long_df["case_id"].astype(str).str.strip().eq("").sum())
    if len(batch2_metric_long_df)
    else 0
)

batch2_validation_rows = [
    validation_row(
        "metric_long_has_rows",
        len(batch2_metric_long_df),
        "> 0",
        len(batch2_metric_long_df) > 0,
        "No strict metrics were normalized.",
    ),
    validation_row(
        "required_metric_sources_normalized",
        len(required_metric_sources) - len(failed_required_sources),
        len(required_metric_sources),
        len(failed_required_sources) == 0,
        "One or more required metric sources could not be normalized.",
    ),
    validation_row(
        "required_metric_sources_non_empty",
        len(required_metric_sources) - len(empty_required_sources),
        len(required_metric_sources),
        len(empty_required_sources) == 0,
        "One or more required metric sources normalized to zero rows.",
    ),
    validation_row(
        "only_strict_metric_columns_used",
        observed_metric_names,
        STRICT_METRIC_COLUMNS,
        len(unexpected_metric_names) == 0,
        f"Unexpected metric names found: {unexpected_metric_names}",
    ),
    validation_row(
        "finite_metric_values_present",
        finite_metric_rows,
        "> 0",
        finite_metric_rows > 0,
        "No finite numeric metric values were found.",
    ),
    validation_row(
        "metric_record_ids_unique",
        duplicate_metric_record_count,
        "0",
        duplicate_metric_record_count == 0,
        "Metric record ids are not unique.",
    ),
    validation_row(
        "case_ids_present",
        missing_case_id_count,
        "0",
        missing_case_id_count == 0,
        "One or more normalized rows have missing case ids.",
    ),
]

batch2_validation_df = pd.DataFrame(batch2_validation_rows)
batch2_passed = bool(batch2_validation_df["passed"].all())

print(f"Batch 2 checks passed: {int(batch2_validation_df['passed'].sum())} / {len(batch2_validation_df)}")
display(batch2_validation_df)

if len(failed_required_sources):
    display(failed_required_sources)

if len(empty_required_sources):
    display(empty_required_sources)

Batch 2 checks passed: 7 / 7


,check_name,actual,expected,passed,failure_message
0,metric_long_has_rows,59692,> 0,True,
1,required_metric_sources_normalized,9,9,True,
2,required_metric_sources_non_empty,9,9,True,
3,only_strict_metric_columns_used,"[clip_similarity_improvement, dinov2_similarit...","[mse_improvement, mae_improvement, psnr_improv...",True,
4,finite_metric_values_present,54939,> 0,True,
5,metric_record_ids_unique,0,0,True,
6,case_ids_present,0,0,True,


In [14]:
# Batch 2 / Cell 5 - Write outputs and update manifest
batch2_metric_long_df.to_csv(BATCH2_METRIC_LONG_PATH, index=False)
batch2_validation_df.to_csv(BATCH2_VALIDATION_PATH, index=False)

stage_manifest = read_json_if_exists(STAGE_MANIFEST_PATH)
stage_manifest.update(
    {
        "notebook_id": NOTEBOOK_ID,
        "notebook_title": NOTEBOOK_TITLE,
        "stage": "batch2_normalize_strict_metrics_long_table",
        "stage_status": "passed" if batch2_passed else "failed",
        "updated_at_utc": utc_now_iso(),
        "outputs": {
            **stage_manifest.get("outputs", {}),
            "multi_model_metric_long_csv": rel(BATCH2_METRIC_LONG_PATH),
            "batch2_metric_long_validation_csv": rel(BATCH2_VALIDATION_PATH),
        },
        "batch2": {
            "metric_sources": int(len(metric_source_plan_df)),
            "normalized_rows": int(len(batch2_metric_long_df)),
            "finite_metric_rows": finite_metric_rows,
            "observed_metric_names": observed_metric_names,
            "strict_metric_columns": STRICT_METRIC_COLUMNS,
            "models": sorted(batch2_metric_long_df["model_name"].dropna().astype(str).unique().tolist())
            if len(batch2_metric_long_df)
            else [],
            "metric_families": sorted(batch2_metric_long_df["metric_family"].dropna().astype(str).unique().tolist())
            if len(batch2_metric_long_df)
            else [],
        },
    }
)

write_json(STAGE_MANIFEST_PATH, stage_manifest)

print(f"Saved metric long table: {rel(BATCH2_METRIC_LONG_PATH)}")
print(f"Saved Batch 2 validation: {rel(BATCH2_VALIDATION_PATH)}")

display(
    batch2_metric_long_df.groupby(["model_name", "metric_family", "metric_name"], dropna=False)
    .agg(
        rows=("metric_value", "size"),
        finite_rows=("metric_value_is_finite", "sum"),
        unique_cases=("case_id", "nunique"),
        unique_candidates=("candidate_id", "nunique"),
    )
    .reset_index()
)

if not batch2_passed:
    display(batch2_validation_df.loc[~batch2_validation_df["passed"], ["check_name", "actual", "expected", "failure_message"]])
    raise RuntimeError("Batch 2 validation failed. Fix metric normalization before Batch 3.")

print("Batch 2 passed. Strict metrics are normalized into the long comparison table.")

Saved metric long table: outputs/27_multi_model_comparison/metrics/multi_model_metric_long.csv
Saved Batch 2 validation: outputs/27_multi_model_comparison/validation/batch2_metric_long_validation.csv


,model_name,metric_family,metric_name,rows,finite_rows,unique_cases,unique_candidates
0,lama,classical,mae_improvement,2260,2260,410,1
1,lama,classical,mse_improvement,2260,2260,410,1
2,lama,classical,psnr_improvement,2260,2260,410,1
3,lama,classical,ssim_improvement,2260,1180,410,1
4,lama,feature_similarity,clip_similarity_improvement,1175,1175,410,1
5,lama,feature_similarity,dinov2_similarity_improvement,1175,1175,410,1
6,lama,feature_similarity,mean_similarity_improvement,1175,1175,410,1
7,lama,lpips,lpips_improvement,1175,1175,410,1
8,opencv_telea,classical,mae_improvement,2295,2295,410,1
9,opencv_telea,classical,mse_improvement,2295,2295,410,1


Batch 2 passed. Strict metrics are normalized into the long comparison table.


In [15]:
# Batch 3 / Cell 2 - Setup paths and helpers
from __future__ import annotations

import json
from typing import Any

import numpy as np
import pandas as pd

required_batch3_globals = [
    "PROJECT_ROOT",
    "OUTPUT_ROOT",
    "VALIDATION_DIR",
    "MANIFESTS_DIR",
    "STRICT_METRIC_COLUMNS",
    "batch0_source_plan_df",
    "batch1_source_frames",
    "batch2_metric_long_df",
    "rel",
    "clean_text",
    "utc_now_iso",
    "read_json_if_exists",
    "write_json",
    "validation_row",
]

missing_batch3_globals = [name for name in required_batch3_globals if name not in globals()]
if missing_batch3_globals:
    raise RuntimeError(
        "Batch 3 requires Batches 0, 1, and 2 to be run first. Missing globals: "
        + ", ".join(missing_batch3_globals)
    )

METRICS_DIR = OUTPUT_ROOT / "metrics"
ANALYSIS_DIR = OUTPUT_ROOT / "analysis"
METRICS_DIR.mkdir(parents=True, exist_ok=True)
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

BATCH3_CASE_COMPARISON_PATH = METRICS_DIR / "multi_model_case_comparison.csv"
BATCH3_SD_POLICY_PATH = ANALYSIS_DIR / "stable_diffusion_candidate_policy.csv"
BATCH3_VALIDATION_PATH = VALIDATION_DIR / "batch3_case_comparison_validation.csv"
STAGE_MANIFEST_PATH = MANIFESTS_DIR / "multi_model_comparison_stage_manifest.json"

BATCH3_CORE_MODEL_SLUGS = ["opencv_telea", "lama", "stable_diffusion", "sdxl"]

def batch3_model_slug(model_name: Any) -> str:
    text = clean_text(model_name).lower()
    return {
        "opencv": "opencv_telea",
        "opencv_telea": "opencv_telea",
        "telea": "opencv_telea",
        "lama": "lama",
        "stable diffusion": "stable_diffusion",
        "stable_diffusion": "stable_diffusion",
        "sd": "stable_diffusion",
        "sdxl": "sdxl",
    }.get(text, text.replace(" ", "_").replace("-", "_"))

def batch3_first_present_column(columns: list[str], candidates: list[str]) -> str:
    lookup = {clean_text(column).lower(): clean_text(column) for column in columns}
    for candidate in candidates:
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]
    return ""

def batch3_unique_join(values: pd.Series) -> str:
    cleaned = sorted(
        {
            clean_text(value)
            for value in values
            if clean_text(value) and clean_text(value).lower() not in {"nan", "none", "null"}
        }
    )
    return "|".join(cleaned)

def batch3_json_list(values: pd.Series) -> str:
    cleaned = sorted(
        {
            clean_text(value)
            for value in values
            if clean_text(value) and clean_text(value).lower() not in {"nan", "none", "null"}
        }
    )
    return json.dumps(cleaned)

def batch3_bool_series(series: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False).astype(bool)
    return series.astype(str).str.strip().str.lower().isin({"true", "1", "yes", "y"})

def batch3_status_priority(status: Any) -> int:
    text = clean_text(status).lower()
    if not text or text in {"nan", "none", "null"}:
        return 0
    if any(token in text for token in ["fail", "error", "timeout", "missing", "invalid", "rejected"]):
        return -10
    if any(token in text for token in ["selected", "primary", "accepted", "chosen"]):
        return 30
    if any(token in text for token in ["success", "succeeded", "complete", "completed", "ok", "pass", "valid"]):
        return 20
    return 5

def batch3_candidate_is_real(candidate_id: Any) -> bool:
    text = clean_text(candidate_id).lower()
    return bool(text and text not in {"single_candidate", "nan", "none", "null", "unknown"})

batch3_metric_long_df = batch2_metric_long_df.copy()
batch3_metric_long_df["model_slug"] = batch3_metric_long_df["model_name"].map(batch3_model_slug)
batch3_metric_long_df["candidate_id"] = batch3_metric_long_df["candidate_id"].astype(str).map(clean_text)
batch3_metric_long_df["case_id"] = batch3_metric_long_df["case_id"].astype(str).map(clean_text)
batch3_metric_long_df["evaluation_region"] = batch3_metric_long_df["evaluation_region"].astype(str).map(clean_text)
batch3_metric_long_df["metric_name"] = batch3_metric_long_df["metric_name"].astype(str).map(clean_text)
batch3_metric_long_df["metric_value"] = pd.to_numeric(batch3_metric_long_df["metric_value"], errors="coerce")
batch3_metric_long_df["metric_value_is_finite"] = np.isfinite(batch3_metric_long_df["metric_value"].to_numpy(dtype=float))

print(f"Batch 3 case comparison output: {rel(BATCH3_CASE_COMPARISON_PATH)}")
print(f"Batch 3 Stable Diffusion policy output: {rel(BATCH3_SD_POLICY_PATH)}")
print(f"Input metric rows: {len(batch3_metric_long_df):,}")
display(batch3_metric_long_df.groupby(["model_slug", "metric_family", "metric_name"], dropna=False).size().reset_index(name="rows"))

Batch 3 case comparison output: outputs/27_multi_model_comparison/metrics/multi_model_case_comparison.csv
Batch 3 Stable Diffusion policy output: outputs/27_multi_model_comparison/analysis/stable_diffusion_candidate_policy.csv
Input metric rows: 59,692


,model_slug,metric_family,metric_name,rows
0,lama,classical,mae_improvement,2260
1,lama,classical,mse_improvement,2260
2,lama,classical,psnr_improvement,2260
3,lama,classical,ssim_improvement,2260
4,lama,feature_similarity,clip_similarity_improvement,1175
5,lama,feature_similarity,dinov2_similarity_improvement,1175
6,lama,feature_similarity,mean_similarity_improvement,1175
7,lama,lpips,lpips_improvement,1175
8,opencv_telea,classical,mae_improvement,2295
9,opencv_telea,classical,mse_improvement,2295


In [16]:
# Batch 3 / Cell 3 - Build Stable Diffusion candidate policy
sd_metric_df = batch3_metric_long_df.loc[
    batch3_metric_long_df["model_slug"].eq("stable_diffusion")
].copy()

sd_metric_df["candidate_is_real"] = sd_metric_df["candidate_id"].map(batch3_candidate_is_real)

sd_candidate_metric_coverage_df = (
    sd_metric_df.loc[sd_metric_df["candidate_is_real"]]
    .groupby(["case_id", "candidate_id"], dropna=False)
    .agg(
        source_metric_rows=("metric_value", "size"),
        finite_metric_rows=("metric_value_is_finite", "sum"),
        metric_name_count=("metric_name", "nunique"),
        evaluation_region_count=("evaluation_region", "nunique"),
        metric_names=("metric_name", batch3_json_list),
        evaluation_regions=("evaluation_region", batch3_json_list),
        source_ids=("source_id", batch3_json_list),
    )
    .reset_index()
)

sd_audit_source_df = batch1_source_frames.get("stable_diffusion_report_audit", pd.DataFrame()).copy()
sd_audit_candidates_df = pd.DataFrame()

if len(sd_audit_source_df):
    sd_audit_source_df.columns = [clean_text(column) for column in sd_audit_source_df.columns]
    audit_columns = list(sd_audit_source_df.columns)

    audit_case_col = batch3_first_present_column(
        audit_columns,
        ["case_id", "canonical_case_id", "report_source_case_id", "source_case_id", "sample_id", "image_id"],
    )
    audit_candidate_col = batch3_first_present_column(
        audit_columns,
        ["report_candidate_id", "stable_diffusion_candidate_id", "candidate_id", "restoration_candidate_id"],
    )
    audit_status_col = batch3_first_present_column(
        audit_columns,
        ["batch3_candidate_status", "candidate_status", "status", "restoration_status", "metric_status"],
    )
    audit_order_col = batch3_first_present_column(
        audit_columns,
        ["candidate_rank", "report_candidate_rank", "selection_rank", "rank", "candidate_index"],
    )
    audit_path_col = batch3_first_present_column(
        audit_columns,
        ["primary_restored_candidate_path", "restored_path", "restored_image_path", "candidate_restored_path"],
    )

    if audit_case_col and audit_candidate_col:
        sd_audit_candidates_df = pd.DataFrame(
            {
                "case_id": sd_audit_source_df[audit_case_col].astype(str).map(clean_text),
                "candidate_id": sd_audit_source_df[audit_candidate_col].astype(str).map(clean_text),
                "audit_status": (
                    sd_audit_source_df[audit_status_col].astype(str).map(clean_text)
                    if audit_status_col
                    else ""
                ),
                "audit_order": (
                    pd.to_numeric(sd_audit_source_df[audit_order_col], errors="coerce")
                    if audit_order_col
                    else np.nan
                ),
                "audit_restored_path": (
                    sd_audit_source_df[audit_path_col].astype(str).map(clean_text)
                    if audit_path_col
                    else ""
                ),
                "audit_case_id_column": audit_case_col,
                "audit_candidate_id_column": audit_candidate_col,
                "audit_status_column": audit_status_col,
            }
        )

        sd_audit_candidates_df = sd_audit_candidates_df.loc[
            sd_audit_candidates_df["case_id"].ne("")
            & sd_audit_candidates_df["candidate_id"].map(batch3_candidate_is_real)
        ].copy()

        sd_audit_candidates_df = (
            sd_audit_candidates_df
            .groupby(["case_id", "candidate_id"], dropna=False)
            .agg(
                audit_status=("audit_status", batch3_unique_join),
                audit_order=("audit_order", "min"),
                audit_restored_path=("audit_restored_path", batch3_unique_join),
                audit_case_id_column=("audit_case_id_column", "first"),
                audit_candidate_id_column=("audit_candidate_id_column", "first"),
                audit_status_column=("audit_status_column", "first"),
            )
            .reset_index()
        )
        sd_audit_candidates_df["audit_observed"] = True

if len(sd_candidate_metric_coverage_df) and len(sd_audit_candidates_df):
    sd_candidate_universe_df = sd_candidate_metric_coverage_df.merge(
        sd_audit_candidates_df,
        on=["case_id", "candidate_id"],
        how="outer",
    )
elif len(sd_candidate_metric_coverage_df):
    sd_candidate_universe_df = sd_candidate_metric_coverage_df.copy()
    sd_candidate_universe_df["audit_observed"] = False
    sd_candidate_universe_df["audit_status"] = ""
    sd_candidate_universe_df["audit_order"] = np.nan
    sd_candidate_universe_df["audit_restored_path"] = ""
elif len(sd_audit_candidates_df):
    sd_candidate_universe_df = sd_audit_candidates_df.copy()
else:
    sd_candidate_universe_df = pd.DataFrame()

coverage_defaults = {
    "source_metric_rows": 0,
    "finite_metric_rows": 0,
    "metric_name_count": 0,
    "evaluation_region_count": 0,
    "metric_names": "[]",
    "evaluation_regions": "[]",
    "source_ids": "[]",
}

for column, default in coverage_defaults.items():
    if column not in sd_candidate_universe_df.columns:
        sd_candidate_universe_df[column] = default
    sd_candidate_universe_df[column] = sd_candidate_universe_df[column].fillna(default)

for column in ["audit_observed", "audit_status", "audit_order", "audit_restored_path"]:
    if column not in sd_candidate_universe_df.columns:
        sd_candidate_universe_df[column] = False if column == "audit_observed" else ""

if len(sd_candidate_universe_df):
    sd_candidate_universe_df["audit_observed"] = sd_candidate_universe_df["audit_observed"].fillna(False).astype(bool)
    sd_candidate_universe_df["audit_status"] = sd_candidate_universe_df["audit_status"].fillna("").astype(str)
    sd_candidate_universe_df["audit_status_priority"] = sd_candidate_universe_df["audit_status"].map(batch3_status_priority)
    sd_candidate_universe_df["finite_metric_rows"] = pd.to_numeric(
        sd_candidate_universe_df["finite_metric_rows"], errors="coerce"
    ).fillna(0).astype(int)
    sd_candidate_universe_df["metric_name_count"] = pd.to_numeric(
        sd_candidate_universe_df["metric_name_count"], errors="coerce"
    ).fillna(0).astype(int)
    sd_candidate_universe_df["evaluation_region_count"] = pd.to_numeric(
        sd_candidate_universe_df["evaluation_region_count"], errors="coerce"
    ).fillna(0).astype(int)
    sd_candidate_universe_df["audit_order_sort"] = pd.to_numeric(
        sd_candidate_universe_df["audit_order"], errors="coerce"
    ).fillna(1_000_000)
    sd_candidate_universe_df["candidate_usable"] = (
        sd_candidate_universe_df["finite_metric_rows"].gt(0)
        & sd_candidate_universe_df["audit_status_priority"].ge(0)
    )

    sd_selected_candidates_df = (
        sd_candidate_universe_df.sort_values(
            [
                "case_id",
                "candidate_usable",
                "finite_metric_rows",
                "metric_name_count",
                "evaluation_region_count",
                "audit_status_priority",
                "audit_order_sort",
                "candidate_id",
            ],
            ascending=[True, False, False, False, False, False, True, True],
            kind="mergesort",
        )
        .drop_duplicates("case_id", keep="first")
        .copy()
    )

    sd_selected_candidates_df["selected_candidate_id"] = sd_selected_candidates_df["candidate_id"]
    sd_selected_candidates_df["selection_policy"] = (
        "max_finite_metric_coverage_then_metric_count_then_audit_status_then_candidate_id"
    )
    sd_selected_candidates_df["selected_reason"] = (
        "Selected without using metric values; finite coverage and audit/status metadata only."
    )
else:
    sd_selected_candidates_df = pd.DataFrame()

sd_all_case_ids = sorted(sd_metric_df["case_id"].dropna().astype(str).map(clean_text).unique().tolist())
selected_case_ids = set(sd_selected_candidates_df["case_id"].astype(str).tolist()) if len(sd_selected_candidates_df) else set()
case_level_only_case_ids = [case_id for case_id in sd_all_case_ids if case_id not in selected_case_ids]

sd_case_level_rows = []
if case_level_only_case_ids:
    sd_case_level_coverage_df = (
        sd_metric_df.loc[sd_metric_df["case_id"].isin(case_level_only_case_ids)]
        .groupby("case_id", dropna=False)
        .agg(
            source_metric_rows=("metric_value", "size"),
            finite_metric_rows=("metric_value_is_finite", "sum"),
            metric_name_count=("metric_name", "nunique"),
            evaluation_region_count=("evaluation_region", "nunique"),
            metric_names=("metric_name", batch3_json_list),
            evaluation_regions=("evaluation_region", batch3_json_list),
            source_ids=("source_id", batch3_json_list),
        )
        .reset_index()
    )

    for _, row in sd_case_level_coverage_df.iterrows():
        sd_case_level_rows.append(
            {
                "case_id": clean_text(row["case_id"]),
                "candidate_id": "single_candidate",
                "selected_candidate_id": "single_candidate",
                "selection_policy": "case_level_metric_no_candidate_dimension",
                "selected_reason": "Stable Diffusion rows for this case did not expose candidate ids.",
                "source_metric_rows": int(row["source_metric_rows"]),
                "finite_metric_rows": int(row["finite_metric_rows"]),
                "metric_name_count": int(row["metric_name_count"]),
                "evaluation_region_count": int(row["evaluation_region_count"]),
                "metric_names": row["metric_names"],
                "evaluation_regions": row["evaluation_regions"],
                "source_ids": row["source_ids"],
                "audit_observed": False,
                "audit_status": "",
                "audit_status_priority": 0,
                "audit_order": np.nan,
                "audit_restored_path": "",
                "candidate_usable": True,
            }
        )

sd_candidate_policy_df = pd.concat(
    [
        sd_selected_candidates_df,
        pd.DataFrame(sd_case_level_rows),
    ],
    ignore_index=True,
    sort=False,
)

if len(sd_candidate_policy_df):
    candidate_counts_df = (
        sd_candidate_universe_df.groupby("case_id", dropna=False)["candidate_id"]
        .nunique()
        .reset_index(name="candidate_count_observed")
        if len(sd_candidate_universe_df)
        else pd.DataFrame({"case_id": sd_candidate_policy_df["case_id"], "candidate_count_observed": 1})
    )

    sd_candidate_policy_df = sd_candidate_policy_df.merge(candidate_counts_df, on="case_id", how="left")
    sd_candidate_policy_df["candidate_count_observed"] = (
        sd_candidate_policy_df["candidate_count_observed"].fillna(1).astype(int)
    )

    policy_columns = [
        "case_id",
        "selected_candidate_id",
        "selection_policy",
        "selected_reason",
        "candidate_count_observed",
        "finite_metric_rows",
        "metric_name_count",
        "evaluation_region_count",
        "metric_names",
        "evaluation_regions",
        "source_ids",
        "audit_observed",
        "audit_status",
        "audit_status_priority",
        "audit_order",
        "audit_restored_path",
        "candidate_usable",
    ]
    sd_candidate_policy_df = sd_candidate_policy_df[
        [column for column in policy_columns if column in sd_candidate_policy_df.columns]
    ].sort_values("case_id", kind="mergesort").reset_index(drop=True)

print(f"Stable Diffusion metric cases: {len(sd_all_case_ids):,}")
print(f"Stable Diffusion policy rows: {len(sd_candidate_policy_df):,}")
display(sd_candidate_policy_df.head(20))

Stable Diffusion metric cases: 465
Stable Diffusion policy rows: 465


E:\HFCache\tmp\ipykernel_11960\3161810185.py:134: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  sd_candidate_universe_df["audit_observed"] = sd_candidate_universe_df["audit_observed"].fillna(False).astype(bool)


,case_id,selected_candidate_id,selection_policy,selected_reason,candidate_count_observed,finite_metric_rows,metric_name_count,evaluation_region_count,metric_names,evaluation_regions,source_ids,audit_observed,audit_status,audit_status_priority,audit_order,audit_restored_path,candidate_usable
0,p001__blur__mild,sd__syn__p001__blur__p00_generic__s2026__fd770...,max_finite_metric_coverage_then_metric_count_t...,Selected without using metric values; finite c...,2,24,5,6,"[""lpips_improvement"", ""mae_improvement"", ""mse_...","[""boundary_region"", ""content_region"", ""full_im...","[""stable_diffusion_classical_metrics"", ""stable...",False,,0,NaN,NaN,True
1,p001__blur__moderate,sd__syn__p001__blur__p00_generic__s2026__bcd43...,max_finite_metric_coverage_then_metric_count_t...,Selected without using metric values; finite c...,2,24,5,6,"[""lpips_improvement"", ""mae_improvement"", ""mse_...","[""boundary_region"", ""content_region"", ""full_im...","[""stable_diffusion_classical_metrics"", ""stable...",False,,0,NaN,NaN,True
2,p001__blur__severe,sd__syn__p001__blur__p00_generic__s2026__a0008...,max_finite_metric_coverage_then_metric_count_t...,Selected without using metric values; finite c...,10,24,5,6,"[""lpips_improvement"", ""mae_improvement"", ""mse_...","[""boundary_region"", ""content_region"", ""full_im...","[""stable_diffusion_classical_metrics"", ""stable...",False,,0,NaN,NaN,True
3,p001__blur_fading__moderate,sd__syn__p001__blur_fading__p00_generic__s2026...,max_finite_metric_coverage_then_metric_count_t...,Selected without using metric values; finite c...,10,24,5,6,"[""lpips_improvement"", ""mae_improvement"", ""mse_...","[""boundary_region"", ""content_region"", ""full_im...","[""stable_diffusion_classical_metrics"", ""stable...",False,,0,NaN,NaN,True
4,p001__dirt_dust__mild,sd__syn__p001__dirt_dust__p00_generic__s2026__...,max_finite_metric_coverage_then_metric_count_t...,Selected without using metric values; finite c...,2,24,5,6,"[""lpips_improvement"", ""mae_improvement"", ""mse_...","[""boundary_region"", ""content_region"", ""full_im...","[""stable_diffusion_classical_metrics"", ""stable...",False,,0,NaN,NaN,True
5,p001__dirt_dust__moderate,sd__syn__p001__dirt_dust__p00_generic__s2026__...,max_finite_metric_coverage_then_metric_count_t...,Selected without using metric values; finite c...,2,24,5,6,"[""lpips_improvement"", ""mae_improvement"", ""mse_...","[""boundary_region"", ""content_region"", ""full_im...","[""stable_diffusion_classical_metrics"", ""stable...",False,,0,NaN,NaN,True
6,p001__dirt_dust__severe,sd__syn__p001__dirt_dust__p00_generic__s2026__...,max_finite_metric_coverage_then_metric_count_t...,Selected without using metric values; finite c...,10,24,5,6,"[""lpips_improvement"", ""mae_improvement"", ""mse_...","[""boundary_region"", ""content_region"", ""full_im...","[""stable_diffusion_classical_metrics"", ""stable...",False,,0,NaN,NaN,True
7,p001__discolouration__mild,sd__syn__p001__discolouration__p00_generic__s2...,max_finite_metric_coverage_then_metric_count_t...,Selected without using metric values; finite c...,2,24,5,6,"[""lpips_improvement"", ""mae_improvement"", ""mse_...","[""boundary_region"", ""content_region"", ""full_im...","[""stable_diffusion_classical_metrics"", ""stable...",False,,0,NaN,NaN,True
8,p001__discolouration__moderate,sd__syn__p001__discolouration__p00_generic__s2...,max_finite_metric_coverage_then_metric_count_t...,Selected without using metric values; finite c...,2,24,5,6,"[""lpips_improvement"", ""mae_improvement"", ""mse_...","[""boundary_region"", ""content_region"", ""full_im...","[""stable_diffusion_classical_metrics"", ""stable...",False,,0,NaN,NaN,True
9,p001__discolouration__severe,sd__syn__p001__discolouration__p00_generic__s2...,max_finite_metric_coverage_then_metric_count_t...,Selected without using metric values; finite c...,10,24,5,6,"[""lpips_improvement"", ""mae_improvement"", ""mse_...","[""boundary_region"", ""content_region"", ""full_im...","[""stable_diffus

In [17]:
# Batch 3 / Cell 4 - Apply candidate policy and build paired case table
policy_lookup_df = (
    sd_candidate_policy_df[["case_id", "selected_candidate_id", "selection_policy", "candidate_count_observed"]].copy()
    if len(sd_candidate_policy_df)
    else pd.DataFrame(columns=["case_id", "selected_candidate_id", "selection_policy", "candidate_count_observed"])
)

batch3_policy_metric_df = batch3_metric_long_df.merge(
    policy_lookup_df,
    on="case_id",
    how="left",
)

batch3_policy_metric_df["is_stable_diffusion"] = batch3_policy_metric_df["model_slug"].eq("stable_diffusion")
batch3_policy_metric_df["candidate_is_real"] = batch3_policy_metric_df["candidate_id"].map(batch3_candidate_is_real)

batch3_policy_metric_df["candidate_policy_keep"] = (
    ~batch3_policy_metric_df["is_stable_diffusion"]
    | ~batch3_policy_metric_df["candidate_is_real"]
    | batch3_policy_metric_df["candidate_id"].eq(batch3_policy_metric_df["selected_candidate_id"])
)

batch3_selected_metric_long_df = batch3_policy_metric_df.loc[
    batch3_policy_metric_df["candidate_policy_keep"]
].copy()

batch3_finite_selected_metric_df = batch3_selected_metric_long_df.loc[
    batch3_selected_metric_long_df["metric_value_is_finite"]
].copy()

batch3_metric_model_df = (
    batch3_finite_selected_metric_df
    .groupby(["case_id", "evaluation_region", "metric_name", "model_slug"], dropna=False)
    .agg(
        metric_value=("metric_value", "mean"),
        metric_value_median=("metric_value", "median"),
        finite_metric_rows=("metric_value", "size"),
        candidate_ids=("candidate_id", batch3_unique_join),
        source_ids=("source_id", batch3_json_list),
    )
    .reset_index()
)

case_index_columns = ["case_id", "evaluation_region", "metric_name"]

value_wide_df = (
    batch3_metric_model_df.pivot_table(
        index=case_index_columns,
        columns="model_slug",
        values="metric_value",
        aggfunc="first",
    )
    .reset_index()
)
value_wide_df.columns = [
    column if isinstance(column, str) else str(column)
    for column in value_wide_df.columns
]
value_wide_df = value_wide_df.rename(
    columns={
        column: f"metric_value__{column}"
        for column in value_wide_df.columns
        if column not in case_index_columns
    }
)

count_wide_df = (
    batch3_metric_model_df.pivot_table(
        index=case_index_columns,
        columns="model_slug",
        values="finite_metric_rows",
        aggfunc="sum",
        fill_value=0,
    )
    .reset_index()
)
count_wide_df.columns = [
    column if isinstance(column, str) else str(column)
    for column in count_wide_df.columns
]
count_wide_df = count_wide_df.rename(
    columns={
        column: f"finite_rows__{column}"
        for column in count_wide_df.columns
        if column not in case_index_columns
    }
)

batch3_case_comparison_df = value_wide_df.merge(
    count_wide_df,
    on=case_index_columns,
    how="left",
)

observed_model_slugs = sorted(batch3_metric_model_df["model_slug"].dropna().astype(str).unique().tolist())
model_slugs_for_columns = list(dict.fromkeys(BATCH3_CORE_MODEL_SLUGS + observed_model_slugs))

for model_slug in model_slugs_for_columns:
    value_column = f"metric_value__{model_slug}"
    count_column = f"finite_rows__{model_slug}"
    has_column = f"has__{model_slug}"

    if value_column not in batch3_case_comparison_df.columns:
        batch3_case_comparison_df[value_column] = np.nan
    if count_column not in batch3_case_comparison_df.columns:
        batch3_case_comparison_df[count_column] = 0

    batch3_case_comparison_df[count_column] = (
        pd.to_numeric(batch3_case_comparison_df[count_column], errors="coerce").fillna(0).astype(int)
    )
    batch3_case_comparison_df[has_column] = batch3_case_comparison_df[value_column].notna()

has_columns = [f"has__{model_slug}" for model_slug in model_slugs_for_columns]
batch3_case_comparison_df["paired_model_count"] = batch3_case_comparison_df[has_columns].sum(axis=1).astype(int)

batch3_case_comparison_df["paired__opencv_telea__lama"] = (
    batch3_case_comparison_df["has__opencv_telea"] & batch3_case_comparison_df["has__lama"]
)
batch3_case_comparison_df["paired__opencv_telea__stable_diffusion"] = (
    batch3_case_comparison_df["has__opencv_telea"] & batch3_case_comparison_df["has__stable_diffusion"]
)
batch3_case_comparison_df["paired__lama__stable_diffusion"] = (
    batch3_case_comparison_df["has__lama"] & batch3_case_comparison_df["has__stable_diffusion"]
)
batch3_case_comparison_df["paired__all_core_models"] = (
    batch3_case_comparison_df["has__opencv_telea"]
    & batch3_case_comparison_df["has__lama"]
    & batch3_case_comparison_df["has__stable_diffusion"]
)

batch3_case_comparison_df["delta__lama_minus_opencv_telea"] = (
    batch3_case_comparison_df["metric_value__lama"] - batch3_case_comparison_df["metric_value__opencv_telea"]
)
batch3_case_comparison_df["delta__stable_diffusion_minus_opencv_telea"] = (
    batch3_case_comparison_df["metric_value__stable_diffusion"] - batch3_case_comparison_df["metric_value__opencv_telea"]
)
batch3_case_comparison_df["delta__stable_diffusion_minus_lama"] = (
    batch3_case_comparison_df["metric_value__stable_diffusion"] - batch3_case_comparison_df["metric_value__lama"]
)

sd_policy_case_df = sd_candidate_policy_df.rename(
    columns={
        "selected_candidate_id": "stable_diffusion_selected_candidate_id",
        "selection_policy": "stable_diffusion_candidate_selection_policy",
        "candidate_count_observed": "stable_diffusion_candidate_count_observed",
        "selected_reason": "stable_diffusion_candidate_selected_reason",
    }
)

sd_policy_keep_columns = [
    "case_id",
    "stable_diffusion_selected_candidate_id",
    "stable_diffusion_candidate_selection_policy",
    "stable_diffusion_candidate_selected_reason",
    "stable_diffusion_candidate_count_observed",
]

batch3_case_comparison_df = batch3_case_comparison_df.merge(
    sd_policy_case_df[[column for column in sd_policy_keep_columns if column in sd_policy_case_df.columns]],
    on="case_id",
    how="left",
)

preferred_columns = (
    case_index_columns
    + [
        "paired_model_count",
        "paired__opencv_telea__lama",
        "paired__opencv_telea__stable_diffusion",
        "paired__lama__stable_diffusion",
        "paired__all_core_models",
        "stable_diffusion_selected_candidate_id",
        "stable_diffusion_candidate_selection_policy",
        "stable_diffusion_candidate_selected_reason",
        "stable_diffusion_candidate_count_observed",
    ]
    + [f"has__{model_slug}" for model_slug in model_slugs_for_columns]
    + [f"metric_value__{model_slug}" for model_slug in model_slugs_for_columns]
    + [f"finite_rows__{model_slug}" for model_slug in model_slugs_for_columns]
    + [
        "delta__lama_minus_opencv_telea",
        "delta__stable_diffusion_minus_opencv_telea",
        "delta__stable_diffusion_minus_lama",
    ]
)

batch3_case_comparison_df = batch3_case_comparison_df[
    [column for column in preferred_columns if column in batch3_case_comparison_df.columns]
].sort_values(case_index_columns, kind="mergesort").reset_index(drop=True)

print(f"Selected metric rows after SD candidate policy: {len(batch3_selected_metric_long_df):,}")
print(f"Paired case comparison rows: {len(batch3_case_comparison_df):,}")
display(batch3_case_comparison_df.head(20))

Selected metric rows after SD candidate policy: 47,492
Paired case comparison rows: 27,285


,case_id,evaluation_region,metric_name,paired_model_count,paired__opencv_telea__lama,paired__opencv_telea__stable_diffusion,paired__lama__stable_diffusion,paired__all_core_models,stable_diffusion_selected_candidate_id,stable_diffusion_candidate_selection_policy,...,metric_value__lama,metric_value__stable_diffusion,metric_value__sdxl,finite_rows__opencv_telea,finite_rows__lama,finite_rows__stable_diffusion,finite_rows__sdxl,delta__lama_minus_opencv_telea,delta__stable_diffusion_minus_opencv_telea,delta__stable_diffusion_minus_lama
0,canonical__p001_loss_large,boundary_region,mae_improvement,2,True,False,False,False,NaN,NaN,...,110.674597,NaN,NaN,1,1,0,0,165.609453,NaN,NaN
1,canonical__p001_loss_large,boundary_region,mse_improvement,2,True,False,False,False,NaN,NaN,...,25396.024811,NaN,NaN,1,1,0,0,25342.642293,NaN,NaN
2,canonical__p001_loss_large,boundary_region,psnr_improvement,2,True,False,False,False,NaN,NaN,...,35.140351,NaN,NaN,1,1,0,0,25.296371,NaN,NaN
3,canonical__p001_loss_large,content_region,clip_similarity_improvement,2,True,False,False,False,NaN,NaN,...,0.148588,NaN,NaN,1,1,0,0,0.134819,NaN,NaN
4,canonical__p001_loss_large,content_region,dinov2_similarity_improvement,2,True,False,False,False,NaN,NaN,...,0.066908,NaN,NaN,1,1,0,0,0.102175,NaN,NaN
5,canonical__p001_loss_large,content_region,lpips_improvement,2,True,False,False,False,NaN,NaN,...,0.251112,NaN,NaN,1,1,0,0,0.020350,NaN,NaN
6,canonical__p001_loss_large,content_region,mae_improvement,2,True,False,False,False,NaN,NaN,...,28.347510,NaN,NaN,1,1,0,0,45.970711,NaN,NaN
7,canonical__p001_loss_large,content_region,mean_similarity_improvement,2,True,False,False,False,NaN,NaN,...,0.107748,NaN,NaN,1,1,0,0,0.118497,NaN,NaN
8,canonical__p001_loss_large,content_region,mse_improvement,2,True,False,False,False,NaN,NaN,...,6556.833107,NaN,NaN,1,1,0,0,6547.611486,NaN,NaN
9,canonical__p001_loss_large,content_region,psnr_improvement,2,True,False,False,False,NaN,NaN,...,19.357619,NaN,NaN,1,1,0,0,15.765115,NaN,NaN


In [18]:
# Batch 3 / Cell 5 - Validate Batch 3 outputs
observed_metric_names = sorted(batch3_metric_model_df["metric_name"].dropna().astype(str).unique().tolist())
unexpected_metric_names = [
    metric_name for metric_name in observed_metric_names
    if metric_name not in STRICT_METRIC_COLUMNS
]

case_key_duplicate_count = int(
    batch3_case_comparison_df.duplicated(case_index_columns).sum()
) if len(batch3_case_comparison_df) else 0

sd_policy_duplicate_count = int(
    sd_candidate_policy_df["case_id"].duplicated().sum()
) if len(sd_candidate_policy_df) else 0

sd_candidate_rows_after_policy_df = batch3_selected_metric_long_df.loc[
    batch3_selected_metric_long_df["model_slug"].eq("stable_diffusion")
    & batch3_selected_metric_long_df["candidate_id"].map(batch3_candidate_is_real)
].copy()

sd_unselected_candidate_rows_after_policy = 0
if len(sd_candidate_rows_after_policy_df):
    sd_unselected_candidate_rows_after_policy = int(
        (
            ~sd_candidate_rows_after_policy_df["candidate_id"].eq(
                sd_candidate_rows_after_policy_df["selected_candidate_id"]
            )
        ).sum()
    )

sd_input_has_rows = len(sd_metric_df) > 0
sd_policy_has_required_rows = (not sd_input_has_rows) or len(sd_candidate_policy_df) > 0

selected_metric_rows = int(len(batch3_selected_metric_long_df))
finite_selected_metric_rows = int(len(batch3_finite_selected_metric_df))

batch3_validation_rows = [
    validation_row(
        "stable_diffusion_policy_available",
        len(sd_candidate_policy_df),
        "> 0 when Stable Diffusion metrics exist",
        sd_policy_has_required_rows,
        "Stable Diffusion metrics exist, but no candidate policy was built.",
    ),
    validation_row(
        "stable_diffusion_policy_unique_case_ids",
        sd_policy_duplicate_count,
        "0",
        sd_policy_duplicate_count == 0,
        "Stable Diffusion candidate policy contains duplicate case ids.",
    ),
    validation_row(
        "stable_diffusion_unselected_candidates_removed",
        sd_unselected_candidate_rows_after_policy,
        "0",
        sd_unselected_candidate_rows_after_policy == 0,
        "Unselected Stable Diffusion candidate rows remain after policy filtering.",
    ),
    validation_row(
        "selected_metric_rows_present",
        selected_metric_rows,
        "> 0",
        selected_metric_rows > 0,
        "No metric rows remain after candidate policy filtering.",
    ),
    validation_row(
        "finite_selected_metric_rows_present",
        finite_selected_metric_rows,
        "> 0",
        finite_selected_metric_rows > 0,
        "No finite metric rows remain after candidate policy filtering.",
    ),
    validation_row(
        "case_comparison_has_rows",
        len(batch3_case_comparison_df),
        "> 0",
        len(batch3_case_comparison_df) > 0,
        "Paired case comparison table is empty.",
    ),
    validation_row(
        "case_comparison_unique_keys",
        case_key_duplicate_count,
        "0",
        case_key_duplicate_count == 0,
        "Paired case comparison table has duplicate case/evaluation_region/metric_name rows.",
    ),
    validation_row(
        "only_strict_metrics_in_case_table",
        observed_metric_names,
        STRICT_METRIC_COLUMNS,
        len(unexpected_metric_names) == 0,
        f"Unexpected metric names found: {unexpected_metric_names}",
    ),
]

batch3_validation_df = pd.DataFrame(batch3_validation_rows)
batch3_passed = bool(batch3_validation_df["passed"].all())

print(f"Batch 3 checks passed: {int(batch3_validation_df['passed'].sum())} / {len(batch3_validation_df)}")
display(batch3_validation_df)

if not batch3_passed:
    display(batch3_validation_df.loc[~batch3_validation_df["passed"], ["check_name", "actual", "expected", "failure_message"]])

Batch 3 checks passed: 8 / 8


,check_name,actual,expected,passed,failure_message
0,stable_diffusion_policy_available,465,> 0 when Stable Diffusion metrics exist,True,
1,stable_diffusion_policy_unique_case_ids,0,0,True,
2,stable_diffusion_unselected_candidates_removed,0,0,True,
3,selected_metric_rows_present,47492,> 0,True,
4,finite_selected_metric_rows_present,44047,> 0,True,
5,case_comparison_has_rows,27285,> 0,True,
6,case_comparison_unique_keys,0,0,True,
7,only_strict_metrics_in_case_table,"[clip_similarity_improvement, dinov2_similarit...","[mse_improvement, mae_improvement, psnr_improv...",True,


In [19]:
# Batch 3 / Cell 6 - Write outputs and update manifest
sd_candidate_policy_df.to_csv(BATCH3_SD_POLICY_PATH, index=False)
batch3_case_comparison_df.to_csv(BATCH3_CASE_COMPARISON_PATH, index=False)
batch3_validation_df.to_csv(BATCH3_VALIDATION_PATH, index=False)

stage_manifest = read_json_if_exists(STAGE_MANIFEST_PATH)
stage_manifest.update(
    {
        "notebook_id": globals().get("NOTEBOOK_ID", "27_multi_model_comparison"),
        "notebook_title": globals().get("NOTEBOOK_TITLE", "Multi-model comparison"),
        "stage": "batch3_case_comparison_and_stable_diffusion_candidate_policy",
        "stage_status": "passed" if batch3_passed else "failed",
        "updated_at_utc": utc_now_iso(),
        "outputs": {
            **stage_manifest.get("outputs", {}),
            "multi_model_case_comparison_csv": rel(BATCH3_CASE_COMPARISON_PATH),
            "stable_diffusion_candidate_policy_csv": rel(BATCH3_SD_POLICY_PATH),
            "batch3_case_comparison_validation_csv": rel(BATCH3_VALIDATION_PATH),
        },
        "batch3": {
            "stable_diffusion_policy_rows": int(len(sd_candidate_policy_df)),
            "selected_metric_rows": int(len(batch3_selected_metric_long_df)),
            "finite_selected_metric_rows": int(len(batch3_finite_selected_metric_df)),
            "case_comparison_rows": int(len(batch3_case_comparison_df)),
            "paired_all_core_model_rows": int(batch3_case_comparison_df["paired__all_core_models"].sum())
            if len(batch3_case_comparison_df)
            else 0,
            "observed_model_slugs": observed_model_slugs,
            "observed_metric_names": observed_metric_names,
            "candidate_policy_note": (
                "Stable Diffusion candidates selected by deterministic coverage/status policy; "
                "metric values are not used for candidate selection."
            ),
        },
    }
)

write_json(STAGE_MANIFEST_PATH, stage_manifest)

print(f"Saved Stable Diffusion candidate policy: {rel(BATCH3_SD_POLICY_PATH)}")
print(f"Saved paired case comparison: {rel(BATCH3_CASE_COMPARISON_PATH)}")
print(f"Saved Batch 3 validation: {rel(BATCH3_VALIDATION_PATH)}")

display(
    batch3_case_comparison_df.groupby("metric_name", dropna=False)
    .agg(
        rows=("case_id", "size"),
        paired_opencv_lama=("paired__opencv_telea__lama", "sum"),
        paired_opencv_sd=("paired__opencv_telea__stable_diffusion", "sum"),
        paired_lama_sd=("paired__lama__stable_diffusion", "sum"),
        paired_all_core=("paired__all_core_models", "sum"),
    )
    .reset_index()
)

if not batch3_passed:
    raise RuntimeError("Batch 3 validation failed. Fix candidate policy or pairing before Batch 4.")

print("Batch 3 passed. Paired case comparison and Stable Diffusion candidate policy are ready.")

Saved Stable Diffusion candidate policy: outputs/27_multi_model_comparison/analysis/stable_diffusion_candidate_policy.csv
Saved paired case comparison: outputs/27_multi_model_comparison/metrics/multi_model_case_comparison.csv
Saved Batch 3 validation: outputs/27_multi_model_comparison/validation/batch3_case_comparison_validation.csv


,metric_name,rows,paired_opencv_lama,paired_opencv_sd,paired_lama_sd,paired_all_core
0,clip_similarity_improvement,2515,1175,0,0,0
1,dinov2_similarity_improvement,2515,1175,0,0,0
2,lpips_improvement,2515,1175,0,0,0
3,mae_improvement,4900,2245,0,0,0
4,mean_similarity_improvement,2515,1175,0,0,0
5,mse_improvement,4900,2245,0,0,0
6,psnr_improvement,4900,2245,0,0,0
7,ssim_improvement,2525,1175,0,0,0


Batch 3 passed. Paired case comparison and Stable Diffusion candidate policy are ready.


In [20]:
# Batch 4 / Cell 2 - Setup paths and helpers
from __future__ import annotations

import json
from typing import Any

import numpy as np
import pandas as pd

required_batch4_globals = [
    "OUTPUT_ROOT",
    "VALIDATION_DIR",
    "MANIFESTS_DIR",
    "STRICT_METRIC_COLUMNS",
    "batch3_selected_metric_long_df",
    "batch3_case_comparison_df",
    "batch1_source_frames",
    "rel",
    "clean_text",
    "utc_now_iso",
    "read_json_if_exists",
    "write_json",
    "validation_row",
]

missing_batch4_globals = [name for name in required_batch4_globals if name not in globals()]
if missing_batch4_globals:
    raise RuntimeError(
        "Batch 4 requires Batches 1, 2, and 3 to be run first. Missing globals: "
        + ", ".join(missing_batch4_globals)
    )

ANALYSIS_DIR = OUTPUT_ROOT / "analysis"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

BATCH4_SUMMARY_TABLES_PATH = ANALYSIS_DIR / "multi_model_summary_tables.csv"
BATCH4_VALIDATION_PATH = VALIDATION_DIR / "batch4_summary_tables_validation.csv"
STAGE_MANIFEST_PATH = MANIFESTS_DIR / "multi_model_comparison_stage_manifest.json"

BATCH4_CASE_ID_COLUMNS = [
    "case_id",
    "canonical_case_id",
    "canonical_id",
    "sample_id",
    "image_id",
    "source_case_id",
    "report_source_case_id",
]

BATCH4_CANDIDATE_ID_COLUMNS = [
    "candidate_id",
    "report_candidate_id",
    "stable_diffusion_candidate_id",
    "restoration_candidate_id",
]

BATCH4_DIMENSION_CANDIDATES = {
    "style_category": [
        "style_category",
        "style",
        "painting_style",
        "art_style",
        "genre",
        "school",
        "period",
        "movement",
    ],
    "content_category": [
        "content_category",
        "category",
        "object_category",
        "semantic_category",
        "subject_category",
        "dataset_category",
        "image_category",
    ],
    "mask_damage_type": [
        "mask_damage_type",
        "damage_type",
        "mask_type",
        "variant_type",
        "damage_variant",
        "mask_variant",
        "damage_label",
        "mask_label",
    ],
    "degradation_type": [
        "degradation_type",
        "synthetic_degradation_type",
        "corruption_type",
        "degradation_variant",
        "prompt_ablation_group",
        "prompt_ablation_subset",
    ],
}

BATCH4_DAMAGE_PERCENT_COLUMNS = [
    "damage_percent",
    "damage_percentage",
    "damage_pct",
    "damaged_area_percent",
    "damaged_area_pct",
    "mask_area_percent",
    "mask_coverage_percent",
    "mask_coverage_pct",
    "damage_ratio",
    "damage_fraction",
    "mask_area_ratio",
    "mask_fraction",
]

BATCH4_RUNTIME_COLUMNS = [
    "runtime_seconds",
    "restoration_runtime_seconds",
    "inference_seconds",
    "processing_seconds",
    "elapsed_seconds",
    "duration_seconds",
    "total_runtime_seconds",
    "batch_runtime_seconds",
]

BATCH4_DEVICE_COLUMNS = [
    "device",
    "device_effective",
    "device_requested",
    "compute_device",
    "gpu",
    "cuda_device",
]

BATCH4_BACKEND_COLUMNS = [
    "backend",
    "model_backend",
    "runtime_backend",
    "restoration_backend",
    "iopaint_model_name",
    "restoration_method",
]

def batch4_first_present_column(columns: list[str], candidates: list[str]) -> str:
    lookup = {clean_text(column).lower(): clean_text(column) for column in columns}
    for candidate in candidates:
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]
    return ""

def batch4_unique_join(values: pd.Series) -> str:
    cleaned = sorted(
        {
            clean_text(value)
            for value in values
            if clean_text(value) and clean_text(value).lower() not in {"nan", "none", "null"}
        }
    )
    return "|".join(cleaned) if cleaned else "unknown"

def batch4_json_list(values: pd.Series) -> str:
    cleaned = sorted(
        {
            clean_text(value)
            for value in values
            if clean_text(value) and clean_text(value).lower() not in {"nan", "none", "null"}
        }
    )
    return json.dumps(cleaned)

def batch4_model_slug(model_name: Any) -> str:
    text = clean_text(model_name).lower()
    return {
        "opencv": "opencv_telea",
        "opencv_telea": "opencv_telea",
        "telea": "opencv_telea",
        "lama": "lama",
        "stable diffusion": "stable_diffusion",
        "stable_diffusion": "stable_diffusion",
        "sd": "stable_diffusion",
        "sdxl": "sdxl",
    }.get(text, text.replace(" ", "_").replace("-", "_"))

def batch4_metric_family_lookup(metric_df: pd.DataFrame) -> dict[str, str]:
    if "metric_family" not in metric_df.columns or "metric_name" not in metric_df.columns:
        return {}
    lookup_df = metric_df[["metric_name", "metric_family"]].dropna().drop_duplicates()
    return dict(zip(lookup_df["metric_name"].astype(str), lookup_df["metric_family"].astype(str)))

print(f"Batch 4 summary output: {rel(BATCH4_SUMMARY_TABLES_PATH)}")

Batch 4 summary output: outputs/27_multi_model_comparison/analysis/multi_model_summary_tables.csv


In [21]:
# Batch 4 / Cell 3 - Build metadata dimensions
def batch4_normalize_damage_percent_from_column(series: pd.Series, column_name: str) -> pd.Series:
    numeric = pd.to_numeric(series, errors="coerce")
    lower_name = column_name.lower()

    finite = numeric[np.isfinite(numeric)]
    if finite.empty:
        return numeric

    # Ratios/fractions are converted to percentages. Percent-like columns that
    # only contain 0..1 values are also treated as fractional percentages.
    should_scale = (
        "ratio" in lower_name
        or "fraction" in lower_name
        or (finite.max() <= 1.0 and finite.min() >= 0.0)
    )

    return numeric * 100.0 if should_scale else numeric

def batch4_damage_bucket(value: Any) -> str:
    if pd.isna(value):
        return "unknown"
    value = float(value)
    if value <= 0:
        return "zero_control"
    if value <= 5:
        return "0_to_5"
    if value <= 15:
        return "5_to_15"
    if value <= 30:
        return "15_to_30"
    if value <= 50:
        return "30_to_50"
    return "over_50"

def batch4_coalesce_text_columns(df: pd.DataFrame, candidates: list[str], default: str = "unknown") -> pd.Series:
    result = pd.Series(default, index=df.index, dtype="object")
    for column in candidates:
        if column not in df.columns:
            continue
        values = df[column].astype(str).map(clean_text)
        usable = values.ne("") & ~values.str.lower().isin({"nan", "none", "null"})
        result = result.mask(result.eq(default) & usable, values)
    return result.fillna(default).replace("", default)

def batch4_coalesce_damage_percent(df: pd.DataFrame) -> pd.Series:
    result = pd.Series(np.nan, index=df.index, dtype="float64")
    for column in BATCH4_DAMAGE_PERCENT_COLUMNS:
        if column not in df.columns:
            continue
        normalized = batch4_normalize_damage_percent_from_column(df[column], column)
        result = result.mask(result.isna() & normalized.notna(), normalized)
    return result

def batch4_extract_case_metadata(source_id: str, source_df: pd.DataFrame) -> pd.DataFrame:
    if source_df is None or not len(source_df):
        return pd.DataFrame()

    working_df = source_df.copy()
    working_df.columns = [clean_text(column) for column in working_df.columns]
    columns = list(working_df.columns)

    case_column = batch4_first_present_column(columns, BATCH4_CASE_ID_COLUMNS)
    if not case_column:
        return pd.DataFrame()

    metadata_df = pd.DataFrame(
        {
            "case_id": working_df[case_column].astype(str).map(clean_text),
            "metadata_source_id": source_id,
        }
    )

    for target_column, candidates in BATCH4_DIMENSION_CANDIDATES.items():
        metadata_df[target_column] = batch4_coalesce_text_columns(working_df, candidates)

    metadata_df["damage_percent"] = batch4_coalesce_damage_percent(working_df)
    metadata_df = metadata_df.loc[metadata_df["case_id"].ne("")].copy()

    return metadata_df

case_metadata_frames = []
for source_id, source_df in batch1_source_frames.items():
    extracted_df = batch4_extract_case_metadata(source_id, source_df)
    if len(extracted_df):
        case_metadata_frames.append(extracted_df)

batch4_case_metadata_raw_df = (
    pd.concat(case_metadata_frames, ignore_index=True)
    if case_metadata_frames
    else pd.DataFrame()
)

if len(batch4_case_metadata_raw_df):
    batch4_case_metadata_df = (
        batch4_case_metadata_raw_df
        .groupby("case_id", dropna=False)
        .agg(
            style_category=("style_category", batch4_unique_join),
            content_category=("content_category", batch4_unique_join),
            mask_damage_type=("mask_damage_type", batch4_unique_join),
            degradation_type=("degradation_type", batch4_unique_join),
            damage_percent=("damage_percent", "median"),
            metadata_source_ids=("metadata_source_id", batch4_json_list),
        )
        .reset_index()
    )
else:
    batch4_case_metadata_df = pd.DataFrame(
        columns=[
            "case_id",
            "style_category",
            "content_category",
            "mask_damage_type",
            "degradation_type",
            "damage_percent",
            "metadata_source_ids",
        ]
    )

batch4_case_metadata_df["damage_percent_bucket"] = batch4_case_metadata_df["damage_percent"].map(batch4_damage_bucket)

print(f"Raw metadata rows: {len(batch4_case_metadata_raw_df):,}")
print(f"Case metadata rows: {len(batch4_case_metadata_df):,}")
display(batch4_case_metadata_df.head(20))

Raw metadata rows: 21,707
Case metadata rows: 875


,case_id,style_category,content_category,mask_damage_type,degradation_type,damage_percent,metadata_source_ids,damage_percent_bucket
0,canonical__p001_loss_large,Baroque|unknown,portrait_figure|unknown,loss_large|unknown,unknown,NaN,"[""lama_classical_metrics"", ""lama_feature_metri...",unknown
1,canonical__p001_loss_small,Baroque|unknown,portrait_figure|unknown,loss_small|unknown,unknown,NaN,"[""lama_classical_metrics"", ""lama_feature_metri...",unknown
2,canonical__p001_mixed_damage,Baroque|unknown,portrait_figure|unknown,mixed_damage|unknown,unknown,NaN,"[""lama_classical_metrics"", ""lama_feature_metri...",unknown
3,canonical__p001_scratch_thin,Baroque|unknown,portrait_figure|unknown,scratch_thin|unknown,unknown,NaN,"[""lama_classical_metrics"", ""lama_feature_metri...",unknown
4,canonical__p001_zero_control,Baroque|unknown,portrait_figure|unknown,unknown|zero_control,unknown,NaN,"[""lama_classical_metrics"", ""lama_feature_metri...",unknown
5,canonical__p002_loss_large,19th century portraiture|unknown,portrait_figure|unknown,loss_large|unknown,unknown,NaN,"[""lama_classical_metrics"", ""lama_feature_metri...",unknown
6,canonical__p002_loss_small,19th century portraiture|unknown,portrait_figure|unknown,loss_small|unknown,unknown,NaN,"[""lama_classical_metrics"", ""lama_feature_metri...",unknown
7,canonical__p002_mixed_damage,19th century portraiture|unknown,portrait_figure|unknown,mixed_damage|unknown,unknown,NaN,"[""lama_classical_metrics"", ""lama_feature_metri...",unknown
8,canonical__p002_scratch_thin,19th century portraiture|unknown,portrait_figure|unknown,scratch_thin|unknown,unknown,NaN,"[""lama_classical_metrics"", ""lama_feature_metri...",unknown
9,canonical__p002_zero_control,19th century portraiture|unknown,portrait_figure|unknown,unknown|zero_control,unknown,NaN,"[""lama_classical_metrics"", ""lama_feature_metri...",unknown


In [22]:
# Batch 4 / Cell 4 - Enrich selected metric rows
batch4_metric_df = batch3_selected_metric_long_df.copy()
batch4_metric_df.columns = [clean_text(column) for column in batch4_metric_df.columns]

if "model_slug" not in batch4_metric_df.columns:
    batch4_metric_df["model_slug"] = batch4_metric_df["model_name"].map(batch4_model_slug)

batch4_metric_df["case_id"] = batch4_metric_df["case_id"].astype(str).map(clean_text)
batch4_metric_df["candidate_id"] = (
    batch4_metric_df["candidate_id"].astype(str).map(clean_text)
    if "candidate_id" in batch4_metric_df.columns
    else "single_candidate"
)
batch4_metric_df["metric_name"] = batch4_metric_df["metric_name"].astype(str).map(clean_text)
batch4_metric_df["metric_family"] = batch4_metric_df["metric_family"].astype(str).map(clean_text)
batch4_metric_df["evaluation_region"] = batch4_metric_df["evaluation_region"].astype(str).map(clean_text)
batch4_metric_df["metric_value"] = pd.to_numeric(batch4_metric_df["metric_value"], errors="coerce")
batch4_metric_df["metric_value_is_finite"] = np.isfinite(batch4_metric_df["metric_value"].to_numpy(dtype=float))

for target_column, candidates in BATCH4_DIMENSION_CANDIDATES.items():
    if target_column not in batch4_metric_df.columns:
        batch4_metric_df[target_column] = batch4_coalesce_text_columns(batch4_metric_df, candidates)
    else:
        direct_values = batch4_metric_df[target_column].astype(str).map(clean_text)
        fallback_values = batch4_coalesce_text_columns(batch4_metric_df, candidates)
        batch4_metric_df[target_column] = direct_values.mask(
            direct_values.eq("") | direct_values.str.lower().isin({"nan", "none", "null"}),
            fallback_values,
        )

batch4_metric_df["damage_percent"] = batch4_coalesce_damage_percent(batch4_metric_df)

batch4_metric_df = batch4_metric_df.merge(
    batch4_case_metadata_df.add_suffix("__case_meta"),
    left_on="case_id",
    right_on="case_id__case_meta",
    how="left",
)

for target_column in ["style_category", "content_category", "mask_damage_type", "degradation_type"]:
    meta_column = f"{target_column}__case_meta"
    if meta_column in batch4_metric_df.columns:
        direct_values = batch4_metric_df[target_column].astype(str).map(clean_text)
        meta_values = batch4_metric_df[meta_column].astype(str).map(clean_text)
        usable_meta = meta_values.ne("") & ~meta_values.str.lower().isin({"nan", "none", "null"})
        batch4_metric_df[target_column] = direct_values.mask(
            direct_values.eq("unknown") & usable_meta,
            meta_values,
        )

if "damage_percent__case_meta" in batch4_metric_df.columns:
    batch4_metric_df["damage_percent"] = batch4_metric_df["damage_percent"].mask(
        batch4_metric_df["damage_percent"].isna(),
        pd.to_numeric(batch4_metric_df["damage_percent__case_meta"], errors="coerce"),
    )

batch4_metric_df["damage_percent_bucket"] = batch4_metric_df["damage_percent"].map(batch4_damage_bucket)

for column in [
    "style_category",
    "content_category",
    "mask_damage_type",
    "degradation_type",
    "damage_percent_bucket",
]:
    batch4_metric_df[column] = (
        batch4_metric_df[column]
        .astype(str)
        .map(clean_text)
        .replace({"": "unknown", "nan": "unknown", "None": "unknown"})
    )

batch4_finite_metric_df = batch4_metric_df.loc[batch4_metric_df["metric_value_is_finite"]].copy()

print(f"Selected metric rows: {len(batch4_metric_df):,}")
print(f"Finite selected metric rows: {len(batch4_finite_metric_df):,}")
display(
    batch4_finite_metric_df[
        [
            "case_id",
            "model_slug",
            "metric_family",
            "metric_name",
            "evaluation_region",
            "style_category",
            "content_category",
            "mask_damage_type",
            "damage_percent_bucket",
            "degradation_type",
            "metric_value",
        ]
    ].head(20)
)

Selected metric rows: 47,492
Finite selected metric rows: 44,047


,case_id,model_slug,metric_family,metric_name,evaluation_region,style_category,content_category,mask_damage_type,damage_percent_bucket,degradation_type,metric_value
0,canonical__p001_loss_large,lama,classical,mae_improvement,boundary_region,Baroque|unknown,portrait_figure|unknown,loss_large|unknown,unknown,unknown,110.674597
1,canonical__p001_loss_large,lama,classical,mse_improvement,boundary_region,Baroque|unknown,portrait_figure|unknown,loss_large|unknown,unknown,unknown,25396.024811
2,canonical__p001_loss_large,lama,classical,psnr_improvement,boundary_region,Baroque|unknown,portrait_figure|unknown,loss_large|unknown,unknown,unknown,35.140351
4,canonical__p001_loss_large,lama,classical,mae_improvement,content_region,Baroque|unknown,portrait_figure|unknown,loss_large|unknown,unknown,unknown,28.347510
5,canonical__p001_loss_large,lama,classical,mse_improvement,content_region,Baroque|unknown,portrait_figure|unknown,loss_large|unknown,unknown,unknown,6556.833107
6,canonical__p001_loss_large,lama,classical,psnr_improvement,content_region,Baroque|unknown,portrait_figure|unknown,loss_large|unknown,unknown,unknown,19.357619
7,canonical__p001_loss_large,lama,classical,ssim_improvement,content_region,Baroque|unknown,portrait_figure|unknown,loss_large|unknown,unknown,unknown,0.084635
8,canonical__p001_loss_large,lama,classical,mae_improvement,full_image,Baroque|unknown,portrait_figure|unknown,loss_large|unknown,unknown,unknown,24.471869
9,canonical__p001_loss_large,lama,classical,mse_improvement,full_image,Baroque|unknown,portrait_figure|unknown,loss_large|unknown,unknown,unknown,5660.389977
10,canonical__p001_loss_large,lama,classical,psnr_improvement,full_image,Baroque|unknown,portrait_figure|unknown,loss_large|unknown,unknown,unknown,19.357617


In [23]:
# Batch 4 / Cell 5 - Metric summary tables
def batch4_numeric_summary(
    df: pd.DataFrame,
    *,
    scope: str,
    summary_kind: str,
    value_column: str,
    value_name: str,
    group_columns: list[str],
    notes: str = "",
) -> pd.DataFrame:
    if not len(df):
        return pd.DataFrame()

    working_df = df.copy()

    for column in group_columns:
        if column not in working_df.columns:
            working_df[column] = "unknown"
        working_df[column] = working_df[column].astype(str).map(clean_text).replace("", "unknown")

    if "case_id" not in working_df.columns:
        working_df["case_id"] = ""
    if "candidate_id" not in working_df.columns:
        working_df["candidate_id"] = ""

    working_df[value_column] = pd.to_numeric(working_df[value_column], errors="coerce")
    working_df["_finite_value"] = np.isfinite(working_df[value_column].to_numpy(dtype=float))

    summary_df = (
        working_df
        .groupby(group_columns, dropna=False)
        .agg(
            rows=(value_column, "size"),
            finite_rows=("_finite_value", "sum"),
            missing_rows=("_finite_value", lambda values: int((~values).sum())),
            case_count=("case_id", "nunique"),
            candidate_count=("candidate_id", "nunique"),
            mean_value=(value_column, "mean"),
            median_value=(value_column, "median"),
            std_value=(value_column, "std"),
            min_value=(value_column, "min"),
            q25_value=(value_column, lambda values: pd.to_numeric(values, errors="coerce").quantile(0.25)),
            q75_value=(value_column, lambda values: pd.to_numeric(values, errors="coerce").quantile(0.75)),
            max_value=(value_column, "max"),
            positive_count=(value_column, lambda values: int((pd.to_numeric(values, errors="coerce") > 0).sum())),
            negative_count=(value_column, lambda values: int((pd.to_numeric(values, errors="coerce") < 0).sum())),
            zero_count=(value_column, lambda values: int((pd.to_numeric(values, errors="coerce") == 0).sum())),
        )
        .reset_index()
    )

    summary_df.insert(0, "summary_scope", scope)
    summary_df.insert(1, "summary_kind", summary_kind)
    summary_df.insert(2, "value_name", value_name)
    summary_df["notes"] = notes

    return summary_df

metric_base_group = ["metric_family", "metric_name", "evaluation_region", "model_slug"]

batch4_metric_summary_frames = []

batch4_metric_summary_frames.append(
    batch4_numeric_summary(
        batch4_metric_df,
        scope="overall_by_metric",
        summary_kind="model_metric_value",
        value_column="metric_value",
        value_name="metric_value",
        group_columns=metric_base_group,
        notes="Main model metric summary after Stable Diffusion candidate policy filtering.",
    )
)

batch4_metric_summary_frames.append(
    batch4_numeric_summary(
        batch4_metric_df,
        scope="metric_family_rollup",
        summary_kind="model_metric_value",
        value_column="metric_value",
        value_name="metric_value",
        group_columns=["metric_family", "evaluation_region", "model_slug"],
        notes="Rollup across metric names; use mainly for coverage/context because metric scales can differ.",
    )
)

for dimension_column in [
    "style_category",
    "content_category",
    "mask_damage_type",
    "damage_percent_bucket",
    "degradation_type",
]:
    batch4_metric_summary_frames.append(
        batch4_numeric_summary(
            batch4_metric_df,
            scope=f"by_{dimension_column}",
            summary_kind="model_metric_value",
            value_column="metric_value",
            value_name="metric_value",
            group_columns=[dimension_column] + metric_base_group,
            notes=f"Metric summary sliced by {dimension_column}. Unknown means the upstream source did not expose that metadata.",
        )
    )

batch4_metric_summary_df = pd.concat(
    [frame for frame in batch4_metric_summary_frames if len(frame)],
    ignore_index=True,
    sort=False,
)

print(f"Metric summary rows: {len(batch4_metric_summary_df):,}")
display(batch4_metric_summary_df.head(30))

Metric summary rows: 3,283


,summary_scope,summary_kind,value_name,metric_family,metric_name,evaluation_region,model_slug,rows,finite_rows,missing_rows,...,max_value,positive_count,negative_count,zero_count,notes,style_category,content_category,mask_damage_type,damage_percent_bucket,degradation_type
0,overall_by_metric,model_metric_value,metric_value,classical,mae_improvement,boundary_region,lama,360,360,0,...,116.675960,310,15,35,Main model metric summary after Stable Diffusi...,NaN,NaN,NaN,NaN,NaN
1,overall_by_metric,model_metric_value,metric_value,classical,mae_improvement,boundary_region,opencv_telea,355,355,0,...,46.250548,81,274,0,Main model metric summary after Stable Diffusi...,NaN,NaN,NaN,NaN,NaN
2,overall_by_metric,model_metric_value,metric_value,classical,mae_improvement,boundary_region,stable_diffusion,415,415,0,...,110.830508,309,21,85,Main model metric summary after Stable Diffusi...,NaN,NaN,NaN,NaN,NaN
3,overall_by_metric,model_metric_value,metric_value,classical,mae_improvement,content_region,lama,410,410,0,...,42.145375,310,45,55,Main model metric summary after Stable Diffusi...,NaN,NaN,NaN,NaN,NaN
4,overall_by_metric,model_metric_value,metric_value,classical,mae_improvement,content_region,opencv_telea,410,410,0,...,20.483670,73,282,55,Main model metric summary after Stable Diffusi...,NaN,NaN,NaN,NaN,NaN
5,overall_by_metric,model_metric_value,metric_value,classical,mae_improvement,content_region,stable_diffusion,465,465,0,...,40.800262,310,105,50,Main model metric summary after Stable Diffusi...,NaN,NaN,NaN,NaN,NaN
6,overall_by_metric,model_metric_value,metric_value,classical,mae_improvement,full_image,lama,410,410,0,...,36.383309,310,45,55,Main model metric summary after Stable Diffusi...,NaN,NaN,NaN,NaN,NaN
7,overall_by_metric,model_metric_value,metric_value,classical,mae_improvement,full_image,opencv_telea,410,410,0,...,16.296253,73,282,55,Main model metric summary after Stable Diffusi...,NaN,NaN,NaN,NaN,NaN
8,overall_by_metric,model_metric_value,metric_value,classical,mae_improvement,full_image,stable_diffusion,465,465,0,...,35.222098,310,105,50,Main model metric summary after Stable Diffusi...,NaN,NaN,NaN,NaN,NaN
9,overall_by_metric,model_metric_value,metric_value,classical,mae_improvement,mask_bbox_crop,lama,360,360,0,...,125.243476,310,45,5,Main model metric summary after Stable Diffusi...,NaN,NaN,NaN,NaN,NaN


In [24]:
# Batch 4 / Cell 6 - Paired delta summaries
batch4_case_comparison_working_df = batch3_case_comparison_df.copy()
batch4_case_comparison_working_df.columns = [clean_text(column) for column in batch4_case_comparison_working_df.columns]

metric_family_by_name = batch4_metric_family_lookup(batch4_metric_df)

if "metric_family" not in batch4_case_comparison_working_df.columns:
    batch4_case_comparison_working_df["metric_family"] = (
        batch4_case_comparison_working_df["metric_name"].astype(str).map(metric_family_by_name).fillna("unknown")
    )

batch4_case_comparison_working_df = batch4_case_comparison_working_df.merge(
    batch4_case_metadata_df,
    on="case_id",
    how="left",
)

for column in ["style_category", "content_category", "mask_damage_type", "degradation_type", "damage_percent_bucket"]:
    if column not in batch4_case_comparison_working_df.columns:
        batch4_case_comparison_working_df[column] = "unknown"
    batch4_case_comparison_working_df[column] = (
        batch4_case_comparison_working_df[column]
        .astype(str)
        .map(clean_text)
        .replace({"": "unknown", "nan": "unknown", "None": "unknown"})
    )

delta_columns = [
    column for column in batch4_case_comparison_working_df.columns
    if column.startswith("delta__")
]

batch4_delta_long_df = batch4_case_comparison_working_df.melt(
    id_vars=[
        "case_id",
        "metric_family",
        "metric_name",
        "evaluation_region",
        "style_category",
        "content_category",
        "mask_damage_type",
        "damage_percent_bucket",
        "degradation_type",
    ],
    value_vars=delta_columns,
    var_name="comparison_pair",
    value_name="delta_value",
)

batch4_delta_long_df["comparison_pair"] = (
    batch4_delta_long_df["comparison_pair"]
    .str.replace("delta__", "", regex=False)
    .str.replace("_minus_", " minus ", regex=False)
)
batch4_delta_long_df["delta_value"] = pd.to_numeric(batch4_delta_long_df["delta_value"], errors="coerce")
batch4_delta_long_df["candidate_id"] = "paired_case"

delta_base_group = ["metric_family", "metric_name", "evaluation_region", "comparison_pair"]

batch4_delta_summary_frames = []

batch4_delta_summary_frames.append(
    batch4_numeric_summary(
        batch4_delta_long_df,
        scope="paired_delta_by_metric",
        summary_kind="paired_delta_value",
        value_column="delta_value",
        value_name="delta_value",
        group_columns=delta_base_group,
        notes="Positive means the first model in comparison_pair has higher improvement delta.",
    )
)

for dimension_column in [
    "style_category",
    "content_category",
    "mask_damage_type",
    "damage_percent_bucket",
    "degradation_type",
]:
    batch4_delta_summary_frames.append(
        batch4_numeric_summary(
            batch4_delta_long_df,
            scope=f"paired_delta_by_{dimension_column}",
            summary_kind="paired_delta_value",
            value_column="delta_value",
            value_name="delta_value",
            group_columns=[dimension_column] + delta_base_group,
            notes=f"Paired model delta summary sliced by {dimension_column}.",
        )
    )

batch4_delta_summary_df = pd.concat(
    [frame for frame in batch4_delta_summary_frames if len(frame)],
    ignore_index=True,
    sort=False,
)

print(f"Delta summary rows: {len(batch4_delta_summary_df):,}")
display(batch4_delta_summary_df.head(30))

Delta summary rows: 6,639


,summary_scope,summary_kind,value_name,metric_family,metric_name,evaluation_region,comparison_pair,rows,finite_rows,missing_rows,...,max_value,positive_count,negative_count,zero_count,notes,style_category,content_category,mask_damage_type,damage_percent_bucket,degradation_type
0,paired_delta_by_metric,paired_delta_value,delta_value,classical,mae_improvement,boundary_region,lama minus opencv_telea,775,355,420,...,173.187223,329,26,0,Positive means the first model in comparison_p...,NaN,NaN,NaN,NaN,NaN
1,paired_delta_by_metric,paired_delta_value,delta_value,classical,mae_improvement,boundary_region,stable_diffusion minus lama,775,0,775,...,NaN,0,0,0,Positive means the first model in comparison_p...,NaN,NaN,NaN,NaN,NaN
2,paired_delta_by_metric,paired_delta_value,delta_value,classical,mae_improvement,boundary_region,stable_diffusion minus opencv_telea,775,0,775,...,NaN,0,0,0,Positive means the first model in comparison_p...,NaN,NaN,NaN,NaN,NaN
3,paired_delta_by_metric,paired_delta_value,delta_value,classical,mae_improvement,content_region,lama minus opencv_telea,875,410,465,...,69.361794,328,27,55,Positive means the first model in comparison_p...,NaN,NaN,NaN,NaN,NaN
4,paired_delta_by_metric,paired_delta_value,delta_value,classical,mae_improvement,content_region,stable_diffusion minus lama,875,0,875,...,NaN,0,0,0,Positive means the first model in comparison_p...,NaN,NaN,NaN,NaN,NaN
5,paired_delta_by_metric,paired_delta_value,delta_value,classical,mae_improvement,content_region,stable_diffusion minus opencv_telea,875,0,875,...,NaN,0,0,0,Positive means the first model in comparison_p...,NaN,NaN,NaN,NaN,NaN
6,paired_delta_by_metric,paired_delta_value,delta_value,classical,mae_improvement,full_image,lama minus opencv_telea,875,410,465,...,59.878733,328,27,55,Positive means the first model in comparison_p...,NaN,NaN,NaN,NaN,NaN
7,paired_delta_by_metric,paired_delta_value,delta_value,classical,mae_improvement,full_image,stable_diffusion minus lama,875,0,875,...,NaN,0,0,0,Positive means the first model in comparison_p...,NaN,NaN,NaN,NaN,NaN
8,paired_delta_by_metric,paired_delta_value,delta_value,classical,mae_improvement,full_image,stable_diffusion minus opencv_telea,875,0,875,...,NaN,0,0,0,Positive means the first model in comparison_p...,NaN,NaN,NaN,NaN,NaN
9,paired_delta_by_metric,paired_delta_value,delta_value,classical,mae_improvement,mask_bbox_crop,lama minus opencv_telea,775,355,420,...,204.463970,328,27,0,Positive means the first model in comparison_p...,NaN,NaN,NaN,NaN,NaN


In [25]:
# Batch 4 / Cell 7 - Runtime and compute summaries
def batch4_extract_runtime_rows(source_id: str, source_df: pd.DataFrame) -> pd.DataFrame:
    if source_df is None or not len(source_df):
        return pd.DataFrame()

    working_df = source_df.copy()
    working_df.columns = [clean_text(column) for column in working_df.columns]
    columns = list(working_df.columns)

    runtime_columns = [column for column in BATCH4_RUNTIME_COLUMNS if column in columns]
    if not runtime_columns:
        return pd.DataFrame()

    case_column = batch4_first_present_column(columns, BATCH4_CASE_ID_COLUMNS)
    candidate_column = batch4_first_present_column(columns, BATCH4_CANDIDATE_ID_COLUMNS)
    device_column = batch4_first_present_column(columns, BATCH4_DEVICE_COLUMNS)
    backend_column = batch4_first_present_column(columns, BATCH4_BACKEND_COLUMNS)
    model_column = batch4_first_present_column(columns, ["model_name", "restoration_method", "method", "model"])

    id_df = pd.DataFrame(
        {
            "source_id": source_id,
            "case_id": working_df[case_column].astype(str).map(clean_text) if case_column else "",
            "candidate_id": working_df[candidate_column].astype(str).map(clean_text) if candidate_column else "",
            "compute_device": working_df[device_column].astype(str).map(clean_text) if device_column else "unknown",
            "compute_backend": working_df[backend_column].astype(str).map(clean_text) if backend_column else "unknown",
            "model_slug": (
                working_df[model_column].astype(str).map(batch4_model_slug)
                if model_column
                else batch4_model_slug(source_id)
            ),
        }
    )

    runtime_frames = []
    for runtime_column in runtime_columns:
        runtime_df = id_df.copy()
        runtime_df["runtime_metric"] = runtime_column
        runtime_df["runtime_seconds"] = pd.to_numeric(working_df[runtime_column], errors="coerce")
        runtime_frames.append(runtime_df)

    return pd.concat(runtime_frames, ignore_index=True)

runtime_frames = []
for source_id, source_df in batch1_source_frames.items():
    extracted_runtime_df = batch4_extract_runtime_rows(source_id, source_df)
    if len(extracted_runtime_df):
        runtime_frames.append(extracted_runtime_df)

batch4_runtime_long_df = (
    pd.concat(runtime_frames, ignore_index=True)
    if runtime_frames
    else pd.DataFrame()
)

if len(batch4_runtime_long_df):
    batch4_runtime_long_df = batch4_runtime_long_df.merge(
        batch4_case_metadata_df,
        on="case_id",
        how="left",
    )

    for column in ["style_category", "content_category", "mask_damage_type", "degradation_type", "damage_percent_bucket"]:
        if column not in batch4_runtime_long_df.columns:
            batch4_runtime_long_df[column] = "unknown"
        batch4_runtime_long_df[column] = (
            batch4_runtime_long_df[column]
            .astype(str)
            .map(clean_text)
            .replace({"": "unknown", "nan": "unknown", "None": "unknown"})
        )

    batch4_runtime_summary_df = batch4_numeric_summary(
        batch4_runtime_long_df,
        scope="runtime_compute",
        summary_kind="runtime_seconds",
        value_column="runtime_seconds",
        value_name="runtime_seconds",
        group_columns=["model_slug", "runtime_metric", "compute_device", "compute_backend", "source_id"],
        notes="Runtime/compute summary from any source exposing runtime-like columns.",
    )

    batch4_runtime_by_damage_df = batch4_numeric_summary(
        batch4_runtime_long_df,
        scope="runtime_compute_by_damage_percent_bucket",
        summary_kind="runtime_seconds",
        value_column="runtime_seconds",
        value_name="runtime_seconds",
        group_columns=["damage_percent_bucket", "model_slug", "runtime_metric", "compute_device", "compute_backend"],
        notes="Runtime summary sliced by damage percentage bucket where case metadata is available.",
    )

    batch4_runtime_summary_df = pd.concat(
        [batch4_runtime_summary_df, batch4_runtime_by_damage_df],
        ignore_index=True,
        sort=False,
    )
else:
    batch4_runtime_summary_df = pd.DataFrame()

print(f"Runtime rows found: {len(batch4_runtime_long_df):,}")
print(f"Runtime summary rows: {len(batch4_runtime_summary_df):,}")
display(batch4_runtime_summary_df.head(30) if len(batch4_runtime_summary_df) else pd.DataFrame({"message": ["No runtime columns found in loaded sources."]}))

Runtime rows found: 2,065
Runtime summary rows: 8


,summary_scope,summary_kind,value_name,model_slug,runtime_metric,compute_device,compute_backend,source_id,rows,finite_rows,...,std_value,min_value,q25_value,q75_value,max_value,positive_count,negative_count,zero_count,notes,damage_percent_bucket
0,runtime_compute,runtime_seconds,runtime_seconds,lama,batch_runtime_seconds,cuda,lama,lama_report_cases,355,355,...,0.000000,525.937764,525.937764,525.937764,525.937764,355,0,0,Runtime/compute summary from any source exposi...,NaN
1,runtime_compute,runtime_seconds,runtime_seconds,lama,runtime_seconds,cuda,lama,lama_report_cases,355,355,...,0.000000,1.460938,1.460938,1.460938,1.460938,355,0,0,Runtime/compute summary from any source exposi...,NaN
2,runtime_compute,runtime_seconds,runtime_seconds,opencv_report_cases,runtime_seconds,unknown,unknown,opencv_report_cases,410,410,...,0.104911,0.246150,0.389133,0.509819,1.198466,410,0,0,Runtime/compute summary from any source exposi...,NaN
3,runtime_compute,runtime_seconds,runtime_seconds,stable_diffusion_inpainting,runtime_seconds,unknown,stable_diffusion_inpainting,stable_diffusion_report_audit,945,945,...,2.989015,0.002359,9.423096,10.337418,10.763753,945,0,0,Runtime/compute summary from any source exposi...,NaN
4,runtime_compute_by_damage_percent_bucket,runtime_seconds,runtime_seconds,lama,batch_runtime_seconds,cuda,lama,NaN,355,355,...,0.000000,525.937764,525.937764,525.937764,525.937764,355,0,0,Runtime summary sliced by damage percentage bu...,unknown
5,runtime_compute_by_damage_percent_bucket,runtime_seconds,runtime_seconds,lama,runtime_seconds,cuda,lama,NaN,355,355,...,0.000000,1.460938,1.460938,1.460938,1.460938,355,0,0,Runtime summary sliced by damage percentage bu...,unknown
6,runtime_compute_by_damage_percent_bucket,runtime_seconds,runtime_seconds,opencv_report_cases,runtime_seconds,unknown,unknown,NaN,410,410,...,0.104911,0.246150,0.389133,0.509819,1.198466,410,0,0,Runtime summary sliced by damage percentage bu...,unknown
7,runtime_compute_by_damage_percent_bucket,runtime_seconds,runtime_seconds,stable_diffusion_inpainting,runtime_seconds,unknown,stable_diffusion_inpainting,NaN,945,945,...,2.989015,0.002359,9.423096,10.337418,10.763753,945,0,0,Runtime summary sliced by damage percentage bu...,unknown


In [26]:
# Batch 4 / Cell 8 - Combine, validate, write
batch4_summary_tables_df = pd.concat(
    [
        batch4_metric_summary_df,
        batch4_delta_summary_df,
        batch4_runtime_summary_df,
    ],
    ignore_index=True,
    sort=False,
)

standard_summary_columns = [
    "summary_scope",
    "summary_kind",
    "value_name",
    "metric_family",
    "metric_name",
    "evaluation_region",
    "model_slug",
    "comparison_pair",
    "style_category",
    "content_category",
    "mask_damage_type",
    "damage_percent_bucket",
    "degradation_type",
    "runtime_metric",
    "compute_device",
    "compute_backend",
    "source_id",
    "rows",
    "finite_rows",
    "missing_rows",
    "case_count",
    "candidate_count",
    "mean_value",
    "median_value",
    "std_value",
    "min_value",
    "q25_value",
    "q75_value",
    "max_value",
    "positive_count",
    "negative_count",
    "zero_count",
    "notes",
]

for column in standard_summary_columns:
    if column not in batch4_summary_tables_df.columns:
        batch4_summary_tables_df[column] = ""

batch4_summary_tables_df = batch4_summary_tables_df[standard_summary_columns].copy()

for column in [
    "summary_scope",
    "summary_kind",
    "value_name",
    "metric_family",
    "metric_name",
    "evaluation_region",
    "model_slug",
    "comparison_pair",
    "style_category",
    "content_category",
    "mask_damage_type",
    "damage_percent_bucket",
    "degradation_type",
    "runtime_metric",
    "compute_device",
    "compute_backend",
    "source_id",
    "notes",
]:
    batch4_summary_tables_df[column] = (
        batch4_summary_tables_df[column]
        .astype(str)
        .map(clean_text)
        .replace({"": "all", "nan": "all", "None": "all"})
    )

for column in [
    "rows",
    "finite_rows",
    "missing_rows",
    "case_count",
    "candidate_count",
    "positive_count",
    "negative_count",
    "zero_count",
]:
    batch4_summary_tables_df[column] = pd.to_numeric(
        batch4_summary_tables_df[column],
        errors="coerce",
    ).fillna(0).astype(int)

for column in [
    "mean_value",
    "median_value",
    "std_value",
    "min_value",
    "q25_value",
    "q75_value",
    "max_value",
]:
    batch4_summary_tables_df[column] = pd.to_numeric(batch4_summary_tables_df[column], errors="coerce")

batch4_summary_tables_df["summary_row_id"] = (
    batch4_summary_tables_df["summary_scope"].astype(str)
    + "::" + batch4_summary_tables_df["summary_kind"].astype(str)
    + "::" + batch4_summary_tables_df["value_name"].astype(str)
    + "::" + batch4_summary_tables_df["metric_family"].astype(str)
    + "::" + batch4_summary_tables_df["metric_name"].astype(str)
    + "::" + batch4_summary_tables_df["evaluation_region"].astype(str)
    + "::" + batch4_summary_tables_df["model_slug"].astype(str)
    + "::" + batch4_summary_tables_df["comparison_pair"].astype(str)
    + "::" + batch4_summary_tables_df["style_category"].astype(str)
    + "::" + batch4_summary_tables_df["content_category"].astype(str)
    + "::" + batch4_summary_tables_df["mask_damage_type"].astype(str)
    + "::" + batch4_summary_tables_df["damage_percent_bucket"].astype(str)
    + "::" + batch4_summary_tables_df["degradation_type"].astype(str)
    + "::" + batch4_summary_tables_df["runtime_metric"].astype(str)
    + "::" + batch4_summary_tables_df["compute_device"].astype(str)
    + "::" + batch4_summary_tables_df["compute_backend"].astype(str)
    + "::" + batch4_summary_tables_df["source_id"].astype(str)
)

batch4_summary_tables_df = batch4_summary_tables_df[
    ["summary_row_id"] + standard_summary_columns
].sort_values(
    ["summary_scope", "summary_kind", "metric_family", "metric_name", "evaluation_region", "model_slug"],
    kind="mergesort",
).reset_index(drop=True)

summary_row_duplicate_count = int(batch4_summary_tables_df["summary_row_id"].duplicated().sum())
summary_rows_with_values = int(batch4_summary_tables_df["finite_rows"].gt(0).sum())
metric_summary_rows = int(batch4_summary_tables_df["summary_kind"].eq("model_metric_value").sum())
delta_summary_rows = int(batch4_summary_tables_df["summary_kind"].eq("paired_delta_value").sum())
runtime_summary_rows = int(batch4_summary_tables_df["summary_kind"].eq("runtime_seconds").sum())

required_scopes = {
    "overall_by_metric",
    "metric_family_rollup",
    "by_style_category",
    "by_content_category",
    "by_mask_damage_type",
    "by_damage_percent_bucket",
    "by_degradation_type",
    "paired_delta_by_metric",
}

observed_scopes = set(batch4_summary_tables_df["summary_scope"].unique().tolist())
missing_required_scopes = sorted(required_scopes - observed_scopes)

batch4_validation_rows = [
    validation_row(
        "summary_table_has_rows",
        len(batch4_summary_tables_df),
        "> 0",
        len(batch4_summary_tables_df) > 0,
        "Batch 4 summary table is empty.",
    ),
    validation_row(
        "summary_row_ids_unique",
        summary_row_duplicate_count,
        "0",
        summary_row_duplicate_count == 0,
        "Batch 4 summary row ids are not unique.",
    ),
    validation_row(
        "summary_rows_have_finite_values",
        summary_rows_with_values,
        "> 0",
        summary_rows_with_values > 0,
        "No summary rows contain finite values.",
    ),
    validation_row(
        "metric_summary_rows_present",
        metric_summary_rows,
        "> 0",
        metric_summary_rows > 0,
        "No model metric summary rows were created.",
    ),
    validation_row(
        "paired_delta_summary_rows_present",
        delta_summary_rows,
        "> 0",
        delta_summary_rows > 0,
        "No paired delta summary rows were created.",
    ),
    validation_row(
        "required_summary_scopes_present",
        sorted(observed_scopes),
        sorted(required_scopes),
        len(missing_required_scopes) == 0,
        f"Missing required summary scopes: {missing_required_scopes}",
    ),
]

batch4_validation_df = pd.DataFrame(batch4_validation_rows)
batch4_passed = bool(batch4_validation_df["passed"].all())

batch4_summary_tables_df.to_csv(BATCH4_SUMMARY_TABLES_PATH, index=False)
batch4_validation_df.to_csv(BATCH4_VALIDATION_PATH, index=False)

stage_manifest = read_json_if_exists(STAGE_MANIFEST_PATH)
stage_manifest.update(
    {
        "notebook_id": globals().get("NOTEBOOK_ID", "27_multi_model_comparison"),
        "notebook_title": globals().get("NOTEBOOK_TITLE", "Multi-model comparison"),
        "stage": "batch4_summary_tables",
        "stage_status": "passed" if batch4_passed else "failed",
        "updated_at_utc": utc_now_iso(),
        "outputs": {
            **stage_manifest.get("outputs", {}),
            "multi_model_summary_tables_csv": rel(BATCH4_SUMMARY_TABLES_PATH),
            "batch4_summary_tables_validation_csv": rel(BATCH4_VALIDATION_PATH),
        },
        "batch4": {
            "summary_rows": int(len(batch4_summary_tables_df)),
            "metric_summary_rows": metric_summary_rows,
            "paired_delta_summary_rows": delta_summary_rows,
            "runtime_summary_rows": runtime_summary_rows,
            "metadata_case_rows": int(len(batch4_case_metadata_df)),
            "runtime_rows_found": int(len(batch4_runtime_long_df)) if len(batch4_runtime_long_df) else 0,
            "observed_summary_scopes": sorted(observed_scopes),
            "note": (
                "Summary table is normalized for final report slicing. "
                "Runtime summaries are included only when runtime-like columns are found."
            ),
        },
    }
)

write_json(STAGE_MANIFEST_PATH, stage_manifest)

print(f"Batch 4 checks passed: {int(batch4_validation_df['passed'].sum())} / {len(batch4_validation_df)}")
display(batch4_validation_df)

print(f"Saved Batch 4 summary table: {rel(BATCH4_SUMMARY_TABLES_PATH)}")
print(f"Saved Batch 4 validation: {rel(BATCH4_VALIDATION_PATH)}")

display(
    batch4_summary_tables_df.groupby(["summary_scope", "summary_kind"], dropna=False)
    .agg(
        rows=("summary_row_id", "size"),
        finite_groups=("finite_rows", lambda values: int((pd.to_numeric(values, errors="coerce") > 0).sum())),
        total_cases=("case_count", "sum"),
    )
    .reset_index()
)

if not batch4_passed:
    display(batch4_validation_df.loc[~batch4_validation_df["passed"], ["check_name", "actual", "expected", "failure_message"]])
    raise RuntimeError("Batch 4 validation failed. Fix summary generation before Batch 5.")

print("Batch 4 passed. Summary tables are ready for report generation.")

Batch 4 checks passed: 6 / 6


,check_name,actual,expected,passed,failure_message
0,summary_table_has_rows,9930,> 0,True,
1,summary_row_ids_unique,0,0,True,
2,summary_rows_have_finite_values,4074,> 0,True,
3,metric_summary_rows_present,3283,> 0,True,
4,paired_delta_summary_rows_present,6639,> 0,True,
5,required_summary_scopes_present,"[by_content_category, by_damage_percent_bucket...","[by_content_category, by_damage_percent_bucket...",True,


Saved Batch 4 summary table: outputs/27_multi_model_comparison/analysis/multi_model_summary_tables.csv
Saved Batch 4 validation: outputs/27_multi_model_comparison/validation/batch4_summary_tables_validation.csv


,summary_scope,summary_kind,rows,finite_groups,total_cases
0,by_content_category,model_metric_value,540,495,43340
1,by_damage_percent_bucket,model_metric_value,108,99,43340
2,by_degradation_type,model_metric_value,567,531,43340
3,by_mask_damage_type,model_metric_value,880,810,43340
4,by_style_category,model_metric_value,1044,957,43340
5,metric_family_rollup,model_metric_value,36,36,14525
6,overall_by_metric,model_metric_value,108,99,43340
7,paired_delta_by_content_category,paired_delta_value,990,165,81855
8,paired_delta_by_damage_percent_bucket,paired_delta_value,99,33,81855
9,paired_delta_by_degradation_type,paired_delta_value,2178,165,81855


Batch 4 passed. Summary tables are ready for report generation.


In [27]:
# Batch 5 / Cell 2 - Setup helpers
from __future__ import annotations

import json
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd

required_batch5_globals = [
    "OUTPUT_ROOT",
    "VALIDATION_DIR",
    "MANIFESTS_DIR",
    "batch3_case_comparison_df",
    "rel",
    "clean_text",
    "utc_now_iso",
    "read_json_if_exists",
    "write_json",
    "validation_row",
]

missing_batch5_globals = [name for name in required_batch5_globals if name not in globals()]
if missing_batch5_globals:
    raise RuntimeError(
        "Batch 5 requires Batches 1-4 to be run first. Missing globals: "
        + ", ".join(missing_batch5_globals)
    )

ANALYSIS_DIR = OUTPUT_ROOT / "analysis"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

if "BATCH3_CASE_COMPARISON_PATH" not in globals():
    BATCH3_CASE_COMPARISON_PATH = ANALYSIS_DIR / "multi_model_case_comparison.csv"

if "BATCH4_SUMMARY_TABLES_PATH" not in globals():
    BATCH4_SUMMARY_TABLES_PATH = ANALYSIS_DIR / "multi_model_summary_tables.csv"

BATCH5_VALIDATION_PATH = VALIDATION_DIR / "batch5_disagreement_ranking_failure_validation.csv"
STAGE_MANIFEST_PATH = MANIFESTS_DIR / "multi_model_comparison_stage_manifest.json"

BATCH5_HIGHER_IS_BETTER_HINTS = [
    "improvement",
    "delta",
    "ssim",
    "psnr",
    "similarity",
    "clip",
    "dino",
    "score",
    "accuracy",
    "agreement",
]

BATCH5_LOWER_IS_BETTER_HINTS = [
    "lpips",
    "mse",
    "mae",
    "rmse",
    "l1",
    "l2",
    "error",
    "distance",
    "fid",
    "loss",
]

BATCH5_DETERMINISTIC_MODEL_SLUGS = ["opencv_telea", "lama"]
BATCH5_GENERATIVE_MODEL_SLUGS = ["stable_diffusion", "sdxl"]

def batch5_metric_direction(metric_name: object) -> str:
    text = clean_text(metric_name).lower()

    if any(hint in text for hint in ["improvement", "delta"]):
        return "higher_is_better"

    if any(hint in text for hint in BATCH5_LOWER_IS_BETTER_HINTS):
        return "lower_is_better"

    if any(hint in text for hint in BATCH5_HIGHER_IS_BETTER_HINTS):
        return "higher_is_better"

    return "unknown"

def batch5_to_bool(value: object) -> bool:
    if isinstance(value, bool):
        return value
    text = clean_text(value).lower()
    return text in {"true", "1", "yes", "y", "passed", "success", "ok"}

def batch5_safe_json(value: object) -> str:
    return json.dumps(value, sort_keys=True)

def batch5_finite_series(series: pd.Series) -> pd.Series:
    numeric = pd.to_numeric(series, errors="coerce")
    return numeric.notna() & np.isfinite(numeric)

batch5_case_comparison_df = batch3_case_comparison_df.copy()
batch5_case_comparison_df.columns = [clean_text(column) for column in batch5_case_comparison_df.columns]

if "case_id" not in batch5_case_comparison_df.columns:
    raise RuntimeError("Batch 5 requires case_id in the case comparison table.")

if "evaluation_region" not in batch5_case_comparison_df.columns:
    batch5_case_comparison_df["evaluation_region"] = "unknown"

if "metric_name" not in batch5_case_comparison_df.columns:
    raise RuntimeError("Batch 5 requires metric_name in the case comparison table.")

if "metric_family" not in batch5_case_comparison_df.columns:
    batch5_case_comparison_df["metric_family"] = "unknown"

batch5_metric_value_columns = [
    column for column in batch5_case_comparison_df.columns
    if column.startswith("metric_value__")
]

batch5_model_slugs = sorted(
    column.replace("metric_value__", "", 1)
    for column in batch5_metric_value_columns
)

if len(batch5_model_slugs) < 2:
    raise RuntimeError(
        "Batch 5 needs at least two model metric_value__ columns for disagreement/ranking analysis."
    )

print(f"Observed model metric columns: {batch5_model_slugs}")
print(f"Batch 5 will update: {rel(BATCH3_CASE_COMPARISON_PATH)}")
print(f"Batch 5 will append summary rows to: {rel(BATCH4_SUMMARY_TABLES_PATH)}")

Observed model metric columns: ['lama', 'opencv_telea', 'sdxl', 'stable_diffusion']
Batch 5 will update: outputs/27_multi_model_comparison/metrics/multi_model_case_comparison.csv
Batch 5 will append summary rows to: outputs/27_multi_model_comparison/analysis/multi_model_summary_tables.csv


In [28]:
# Batch 5 / Cell 3 - Row-level metric disagreement and rankings
def batch5_rank_case_metric_row(row: pd.Series) -> dict:
    metric_direction = batch5_metric_direction(row.get("metric_name", ""))
    values = {}

    for model_slug in batch5_model_slugs:
        value_column = f"metric_value__{model_slug}"
        value = pd.to_numeric(row.get(value_column, np.nan), errors="coerce")
        if pd.notna(value) and np.isfinite(value):
            values[model_slug] = float(value)

    finite_model_count = len(values)

    output = {
        "batch5_metric_direction": metric_direction,
        "batch5_finite_model_count": finite_model_count,
        "batch5_rankable_metric": bool(metric_direction in {"higher_is_better", "lower_is_better"} and finite_model_count >= 2),
        "batch5_best_model_slug": "unranked",
        "batch5_worst_model_slug": "unranked",
        "batch5_best_tie_model_count": 0,
        "batch5_model_rank_order_json": "[]",
        "batch5_top2_margin_raw": np.nan,
        "batch5_top2_margin_relative": np.nan,
        "batch5_metric_value_range_raw": np.nan,
        "batch5_metric_value_relative_range": np.nan,
    }

    for model_slug in batch5_model_slugs:
        output[f"batch5_rank__{model_slug}"] = np.nan

    if finite_model_count >= 2:
        raw_values = np.array(list(values.values()), dtype=float)
        raw_range = float(np.nanmax(raw_values) - np.nanmin(raw_values))
        raw_median_abs = float(max(abs(np.nanmedian(raw_values)), 1e-12))
        output["batch5_metric_value_range_raw"] = raw_range
        output["batch5_metric_value_relative_range"] = raw_range / raw_median_abs

    if not output["batch5_rankable_metric"]:
        return output

    scored_items = []
    for model_slug, metric_value in values.items():
        score = metric_value if metric_direction == "higher_is_better" else -metric_value
        scored_items.append((model_slug, score, metric_value))

    scored_items = sorted(scored_items, key=lambda item: item[1], reverse=True)
    max_abs_score = max(abs(item[1]) for item in scored_items)
    tie_tolerance = max(1e-12, 1e-12 * max_abs_score)

    current_rank = 0
    previous_score = None
    rank_order = []

    for position, (model_slug, score, metric_value) in enumerate(scored_items, start=1):
        if previous_score is None or abs(score - previous_score) > tie_tolerance:
            current_rank = position
            previous_score = score

        output[f"batch5_rank__{model_slug}"] = current_rank
        rank_order.append(
            {
                "model_slug": model_slug,
                "rank": int(current_rank),
                "metric_value": float(metric_value),
            }
        )

    best_score = scored_items[0][1]
    worst_score = scored_items[-1][1]
    best_models = [
        model_slug for model_slug, score, _ in scored_items
        if abs(score - best_score) <= tie_tolerance
    ]

    unique_scores = []
    for _, score, _ in scored_items:
        if not unique_scores or abs(score - unique_scores[-1]) > tie_tolerance:
            unique_scores.append(score)

    if len(unique_scores) >= 2:
        top2_margin = float(unique_scores[0] - unique_scores[1])
        top2_denominator = float(max(abs(unique_scores[0]), abs(unique_scores[1]), 1e-12))
        output["batch5_top2_margin_raw"] = top2_margin
        output["batch5_top2_margin_relative"] = top2_margin / top2_denominator
    else:
        output["batch5_top2_margin_raw"] = 0.0
        output["batch5_top2_margin_relative"] = 0.0

    output["batch5_best_model_slug"] = "|".join(best_models)
    output["batch5_worst_model_slug"] = scored_items[-1][0]
    output["batch5_best_tie_model_count"] = int(len(best_models))
    output["batch5_model_rank_order_json"] = batch5_safe_json(rank_order)

    return output

batch5_rank_df = batch5_case_comparison_df.apply(
    batch5_rank_case_metric_row,
    axis=1,
    result_type="expand",
)

for column in batch5_rank_df.columns:
    batch5_case_comparison_df[column] = batch5_rank_df[column]

disagreement_group_columns = ["metric_family", "metric_name", "evaluation_region"]
batch5_case_comparison_df["batch5_metric_disagreement_percentile"] = np.nan

for _, group_index in batch5_case_comparison_df.groupby(disagreement_group_columns, dropna=False).groups.items():
    group_index = list(group_index)
    values = pd.to_numeric(
        batch5_case_comparison_df.loc[group_index, "batch5_metric_value_relative_range"],
        errors="coerce",
    )
    if values.notna().any():
        batch5_case_comparison_df.loc[group_index, "batch5_metric_disagreement_percentile"] = (
            values.rank(method="average", pct=True)
        )

def batch5_disagreement_label(row: pd.Series) -> str:
    if int(row.get("batch5_finite_model_count", 0)) < 2:
        return "insufficient_models"

    percentile = pd.to_numeric(row.get("batch5_metric_disagreement_percentile", np.nan), errors="coerce")
    relative_range = pd.to_numeric(row.get("batch5_metric_value_relative_range", np.nan), errors="coerce")

    if pd.isna(percentile) or pd.isna(relative_range):
        return "unscored"

    if relative_range == 0:
        return "none"

    if percentile >= 0.75:
        return "high"

    if percentile >= 0.50:
        return "moderate"

    return "low"

batch5_case_comparison_df["batch5_metric_disagreement_flag"] = batch5_case_comparison_df.apply(
    batch5_disagreement_label,
    axis=1,
)

print("Batch 5 row-level ranking preview:")
display(
    batch5_case_comparison_df[
        [
            "case_id",
            "evaluation_region",
            "metric_family",
            "metric_name",
            "batch5_metric_direction",
            "batch5_finite_model_count",
            "batch5_best_model_slug",
            "batch5_top2_margin_relative",
            "batch5_metric_value_relative_range",
            "batch5_metric_disagreement_flag",
        ]
    ].head(25)
)

Batch 5 row-level ranking preview:


,case_id,evaluation_region,metric_family,metric_name,batch5_metric_direction,batch5_finite_model_count,batch5_best_model_slug,batch5_top2_margin_relative,batch5_metric_value_relative_range,batch5_metric_disagreement_flag
0,canonical__p001_loss_large,boundary_region,unknown,mae_improvement,higher_is_better,2,lama,1.496364,5.942240,high
1,canonical__p001_loss_large,boundary_region,unknown,mse_improvement,higher_is_better,2,lama,0.997898,1.991610,low
2,canonical__p001_loss_large,boundary_region,unknown,psnr_improvement,higher_is_better,2,lama,0.719867,1.124675,low
3,canonical__p001_loss_large,content_region,unknown,clip_similarity_improvement,higher_is_better,2,lama,0.907338,1.660785,high
4,canonical__p001_loss_large,content_region,unknown,dinov2_similarity_improvement,higher_is_better,2,lama,1.527111,6.458641,high
5,canonical__p001_loss_large,content_region,unknown,lpips_improvement,higher_is_better,2,lama,0.081041,0.084463,moderate
6,canonical__p001_loss_large,content_region,unknown,mae_improvement,higher_is_better,2,lama,1.621684,8.573179,high
7,canonical__p001_loss_large,content_region,unknown,mean_similarity_improvement,higher_is_better,2,lama,1.099767,2.443293,high
8,canonical__p001_loss_large,content_region,unknown,mse_improvement,higher_is_better,2,lama,0.998594,1.994382,moderate
9,canonical__p001_loss_large,content_region,unknown,psnr_improvement,higher_is_better,2,lama,0.814414,1.373859,low


In [29]:
# Batch 5 / Cell 4 - Deterministic vs generative failure flags
batch5_deterministic_observed_slugs = [
    model_slug for model_slug in BATCH5_DETERMINISTIC_MODEL_SLUGS
    if model_slug in batch5_model_slugs
]

batch5_generative_observed_slugs = [
    model_slug for model_slug in BATCH5_GENERATIVE_MODEL_SLUGS
    if model_slug in batch5_model_slugs
]

def batch5_group_has_count(model_slugs: list[str]) -> pd.Series:
    series_list = []

    for model_slug in model_slugs:
        has_column = f"has__{model_slug}"
        value_column = f"metric_value__{model_slug}"

        if has_column in batch5_case_comparison_df.columns:
            series_list.append(batch5_case_comparison_df[has_column].map(batch5_to_bool).astype(int))
        elif value_column in batch5_case_comparison_df.columns:
            series_list.append(batch5_finite_series(batch5_case_comparison_df[value_column]).astype(int))

    if not series_list:
        return pd.Series(0, index=batch5_case_comparison_df.index, dtype=int)

    return pd.concat(series_list, axis=1).sum(axis=1).astype(int)

def batch5_group_finite_count(model_slugs: list[str]) -> pd.Series:
    series_list = []

    for model_slug in model_slugs:
        value_column = f"metric_value__{model_slug}"
        if value_column in batch5_case_comparison_df.columns:
            series_list.append(batch5_finite_series(batch5_case_comparison_df[value_column]).astype(int))

    if not series_list:
        return pd.Series(0, index=batch5_case_comparison_df.index, dtype=int)

    return pd.concat(series_list, axis=1).sum(axis=1).astype(int)

batch5_case_comparison_df["batch5_deterministic_group_model_count"] = len(batch5_deterministic_observed_slugs)
batch5_case_comparison_df["batch5_generative_group_model_count"] = len(batch5_generative_observed_slugs)

batch5_case_comparison_df["batch5_deterministic_available_model_count"] = batch5_group_has_count(
    batch5_deterministic_observed_slugs
)
batch5_case_comparison_df["batch5_generative_available_model_count"] = batch5_group_has_count(
    batch5_generative_observed_slugs
)

batch5_case_comparison_df["batch5_deterministic_finite_model_count"] = batch5_group_finite_count(
    batch5_deterministic_observed_slugs
)
batch5_case_comparison_df["batch5_generative_finite_model_count"] = batch5_group_finite_count(
    batch5_generative_observed_slugs
)

batch5_case_comparison_df["batch5_cross_family_comparison_available"] = (
    batch5_case_comparison_df["batch5_deterministic_finite_model_count"].gt(0)
    & batch5_case_comparison_df["batch5_generative_finite_model_count"].gt(0)
)

batch5_case_comparison_df["batch5_only_deterministic_finite"] = (
    batch5_case_comparison_df["batch5_deterministic_finite_model_count"].gt(0)
    & batch5_case_comparison_df["batch5_generative_finite_model_count"].eq(0)
)

batch5_case_comparison_df["batch5_only_generative_finite"] = (
    batch5_case_comparison_df["batch5_generative_finite_model_count"].gt(0)
    & batch5_case_comparison_df["batch5_deterministic_finite_model_count"].eq(0)
)

batch5_case_comparison_df["batch5_deterministic_metric_failure"] = (
    len(batch5_deterministic_observed_slugs) > 0
) & batch5_case_comparison_df["batch5_deterministic_finite_model_count"].eq(0)

batch5_case_comparison_df["batch5_generative_metric_failure"] = (
    len(batch5_generative_observed_slugs) > 0
) & batch5_case_comparison_df["batch5_generative_finite_model_count"].eq(0)

def batch5_best_model_family(best_model_slug: object) -> str:
    best_text = clean_text(best_model_slug)
    best_models = [part for part in best_text.split("|") if part]

    if not best_models:
        return "unranked"

    families = set()
    for model_slug in best_models:
        if model_slug in batch5_deterministic_observed_slugs:
            families.add("deterministic_or_non_sampling")
        elif model_slug in batch5_generative_observed_slugs:
            families.add("generative_diffusion")
        else:
            families.add("other")

    return "|".join(sorted(families))

batch5_case_comparison_df["batch5_best_model_family"] = batch5_case_comparison_df[
    "batch5_best_model_slug"
].map(batch5_best_model_family)

batch5_case_comparison_df["batch5_generative_beats_all_deterministic"] = (
    batch5_case_comparison_df["batch5_cross_family_comparison_available"]
    & batch5_case_comparison_df["batch5_best_model_family"].eq("generative_diffusion")
    & batch5_case_comparison_df["batch5_best_tie_model_count"].eq(1)
)

batch5_case_comparison_df["batch5_deterministic_beats_all_generative"] = (
    batch5_case_comparison_df["batch5_cross_family_comparison_available"]
    & batch5_case_comparison_df["batch5_best_model_family"].eq("deterministic_or_non_sampling")
    & batch5_case_comparison_df["batch5_best_tie_model_count"].eq(1)
)

if "has__stable_diffusion" in batch5_case_comparison_df.columns:
    batch5_sd_present = batch5_case_comparison_df["has__stable_diffusion"].map(batch5_to_bool)
elif "metric_value__stable_diffusion" in batch5_case_comparison_df.columns:
    batch5_sd_present = batch5_finite_series(batch5_case_comparison_df["metric_value__stable_diffusion"])
else:
    batch5_sd_present = pd.Series(False, index=batch5_case_comparison_df.index)

if "stable_diffusion_selected_candidate_id" in batch5_case_comparison_df.columns:
    batch5_sd_selected_missing = (
        batch5_case_comparison_df["stable_diffusion_selected_candidate_id"]
        .astype(str)
        .map(clean_text)
        .str.lower()
        .isin({"", "nan", "none", "null"})
    )
else:
    batch5_sd_selected_missing = pd.Series(False, index=batch5_case_comparison_df.index)

batch5_case_comparison_df["batch5_stable_diffusion_policy_missing"] = (
    batch5_sd_present & batch5_sd_selected_missing
)

def batch5_failure_flags(row: pd.Series) -> str:
    flags = []

    if bool(row.get("batch5_deterministic_metric_failure", False)):
        flags.append("deterministic_or_non_sampling_metric_failure")

    if bool(row.get("batch5_generative_metric_failure", False)):
        flags.append("generative_diffusion_metric_failure")

    if bool(row.get("batch5_only_deterministic_finite", False)):
        flags.append("only_deterministic_or_non_sampling_finite")

    if bool(row.get("batch5_only_generative_finite", False)):
        flags.append("only_generative_diffusion_finite")

    if not bool(row.get("batch5_cross_family_comparison_available", False)):
        flags.append("cross_family_comparison_unavailable")

    if bool(row.get("batch5_stable_diffusion_policy_missing", False)):
        flags.append("stable_diffusion_candidate_policy_missing")

    if clean_text(row.get("batch5_metric_disagreement_flag", "")) == "high":
        flags.append("high_metric_disagreement")

    if int(row.get("batch5_best_tie_model_count", 0)) > 1:
        flags.append("top_rank_tie")

    if not bool(row.get("batch5_rankable_metric", False)):
        flags.append("metric_not_rankable_direction_unknown_or_insufficient_models")

    return batch5_safe_json(flags)

batch5_case_comparison_df["batch5_failure_flags_json"] = batch5_case_comparison_df.apply(
    batch5_failure_flags,
    axis=1,
)

batch5_case_comparison_df["batch5_failure_flag_count"] = batch5_case_comparison_df[
    "batch5_failure_flags_json"
].map(lambda value: len(json.loads(value)))

print(f"Deterministic/non-sampling group: {batch5_deterministic_observed_slugs}")
print(f"Generative diffusion group: {batch5_generative_observed_slugs}")

display(
    batch5_case_comparison_df[
        [
            "case_id",
            "evaluation_region",
            "metric_name",
            "batch5_best_model_slug",
            "batch5_best_model_family",
            "batch5_cross_family_comparison_available",
            "batch5_generative_beats_all_deterministic",
            "batch5_deterministic_beats_all_generative",
            "batch5_failure_flags_json",
        ]
    ].head(25)
)

Deterministic/non-sampling group: ['opencv_telea', 'lama']
Generative diffusion group: ['stable_diffusion', 'sdxl']


,case_id,evaluation_region,metric_name,batch5_best_model_slug,batch5_best_model_family,batch5_cross_family_comparison_available,batch5_generative_beats_all_deterministic,batch5_deterministic_beats_all_generative,batch5_failure_flags_json
0,canonical__p001_loss_large,boundary_region,mae_improvement,lama,deterministic_or_non_sampling,False,False,False,"[""generative_diffusion_metric_failure"", ""only_..."
1,canonical__p001_loss_large,boundary_region,mse_improvement,lama,deterministic_or_non_sampling,False,False,False,"[""generative_diffusion_metric_failure"", ""only_..."
2,canonical__p001_loss_large,boundary_region,psnr_improvement,lama,deterministic_or_non_sampling,False,False,False,"[""generative_diffusion_metric_failure"", ""only_..."
3,canonical__p001_loss_large,content_region,clip_similarity_improvement,lama,deterministic_or_non_sampling,False,False,False,"[""generative_diffusion_metric_failure"", ""only_..."
4,canonical__p001_loss_large,content_region,dinov2_similarity_improvement,lama,deterministic_or_non_sampling,False,False,False,"[""generative_diffusion_metric_failure"", ""only_..."
5,canonical__p001_loss_large,content_region,lpips_improvement,lama,deterministic_or_non_sampling,False,False,False,"[""generative_diffusion_metric_failure"", ""only_..."
6,canonical__p001_loss_large,content_region,mae_improvement,lama,deterministic_or_non_sampling,False,False,False,"[""generative_diffusion_metric_failure"", ""only_..."
7,canonical__p001_loss_large,content_region,mean_similarity_improvement,lama,deterministic_or_non_sampling,False,False,False,"[""generative_diffusion_metric_failure"", ""only_..."
8,canonical__p001_loss_large,content_region,mse_improvement,lama,deterministic_or_non_sampling,False,False,False,"[""generative_diffusion_metric_failure"", ""only_..."
9,canonical__p001_loss_large,content_region,psnr_improvement,lama,deterministic_or_non_sampling,False,False,False,"[""generative_diffusion_metric_failure"", ""only_..."


In [31]:
# Batch 5 / Cell 4B - Reclassify coverage limitations vs true analysis warnings

def batch5_flag_list_to_json(flags: list[str]) -> str:
    return batch5_safe_json(sorted(set(flag for flag in flags if clean_text(flag))))

def batch5_reclassified_flags(row: pd.Series) -> pd.Series:
    coverage_flags = []
    warning_flags = []
    failure_flags = []

    deterministic_available = int(row.get("batch5_deterministic_available_model_count", 0))
    generative_available = int(row.get("batch5_generative_available_model_count", 0))
    deterministic_finite = int(row.get("batch5_deterministic_finite_model_count", 0))
    generative_finite = int(row.get("batch5_generative_finite_model_count", 0))

    if not bool(row.get("batch5_cross_family_comparison_available", False)):
        coverage_flags.append("cross_family_comparison_unavailable")

    if bool(row.get("batch5_only_deterministic_finite", False)):
        coverage_flags.append("only_deterministic_or_non_sampling_finite")

    if bool(row.get("batch5_only_generative_finite", False)):
        coverage_flags.append("only_generative_diffusion_finite")

    if not bool(row.get("batch5_rankable_metric", False)):
        coverage_flags.append("metric_not_rankable_direction_unknown_or_insufficient_models")

    if deterministic_available > 0 and deterministic_finite == 0:
        failure_flags.append("deterministic_or_non_sampling_metric_nonfinite")

    if generative_available > 0 and generative_finite == 0:
        failure_flags.append("generative_diffusion_metric_nonfinite")

    if bool(row.get("batch5_stable_diffusion_policy_missing", False)):
        failure_flags.append("stable_diffusion_candidate_policy_missing")

    if clean_text(row.get("batch5_metric_disagreement_flag", "")) == "high":
        warning_flags.append("high_metric_disagreement")

    if int(row.get("batch5_best_tie_model_count", 0)) > 1:
        warning_flags.append("top_rank_tie")

    diagnostic_flags = coverage_flags + warning_flags + failure_flags

    return pd.Series({
        "batch5_coverage_flags_json": batch5_flag_list_to_json(coverage_flags),
        "batch5_warning_flags_json": batch5_flag_list_to_json(warning_flags),
        "batch5_true_failure_flags_json": batch5_flag_list_to_json(failure_flags),
        "batch5_failure_flags_json": batch5_flag_list_to_json(warning_flags + failure_flags),
        "batch5_diagnostic_flags_json": batch5_flag_list_to_json(diagnostic_flags),
        "batch5_coverage_flag_count": len(coverage_flags),
        "batch5_warning_flag_count": len(warning_flags),
        "batch5_true_failure_flag_count": len(failure_flags),
        "batch5_failure_flag_count": len(warning_flags + failure_flags),
        "batch5_diagnostic_flag_count": len(diagnostic_flags),
    })

batch5_reclassified_flag_df = batch5_case_comparison_df.apply(batch5_reclassified_flags, axis=1)

for column in batch5_reclassified_flag_df.columns:
    batch5_case_comparison_df[column] = batch5_reclassified_flag_df[column]

print("Reclassified Batch 5 flags:")
print(f"Rows: {len(batch5_case_comparison_df):,}")
print(f"Coverage-limited rows: {int(batch5_case_comparison_df['batch5_coverage_flag_count'].gt(0).sum()):,}")
print(f"Warning rows: {int(batch5_case_comparison_df['batch5_warning_flag_count'].gt(0).sum()):,}")
print(f"True failure rows: {int(batch5_case_comparison_df['batch5_true_failure_flag_count'].gt(0).sum()):,}")
print(f"Failure/warning rows used by legacy failure column: {int(batch5_case_comparison_df['batch5_failure_flag_count'].gt(0).sum()):,}")

display_columns = [
    "case_id",
    "evaluation_region",
    "metric_name",
    "batch5_cross_family_comparison_available",
    "batch5_metric_disagreement_flag",
    "batch5_coverage_flags_json",
    "batch5_warning_flags_json",
    "batch5_true_failure_flags_json",
    "batch5_failure_flags_json",
]

display_columns = [
    column for column in display_columns
    if column in batch5_case_comparison_df.columns
]

display(batch5_case_comparison_df[display_columns].head(30))

Reclassified Batch 5 flags:
Rows: 27,285
Coverage-limited rows: 27,285
Warning rows: 4,854
True failure rows: 0
Failure/warning rows used by legacy failure column: 4,854


,case_id,evaluation_region,metric_name,batch5_cross_family_comparison_available,batch5_metric_disagreement_flag,batch5_coverage_flags_json,batch5_warning_flags_json,batch5_true_failure_flags_json,batch5_failure_flags_json
0,canonical__p001_loss_large,boundary_region,mae_improvement,False,high,"[""cross_family_comparison_unavailable"", ""only_...","[""high_metric_disagreement""]",[],"[""high_metric_disagreement""]"
1,canonical__p001_loss_large,boundary_region,mse_improvement,False,low,"[""cross_family_comparison_unavailable"", ""only_...",[],[],[]
2,canonical__p001_loss_large,boundary_region,psnr_improvement,False,low,"[""cross_family_comparison_unavailable"", ""only_...",[],[],[]
3,canonical__p001_loss_large,content_region,clip_similarity_improvement,False,high,"[""cross_family_comparison_unavailable"", ""only_...","[""high_metric_disagreement""]",[],"[""high_metric_disagreement""]"
4,canonical__p001_loss_large,content_region,dinov2_similarity_improvement,False,high,"[""cross_family_comparison_unavailable"", ""only_...","[""high_metric_disagreement""]",[],"[""high_metric_disagreement""]"
5,canonical__p001_loss_large,content_region,lpips_improvement,False,moderate,"[""cross_family_comparison_unavailable"", ""only_...",[],[],[]
6,canonical__p001_loss_large,content_region,mae_improvement,False,high,"[""cross_family_comparison_unavailable"", ""only_...","[""high_metric_disagreement""]",[],"[""high_metric_disagreement""]"
7,canonical__p001_loss_large,content_region,mean_similarity_improvement,False,high,"[""cross_family_comparison_unavailable"", ""only_...","[""high_metric_disagreement""]",[],"[""high_metric_disagreement""]"
8,canonical__p001_loss_large,content_region,mse_improvement,False,moderate,"[""cross_family_comparison_unavailable"", ""only_...",[],[],[]
9,canonical__p001_loss_large,content_region,psnr_improvement,False,low,"[""cross_family_comparison_unavailable"", ""only_...",[],[],[]


In [32]:
# Batch 5 / Cell 5 - Case-level ranking stability
def batch5_case_stability_record(group_df: pd.DataFrame) -> dict:
    rankable_df = group_df.loc[group_df["batch5_rankable_metric"].astype(bool)].copy()

    rankable_metric_rows = int(len(rankable_df))
    high_disagreement_rows = int(group_df["batch5_metric_disagreement_flag"].eq("high").sum())
    failure_flag_rows = int(group_df["batch5_failure_flag_count"].gt(0).sum())

    record = {
        "batch5_case_rankable_metric_rows": rankable_metric_rows,
        "batch5_case_high_disagreement_rows": high_disagreement_rows,
        "batch5_case_failure_flag_rows": failure_flag_rows,
        "batch5_unique_best_model_count": 0,
        "batch5_top_best_model_slug": "none",
        "batch5_top_best_model_vote_share": np.nan,
        "batch5_best_model_vote_counts_json": "{}",
        "batch5_min_pairwise_majority_share": np.nan,
        "batch5_mean_top2_margin_relative": np.nan,
        "batch5_median_top2_margin_relative": np.nan,
        "batch5_mean_metric_value_relative_range": np.nan,
        "batch5_rank_std_by_model_json": "{}",
        "batch5_ranking_stability_label": "no_rankable_metrics",
    }

    if rankable_metric_rows == 0:
        return record

    best_counts = rankable_df["batch5_best_model_slug"].value_counts(dropna=False)
    best_counts_dict = {
        clean_text(model_slug): int(count)
        for model_slug, count in best_counts.items()
    }

    top_best_model = clean_text(best_counts.index[0])
    top_best_count = int(best_counts.iloc[0])
    top_vote_share = top_best_count / rankable_metric_rows

    pairwise_majority_shares = []
    for left_model, right_model in combinations(batch5_model_slugs, 2):
        left_rank_column = f"batch5_rank__{left_model}"
        right_rank_column = f"batch5_rank__{right_model}"

        if left_rank_column not in rankable_df.columns or right_rank_column not in rankable_df.columns:
            continue

        pair_df = rankable_df[[left_rank_column, right_rank_column]].apply(
            pd.to_numeric,
            errors="coerce",
        ).dropna()

        if not len(pair_df):
            continue

        left_wins = int((pair_df[left_rank_column] < pair_df[right_rank_column]).sum())
        right_wins = int((pair_df[right_rank_column] < pair_df[left_rank_column]).sum())
        ties = int((pair_df[right_rank_column] == pair_df[left_rank_column]).sum())

        pairwise_majority_shares.append(max(left_wins, right_wins, ties) / len(pair_df))

    rank_std_by_model = {}
    for model_slug in batch5_model_slugs:
        rank_column = f"batch5_rank__{model_slug}"
        if rank_column not in rankable_df.columns:
            continue

        rank_values = pd.to_numeric(rankable_df[rank_column], errors="coerce").dropna()
        if len(rank_values) > 1:
            rank_std_by_model[model_slug] = float(rank_values.std())
        elif len(rank_values) == 1:
            rank_std_by_model[model_slug] = 0.0

    min_pairwise_majority = (
        float(min(pairwise_majority_shares))
        if pairwise_majority_shares
        else np.nan
    )

    high_disagreement_share = high_disagreement_rows / max(rankable_metric_rows, 1)

    if (
        top_vote_share >= 0.75
        and (pd.isna(min_pairwise_majority) or min_pairwise_majority >= 0.75)
        and high_disagreement_share <= 0.25
    ):
        stability_label = "stable"
    elif (
        top_vote_share < 0.50
        or (pd.notna(min_pairwise_majority) and min_pairwise_majority < 0.60)
        or high_disagreement_share >= 0.50
    ):
        stability_label = "unstable"
    else:
        stability_label = "mixed"

    record.update(
        {
            "batch5_unique_best_model_count": int(len(best_counts_dict)),
            "batch5_top_best_model_slug": top_best_model,
            "batch5_top_best_model_vote_share": float(top_vote_share),
            "batch5_best_model_vote_counts_json": batch5_safe_json(best_counts_dict),
            "batch5_min_pairwise_majority_share": min_pairwise_majority,
            "batch5_mean_top2_margin_relative": float(
                pd.to_numeric(rankable_df["batch5_top2_margin_relative"], errors="coerce").mean()
            ),
            "batch5_median_top2_margin_relative": float(
                pd.to_numeric(rankable_df["batch5_top2_margin_relative"], errors="coerce").median()
            ),
            "batch5_mean_metric_value_relative_range": float(
                pd.to_numeric(rankable_df["batch5_metric_value_relative_range"], errors="coerce").mean()
            ),
            "batch5_rank_std_by_model_json": batch5_safe_json(rank_std_by_model),
            "batch5_ranking_stability_label": stability_label,
        }
    )

    return record

batch5_stability_group_columns = ["case_id", "evaluation_region"]
batch5_case_stability_records = []

for group_keys, group_df in batch5_case_comparison_df.groupby(
    batch5_stability_group_columns,
    dropna=False,
    sort=False,
):
    if not isinstance(group_keys, tuple):
        group_keys = (group_keys,)

    record = {
        column: value
        for column, value in zip(batch5_stability_group_columns, group_keys)
    }
    record.update(batch5_case_stability_record(group_df))
    batch5_case_stability_records.append(record)

batch5_case_stability_df = pd.DataFrame(batch5_case_stability_records)

batch5_case_comparison_df = batch5_case_comparison_df.merge(
    batch5_case_stability_df,
    on=batch5_stability_group_columns,
    how="left",
)

print(f"Case-level ranking stability rows: {len(batch5_case_stability_df):,}")
display(batch5_case_stability_df.head(25))

Case-level ranking stability rows: 4,900


,case_id,evaluation_region,batch5_case_rankable_metric_rows,batch5_case_high_disagreement_rows,batch5_case_failure_flag_rows,batch5_unique_best_model_count,batch5_top_best_model_slug,batch5_top_best_model_vote_share,batch5_best_model_vote_counts_json,batch5_min_pairwise_majority_share,batch5_mean_top2_margin_relative,batch5_median_top2_margin_relative,batch5_mean_metric_value_relative_range,batch5_rank_std_by_model_json,batch5_ranking_stability_label
0,canonical__p001_loss_large,boundary_region,3,1,1,1,lama,1.000,"{""lama"": 3}",1.000,1.071376,0.997898,3.019508,"{""lama"": 0.0, ""opencv_telea"": 0.0}",mixed
1,canonical__p001_loss_large,content_region,8,4,4,2,lama,0.875,"{""lama"": 7, ""opencv_telea"": 1}",0.875,0.885777,0.952966,2.828193,"{""lama"": 0.3535533905932738, ""opencv_telea"": 0...",unstable
2,canonical__p001_loss_large,full_image,8,4,4,2,lama,0.875,"{""lama"": 7, ""opencv_telea"": 1}",0.875,0.923134,0.985862,3.480552,"{""lama"": 0.3535533905932738, ""opencv_telea"": 0...",unstable
3,canonical__p001_loss_large,mask_bbox_crop,8,3,3,2,lama,0.875,"{""lama"": 7, ""opencv_telea"": 1}",0.875,0.873271,0.969565,2.557231,"{""lama"": 0.3535533905932738, ""opencv_telea"": 0...",mixed
4,canonical__p001_loss_large,masked_region,3,1,1,1,lama,1.000,"{""lama"": 3}",1.000,1.144897,0.998594,3.980473,"{""lama"": 0.0, ""opencv_telea"": 0.0}",mixed
5,canonical__p001_loss_large,outside_mask_region,3,0,3,1,lama|opencv_telea,1.000,"{""lama|opencv_telea"": 3}",1.000,0.000000,0.000000,0.000000,"{""lama"": 0.0, ""opencv_telea"": 0.0}",stable
6,canonical__p001_loss_small,boundary_region,3,1,1,1,lama,1.000,"{""lama"": 3}",1.000,1.072687,0.998266,2.949452,"{""lama"": 0.0, ""opencv_telea"": 0.0}",mixed
7,canonical__p001_loss_small,content_region,8,1,1,2,lama,0.875,"{""lama"": 7, ""opencv_telea"": 1}",0.875,0.430031,0.036707,1.329139,"{""lama"": 0.3535533905932738, ""opencv_telea"": 0...",stable
8,canonical__p001_loss_small,full_image,8,1,1,2,lama,0.625,"{""lama"": 5, ""opencv_telea"": 3}",0.625,0.421410,0.012822,1.320338,"{""lama"": 0.5175491695067657, ""opencv_telea"": 0...",mixed
9,canonical__p001_loss_small,mask_bbox_crop,8,1,1,2,lama,0.875,"{""lama"": 7, ""opencv_telea"": 1}",0.875,0.425012,0.019781,1.323987,"{""lama"": 0.3535533905932738, ""opencv_telea"": 0...",stable


In [33]:
# Batch 5 / Cell 6 - Build Batch 5 summary rows
batch5_standard_summary_columns = [
    "summary_scope",
    "summary_kind",
    "value_name",
    "metric_family",
    "metric_name",
    "evaluation_region",
    "model_slug",
    "comparison_pair",
    "style_category",
    "content_category",
    "mask_damage_type",
    "damage_percent_bucket",
    "degradation_type",
    "runtime_metric",
    "compute_device",
    "compute_backend",
    "source_id",
    "rows",
    "finite_rows",
    "missing_rows",
    "case_count",
    "candidate_count",
    "mean_value",
    "median_value",
    "std_value",
    "min_value",
    "q25_value",
    "q75_value",
    "max_value",
    "positive_count",
    "negative_count",
    "zero_count",
    "notes",
]

def batch5_numeric_summary(
    df: pd.DataFrame,
    *,
    scope: str,
    kind: str,
    value_column: str,
    value_name: str,
    group_columns: list[str],
    notes: str,
) -> pd.DataFrame:
    if not len(df):
        return pd.DataFrame(columns=batch5_standard_summary_columns)

    working_df = df.copy()

    for column in group_columns:
        if column not in working_df.columns:
            working_df[column] = "all"
        working_df[column] = working_df[column].astype(str).map(clean_text).replace("", "all")

    if "case_id" not in working_df.columns:
        working_df["case_id"] = ""

    working_df[value_column] = pd.to_numeric(working_df[value_column], errors="coerce")
    working_df["_finite_value"] = working_df[value_column].notna() & np.isfinite(working_df[value_column])

    summary_df = (
        working_df.groupby(group_columns, dropna=False)
        .agg(
            rows=(value_column, "size"),
            finite_rows=("_finite_value", "sum"),
            missing_rows=("_finite_value", lambda values: int((~values).sum())),
            case_count=("case_id", "nunique"),
            mean_value=(value_column, "mean"),
            median_value=(value_column, "median"),
            std_value=(value_column, "std"),
            min_value=(value_column, "min"),
            q25_value=(value_column, lambda values: pd.to_numeric(values, errors="coerce").quantile(0.25)),
            q75_value=(value_column, lambda values: pd.to_numeric(values, errors="coerce").quantile(0.75)),
            max_value=(value_column, "max"),
            positive_count=(value_column, lambda values: int((pd.to_numeric(values, errors="coerce") > 0).sum())),
            negative_count=(value_column, lambda values: int((pd.to_numeric(values, errors="coerce") < 0).sum())),
            zero_count=(value_column, lambda values: int((pd.to_numeric(values, errors="coerce") == 0).sum())),
        )
        .reset_index()
    )

    summary_df.insert(0, "summary_scope", scope)
    summary_df.insert(1, "summary_kind", kind)
    summary_df.insert(2, "value_name", value_name)
    summary_df["candidate_count"] = 0
    summary_df["notes"] = notes

    for column in batch5_standard_summary_columns:
        if column not in summary_df.columns:
            summary_df[column] = "all"

    return summary_df[batch5_standard_summary_columns]

batch5_summary_frames = []

batch5_summary_frames.append(
    batch5_numeric_summary(
        batch5_case_comparison_df,
        scope="batch5_metric_disagreement_by_metric",
        kind="metric_value_relative_range",
        value_column="batch5_metric_value_relative_range",
        value_name="relative_range_across_models",
        group_columns=["metric_family", "metric_name", "evaluation_region"],
        notes="Relative spread of finite model metric values for the same case/region/metric.",
    )
)

batch5_summary_frames.append(
    batch5_numeric_summary(
        batch5_case_comparison_df,
        scope="batch5_top2_ranking_margin_by_metric",
        kind="direction_aware_top2_margin",
        value_column="batch5_top2_margin_relative",
        value_name="relative_top2_margin",
        group_columns=["metric_family", "metric_name", "evaluation_region"],
        notes="Direction-aware margin between first and second ranked model. Larger means more stable separation.",
    )
)

batch5_stability_summary_input_df = batch5_case_stability_df.copy()
batch5_stability_summary_input_df["comparison_pair"] = batch5_stability_summary_input_df[
    "batch5_ranking_stability_label"
]

batch5_summary_frames.append(
    batch5_numeric_summary(
        batch5_stability_summary_input_df,
        scope="batch5_case_ranking_stability",
        kind="top_model_vote_share",
        value_column="batch5_top_best_model_vote_share",
        value_name="top_model_vote_share",
        group_columns=["evaluation_region", "comparison_pair"],
        notes="Case-level stability across rankable metrics. comparison_pair stores the stability label.",
    )
)

batch5_rankable_vote_df = batch5_case_comparison_df.loc[
    batch5_case_comparison_df["batch5_rankable_metric"].astype(bool)
    & ~batch5_case_comparison_df["batch5_best_model_slug"].astype(str).str.contains(r"\|", regex=True)
].copy()

if len(batch5_rankable_vote_df):
    vote_group_columns = ["metric_family", "metric_name", "evaluation_region", "batch5_best_model_slug"]
    vote_count_df = (
        batch5_rankable_vote_df.groupby(vote_group_columns, dropna=False)
        .agg(
            rows=("case_id", "size"),
            case_count=("case_id", "nunique"),
        )
        .reset_index()
    )

    vote_total_df = (
        batch5_rankable_vote_df.groupby(["metric_family", "metric_name", "evaluation_region"], dropna=False)
        .size()
        .reset_index(name="total_rankable_rows")
    )

    vote_count_df = vote_count_df.merge(
        vote_total_df,
        on=["metric_family", "metric_name", "evaluation_region"],
        how="left",
    )

    batch5_best_vote_summary_df = pd.DataFrame(
        {
            "summary_scope": "batch5_best_model_votes_by_metric",
            "summary_kind": "best_model_vote_share",
            "value_name": "best_model_vote_share",
            "metric_family": vote_count_df["metric_family"],
            "metric_name": vote_count_df["metric_name"],
            "evaluation_region": vote_count_df["evaluation_region"],
            "model_slug": vote_count_df["batch5_best_model_slug"],
            "rows": vote_count_df["rows"],
            "finite_rows": vote_count_df["rows"],
            "missing_rows": 0,
            "case_count": vote_count_df["case_count"],
            "candidate_count": 0,
            "mean_value": vote_count_df["rows"] / vote_count_df["total_rankable_rows"],
            "median_value": vote_count_df["rows"] / vote_count_df["total_rankable_rows"],
            "std_value": np.nan,
            "min_value": vote_count_df["rows"] / vote_count_df["total_rankable_rows"],
            "q25_value": vote_count_df["rows"] / vote_count_df["total_rankable_rows"],
            "q75_value": vote_count_df["rows"] / vote_count_df["total_rankable_rows"],
            "max_value": vote_count_df["rows"] / vote_count_df["total_rankable_rows"],
            "positive_count": vote_count_df["rows"],
            "negative_count": 0,
            "zero_count": 0,
            "notes": "Share of rankable rows where this model is the direction-aware best model.",
        }
    )

    for column in batch5_standard_summary_columns:
        if column not in batch5_best_vote_summary_df.columns:
            batch5_best_vote_summary_df[column] = "all"

    batch5_summary_frames.append(batch5_best_vote_summary_df[batch5_standard_summary_columns])

failure_records = []
for _, row in batch5_case_comparison_df.iterrows():
    flags = json.loads(row["batch5_failure_flags_json"])
    for flag in flags:
        failure_records.append(
            {
                "case_id": row["case_id"],
                "metric_family": row["metric_family"],
                "metric_name": row["metric_name"],
                "evaluation_region": row["evaluation_region"],
                "comparison_pair": flag,
                "failure_event_value": 1,
            }
        )

batch5_failure_event_df = pd.DataFrame(failure_records)

if len(batch5_failure_event_df):
    batch5_failure_summary_df = batch5_numeric_summary(
        batch5_failure_event_df,
        scope="batch5_failure_flags_by_metric",
        kind="deterministic_vs_generative_failure_flag",
        value_column="failure_event_value",
        value_name="failure_flag_count",
        group_columns=["metric_family", "metric_name", "evaluation_region", "comparison_pair"],
        notes="Failure and coverage flags. comparison_pair stores the flag name.",
    )

    denominator_df = (
        batch5_case_comparison_df.groupby(["metric_family", "metric_name", "evaluation_region"], dropna=False)
        .size()
        .reset_index(name="denominator_rows")
    )

    batch5_failure_summary_df = batch5_failure_summary_df.merge(
        denominator_df,
        on=["metric_family", "metric_name", "evaluation_region"],
        how="left",
    )

    batch5_failure_summary_df["mean_value"] = (
        batch5_failure_summary_df["rows"] / batch5_failure_summary_df["denominator_rows"]
    )
    batch5_failure_summary_df["notes"] = (
        "Failure and coverage flags. mean_value is the share of rows with this flag; "
        "comparison_pair stores the flag name."
    )
    batch5_failure_summary_df = batch5_failure_summary_df[batch5_standard_summary_columns]
    batch5_summary_frames.append(batch5_failure_summary_df)

batch5_summary_rows_df = pd.concat(
    [frame for frame in batch5_summary_frames if len(frame)],
    ignore_index=True,
    sort=False,
)

for column in batch5_standard_summary_columns:
    if column not in batch5_summary_rows_df.columns:
        batch5_summary_rows_df[column] = "all"

batch5_summary_rows_df = batch5_summary_rows_df[batch5_standard_summary_columns].copy()

print(f"Batch 5 summary rows created: {len(batch5_summary_rows_df):,}")
display(
    batch5_summary_rows_df.groupby(["summary_scope", "summary_kind"], dropna=False)
    .size()
    .reset_index(name="rows")
)

Batch 5 summary rows created: 197


,summary_scope,summary_kind,rows
0,batch5_best_model_votes_by_metric,best_model_vote_share,60
1,batch5_case_ranking_stability,top_model_vote_share,22
2,batch5_failure_flags_by_metric,deterministic_vs_generative_failure_flag,49
3,batch5_metric_disagreement_by_metric,metric_value_relative_range,33
4,batch5_top2_ranking_margin_by_metric,direction_aware_top2_margin,33


In [34]:
# Batch 5 / Cell 7 - Merge summary rows, validate, and write outputs
if "batch4_summary_tables_df" in globals():
    batch5_existing_summary_df = batch4_summary_tables_df.copy()
elif BATCH4_SUMMARY_TABLES_PATH.exists():
    batch5_existing_summary_df = pd.read_csv(BATCH4_SUMMARY_TABLES_PATH)
else:
    batch5_existing_summary_df = pd.DataFrame(columns=["summary_row_id"] + batch5_standard_summary_columns)

for column in ["summary_row_id"] + batch5_standard_summary_columns:
    if column not in batch5_existing_summary_df.columns:
        batch5_existing_summary_df[column] = "all"

batch5_existing_summary_df = batch5_existing_summary_df.loc[
    ~batch5_existing_summary_df["summary_scope"].astype(str).str.startswith("batch5_")
].copy()

batch5_full_summary_tables_df = pd.concat(
    [
        batch5_existing_summary_df[["summary_row_id"] + batch5_standard_summary_columns],
        batch5_summary_rows_df.assign(summary_row_id=""),
    ],
    ignore_index=True,
    sort=False,
)

for column in batch5_standard_summary_columns:
    if column not in batch5_full_summary_tables_df.columns:
        batch5_full_summary_tables_df[column] = "all"

text_summary_columns = [
    "summary_scope",
    "summary_kind",
    "value_name",
    "metric_family",
    "metric_name",
    "evaluation_region",
    "model_slug",
    "comparison_pair",
    "style_category",
    "content_category",
    "mask_damage_type",
    "damage_percent_bucket",
    "degradation_type",
    "runtime_metric",
    "compute_device",
    "compute_backend",
    "source_id",
    "notes",
]

for column in text_summary_columns:
    batch5_full_summary_tables_df[column] = (
        batch5_full_summary_tables_df[column]
        .astype(str)
        .map(clean_text)
        .replace({"": "all", "nan": "all", "None": "all"})
    )

integer_summary_columns = [
    "rows",
    "finite_rows",
    "missing_rows",
    "case_count",
    "candidate_count",
    "positive_count",
    "negative_count",
    "zero_count",
]

for column in integer_summary_columns:
    batch5_full_summary_tables_df[column] = pd.to_numeric(
        batch5_full_summary_tables_df[column],
        errors="coerce",
    ).fillna(0).astype(int)

numeric_summary_columns = [
    "mean_value",
    "median_value",
    "std_value",
    "min_value",
    "q25_value",
    "q75_value",
    "max_value",
]

for column in numeric_summary_columns:
    batch5_full_summary_tables_df[column] = pd.to_numeric(
        batch5_full_summary_tables_df[column],
        errors="coerce",
    )

summary_id_columns = [
    "summary_scope",
    "summary_kind",
    "value_name",
    "metric_family",
    "metric_name",
    "evaluation_region",
    "model_slug",
    "comparison_pair",
    "style_category",
    "content_category",
    "mask_damage_type",
    "damage_percent_bucket",
    "degradation_type",
    "runtime_metric",
    "compute_device",
    "compute_backend",
    "source_id",
]

batch5_full_summary_tables_df["summary_row_id"] = (
    batch5_full_summary_tables_df[summary_id_columns]
    .astype(str)
    .agg("::".join, axis=1)
)

batch5_full_summary_tables_df = batch5_full_summary_tables_df[
    ["summary_row_id"] + batch5_standard_summary_columns
].sort_values(
    ["summary_scope", "summary_kind", "metric_family", "metric_name", "evaluation_region", "model_slug"],
    kind="mergesort",
).reset_index(drop=True)

case_key_columns = (
    case_index_columns
    if "case_index_columns" in globals()
    else ["case_id", "evaluation_region", "metric_name"]
)
case_key_columns = [column for column in case_key_columns if column in batch5_case_comparison_df.columns]

batch5_required_case_columns = [
    "batch5_metric_direction",
    "batch5_finite_model_count",
    "batch5_rankable_metric",
    "batch5_best_model_slug",
    "batch5_metric_disagreement_flag",
    "batch5_failure_flags_json",
    "batch5_ranking_stability_label",
]

missing_required_case_columns = [
    column for column in batch5_required_case_columns
    if column not in batch5_case_comparison_df.columns
]

case_duplicate_count = (
    int(batch5_case_comparison_df.duplicated(case_key_columns).sum())
    if case_key_columns
    else 0
)

summary_duplicate_count = int(batch5_full_summary_tables_df["summary_row_id"].duplicated().sum())
rankable_metric_rows = int(batch5_case_comparison_df["batch5_rankable_metric"].astype(bool).sum())
batch5_summary_row_count = int(
    batch5_full_summary_tables_df["summary_scope"].astype(str).str.startswith("batch5_").sum()
)
failure_flag_rows = int(batch5_case_comparison_df["batch5_failure_flag_count"].gt(0).sum())

batch5_validation_rows = [
    validation_row(
        "batch5_case_table_has_rows",
        len(batch5_case_comparison_df),
        "> 0",
        len(batch5_case_comparison_df) > 0,
        "Batch 5 case comparison table is empty.",
    ),
    validation_row(
        "batch5_required_case_columns_present",
        missing_required_case_columns,
        "[]",
        len(missing_required_case_columns) == 0,
        f"Missing Batch 5 columns: {missing_required_case_columns}",
    ),
    validation_row(
        "batch5_case_keys_unique",
        case_duplicate_count,
        "0",
        case_duplicate_count == 0,
        "Batch 5 introduced duplicate case/evaluation/metric keys.",
    ),
    validation_row(
        "batch5_rankable_metric_rows_present",
        rankable_metric_rows,
        "> 0",
        rankable_metric_rows > 0,
        "No rankable metric rows were produced. Check metric direction inference.",
    ),
    validation_row(
        "batch5_summary_rows_present",
        batch5_summary_row_count,
        "> 0",
        batch5_summary_row_count > 0,
        "No Batch 5 summary rows were appended.",
    ),
    validation_row(
        "batch5_summary_row_ids_unique",
        summary_duplicate_count,
        "0",
        summary_duplicate_count == 0,
        "Summary row ids are duplicated after adding Batch 5 rows.",
    ),
    validation_row(
        "batch5_failure_flag_column_populated",
        failure_flag_rows,
        ">= 0",
        failure_flag_rows >= 0,
        "Failure flag count could not be computed.",
    ),
]

batch5_validation_df = pd.DataFrame(batch5_validation_rows)
batch5_passed = bool(batch5_validation_df["passed"].all())

if missing_required_case_columns:
    display(
        batch5_validation_df.loc[
            ~batch5_validation_df["passed"],
            ["check_name", "actual", "expected", "failure_message"],
        ]
    )
    raise RuntimeError(
        "Batch 5 validation failed because required columns are missing. "
        "Rerun Batch 5 Cell 5 before Cell 7. Missing columns: "
        f"{missing_required_case_columns}"
    )

batch5_case_comparison_df.to_csv(BATCH3_CASE_COMPARISON_PATH, index=False)
batch5_full_summary_tables_df.to_csv(BATCH4_SUMMARY_TABLES_PATH, index=False)
batch5_validation_df.to_csv(BATCH5_VALIDATION_PATH, index=False)

stage_manifest = read_json_if_exists(STAGE_MANIFEST_PATH)
stage_manifest.update(
    {
        "notebook_id": globals().get("NOTEBOOK_ID", "27_multi_model_comparison"),
        "notebook_title": globals().get("NOTEBOOK_TITLE", "Multi-model comparison"),
        "stage": "batch5_disagreement_ranking_failure_flags",
        "stage_status": "passed" if batch5_passed else "failed",
        "updated_at_utc": utc_now_iso(),
        "outputs": {
            **stage_manifest.get("outputs", {}),
            "multi_model_case_comparison_csv": rel(BATCH3_CASE_COMPARISON_PATH),
            "multi_model_summary_tables_csv": rel(BATCH4_SUMMARY_TABLES_PATH),
            "batch5_validation_csv": rel(BATCH5_VALIDATION_PATH),
        },
        "batch5": {
            "case_comparison_rows": int(len(batch5_case_comparison_df)),
            "rankable_metric_rows": rankable_metric_rows,
            "failure_flag_rows": failure_flag_rows,
            "batch5_summary_rows": batch5_summary_row_count,
            "deterministic_or_non_sampling_models": batch5_deterministic_observed_slugs,
            "generative_diffusion_models": batch5_generative_observed_slugs,
            "note": (
                "Rankings are direction-aware. Metrics with unknown direction or insufficient finite model values "
                "are flagged and excluded from ranking stability calculations."
            ),
        },
    }
)

write_json(STAGE_MANIFEST_PATH, stage_manifest)

batch3_case_comparison_df = batch5_case_comparison_df.copy()
batch4_summary_tables_df = batch5_full_summary_tables_df.copy()

print(f"Batch 5 checks passed: {int(batch5_validation_df['passed'].sum())} / {len(batch5_validation_df)}")
display(batch5_validation_df)

print(f"Updated case comparison: {rel(BATCH3_CASE_COMPARISON_PATH)}")
print(f"Updated summary tables: {rel(BATCH4_SUMMARY_TABLES_PATH)}")
print(f"Saved Batch 5 validation: {rel(BATCH5_VALIDATION_PATH)}")

group_columns = [
    "batch5_metric_disagreement_flag",
    "batch5_ranking_stability_label",
]

if all(column in batch5_case_comparison_df.columns for column in group_columns):
    display(
        batch5_case_comparison_df.groupby(group_columns, dropna=False)
        .agg(
            rows=("case_id", "size"),
            rankable_rows=("batch5_rankable_metric", "sum"),
            failure_flag_rows=(
                "batch5_failure_flag_count",
                lambda values: int((pd.to_numeric(values, errors="coerce") > 0).sum()),
            ),
        )
        .reset_index()
    )
else:
    missing_group_columns = [
        column for column in group_columns
        if column not in batch5_case_comparison_df.columns
    ]
    print(f"Skipping Batch 5 grouped preview because columns are missing: {missing_group_columns}")

if not batch5_passed:
    display(batch5_validation_df.loc[~batch5_validation_df["passed"], ["check_name", "actual", "expected", "failure_message"]])
    raise RuntimeError("Batch 5 validation failed. Fix disagreement/ranking/failure logic before Batch 6.")

print("Batch 5 passed. Disagreement, ranking stability, and failure flags are ready for the final report.")

Batch 5 checks passed: 7 / 7


,check_name,actual,expected,passed,failure_message
0,batch5_case_table_has_rows,27285,> 0,True,
1,batch5_required_case_columns_present,[],[],True,
2,batch5_case_keys_unique,0,0,True,
3,batch5_rankable_metric_rows_present,12610,> 0,True,
4,batch5_summary_rows_present,197,> 0,True,
5,batch5_summary_row_ids_unique,0,0,True,
6,batch5_failure_flag_column_populated,4854,>= 0,True,


Updated case comparison: outputs/27_multi_model_comparison/metrics/multi_model_case_comparison.csv
Updated summary tables: outputs/27_multi_model_comparison/analysis/multi_model_summary_tables.csv
Saved Batch 5 validation: outputs/27_multi_model_comparison/validation/batch5_disagreement_ranking_failure_validation.csv


,batch5_metric_disagreement_flag,batch5_ranking_stability_label,rows,rankable_rows,failure_flag_rows
0,high,mixed,589,589,589
1,high,stable,404,404,404
2,high,unstable,1901,1901,1901
3,insufficient_models,no_rankable_metrics,14675,0,0
4,low,mixed,1237,1237,0
5,low,stable,2870,2870,0
6,low,unstable,755,755,0
7,moderate,mixed,607,607,0
8,moderate,stable,1749,1749,0
9,moderate,unstable,538,538,0


Batch 5 passed. Disagreement, ranking stability, and failure flags are ready for the final report.


In [37]:
# Batch 6 / Cell 1 - Load Batch 5 outputs and validate final-report readiness

batch6_case_comparison_df = pd.read_csv(BATCH3_CASE_COMPARISON_PATH)
batch6_summary_tables_df = pd.read_csv(BATCH4_SUMMARY_TABLES_PATH)

batch6_required_columns = [
    "case_id",
    "evaluation_region",
    "metric_name",
    "batch5_rankable_metric",
    "batch5_metric_disagreement_flag",
    "batch5_ranking_stability_label",
    "batch5_coverage_flag_count",
    "batch5_warning_flag_count",
    "batch5_true_failure_flag_count",
    "batch5_failure_flag_count",
    "batch5_coverage_flags_json",
    "batch5_warning_flags_json",
    "batch5_true_failure_flags_json",
    "batch5_failure_flags_json",
]

missing_batch6_columns = [
    column for column in batch6_required_columns
    if column not in batch6_case_comparison_df.columns
]

if missing_batch6_columns:
    raise RuntimeError(f"Batch 6 cannot start. Missing columns: {missing_batch6_columns}")

batch6_total_rows = len(batch6_case_comparison_df)
batch6_rankable_rows = int(batch6_case_comparison_df["batch5_rankable_metric"].astype(bool).sum())
batch6_insufficient_rows = int(
    batch6_case_comparison_df["batch5_metric_disagreement_flag"]
    .astype(str)
    .eq("insufficient_models")
    .sum()
)
batch6_warning_rows = int(batch6_case_comparison_df["batch5_warning_flag_count"].gt(0).sum())
batch6_true_failure_rows = int(batch6_case_comparison_df["batch5_true_failure_flag_count"].gt(0).sum())
batch6_legacy_failure_rows = int(batch6_case_comparison_df["batch5_failure_flag_count"].gt(0).sum())
batch6_coverage_limited_rows = int(batch6_case_comparison_df["batch5_coverage_flag_count"].gt(0).sum())

if "batch5_cross_family_comparison_available" in batch6_case_comparison_df.columns:
    batch6_cross_family_available_rows = int(
        batch6_case_comparison_df["batch5_cross_family_comparison_available"].astype(bool).sum()
    )
else:
    batch6_cross_family_available_rows = 0

print("Batch 6 readiness check")
print(f"Rows: {batch6_total_rows:,}")
print(f"Rankable rows: {batch6_rankable_rows:,}")
print(f"Insufficient-model rows: {batch6_insufficient_rows:,}")
print(f"Coverage-limited rows: {batch6_coverage_limited_rows:,}")
print(f"Warning rows: {batch6_warning_rows:,}")
print(f"True failure rows: {batch6_true_failure_rows:,}")
print(f"Legacy failure/warning rows: {batch6_legacy_failure_rows:,}")
print(f"Cross-family comparable rows: {batch6_cross_family_available_rows:,}")

E:\HFCache\tmp\ipykernel_11960\1976935495.py:3: DtypeWarning: Columns (8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  batch6_case_comparison_df = pd.read_csv(BATCH3_CASE_COMPARISON_PATH)


Batch 6 readiness check
Rows: 27,285
Rankable rows: 12,610
Insufficient-model rows: 14,675
Coverage-limited rows: 27,285
Warning rows: 4,854
True failure rows: 0
Legacy failure/warning rows: 4,854
Cross-family comparable rows: 0


In [38]:
# Batch 6 / Cell 1 - Paths, load Batch 5 outputs, readiness check

from pathlib import Path
import json
import math
import shutil
import textwrap
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont

NOTEBOOK_ID = globals().get("NOTEBOOK_ID", "27_multi_model_comparison")
NOTEBOOK_TITLE = globals().get("NOTEBOOK_TITLE", "Multi-model comparison")

NOTEBOOK_OUTPUT_DIR = Path(globals().get("NOTEBOOK_OUTPUT_DIR", Path("outputs") / NOTEBOOK_ID))
METRICS_DIR = Path(globals().get("METRICS_DIR", NOTEBOOK_OUTPUT_DIR / "metrics"))
ANALYSIS_DIR = Path(globals().get("ANALYSIS_DIR", NOTEBOOK_OUTPUT_DIR / "analysis"))
FIGURES_DIR = Path(globals().get("FIGURES_DIR", NOTEBOOK_OUTPUT_DIR / "figures"))
VALIDATION_DIR = Path(globals().get("VALIDATION_DIR", NOTEBOOK_OUTPUT_DIR / "validation"))

for directory in [METRICS_DIR, ANALYSIS_DIR, FIGURES_DIR, VALIDATION_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

BATCH6_REPRESENTATIVE_CASES_PATH = ANALYSIS_DIR / "multi_model_representative_cases.csv"
BATCH6_FIGURE_MANIFEST_PATH = FIGURES_DIR / "multi_model_figure_manifest.csv"
BATCH6_PANEL_DIR = FIGURES_DIR / "multi_model_comparison_panels"
BATCH6_VALIDATION_PATH = VALIDATION_DIR / "batch6_representative_cases_validation.csv"

BATCH6_PANEL_DIR.mkdir(parents=True, exist_ok=True)

if "BATCH3_CASE_COMPARISON_PATH" not in globals():
    BATCH3_CASE_COMPARISON_PATH = METRICS_DIR / "multi_model_case_comparison.csv"

if "BATCH4_SUMMARY_TABLES_PATH" not in globals():
    BATCH4_SUMMARY_TABLES_PATH = ANALYSIS_DIR / "multi_model_summary_tables.csv"

if "STAGE_MANIFEST_PATH" not in globals():
    STAGE_MANIFEST_PATH = NOTEBOOK_OUTPUT_DIR / "stage_manifest.json"

def batch6_clean_text(value, default=""):
    if pd.isna(value):
        return default
    return str(value).strip()

def batch6_safe_json_loads(value):
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    text_value = str(value).strip()
    if not text_value:
        return []
    try:
        loaded = json.loads(text_value)
        return loaded if isinstance(loaded, list) else []
    except Exception:
        return []

def batch6_safe_json_dumps(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True)

def batch6_rel(path):
    if "rel" in globals():
        return rel(path)
    try:
        return str(Path(path).relative_to(Path.cwd()))
    except Exception:
        return str(path)

def batch6_validation_row(check_name, actual, expected, passed, failure_message):
    if "validation_row" in globals():
        return validation_row(check_name, actual, expected, passed, failure_message)
    return {
        "check_name": check_name,
        "actual": actual,
        "expected": expected,
        "passed": bool(passed),
        "failure_message": failure_message,
    }

def batch6_read_json_if_exists(path):
    if "read_json_if_exists" in globals():
        return read_json_if_exists(path)
    path = Path(path)
    if not path.exists():
        return {}
    return json.loads(path.read_text(encoding="utf-8"))

def batch6_write_json(path, payload):
    if "write_json" in globals():
        write_json(path, payload)
    else:
        Path(path).write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")

def batch6_utc_now_iso():
    if "utc_now_iso" in globals():
        return utc_now_iso()
    return pd.Timestamp.utcnow().isoformat()

batch6_case_comparison_df = pd.read_csv(BATCH3_CASE_COMPARISON_PATH)
batch6_summary_tables_df = pd.read_csv(BATCH4_SUMMARY_TABLES_PATH)

batch6_required_columns = [
    "case_id",
    "evaluation_region",
    "metric_name",
    "batch5_rankable_metric",
    "batch5_metric_disagreement_flag",
    "batch5_ranking_stability_label",
    "batch5_coverage_flag_count",
    "batch5_warning_flag_count",
    "batch5_true_failure_flag_count",
    "batch5_failure_flag_count",
    "batch5_coverage_flags_json",
    "batch5_warning_flags_json",
    "batch5_true_failure_flags_json",
    "batch5_failure_flags_json",
]

missing_batch6_columns = [
    column for column in batch6_required_columns
    if column not in batch6_case_comparison_df.columns
]

if missing_batch6_columns:
    raise RuntimeError(f"Batch 6 cannot start. Missing required Batch 5 columns: {missing_batch6_columns}")

print("Batch 6 inputs loaded.")
print(f"Case comparison rows: {len(batch6_case_comparison_df):,}")
print(f"Summary rows: {len(batch6_summary_tables_df):,}")
print(f"Representative cases target: {batch6_rel(BATCH6_REPRESENTATIVE_CASES_PATH)}")
print(f"Figure manifest target: {batch6_rel(BATCH6_FIGURE_MANIFEST_PATH)}")

E:\HFCache\tmp\ipykernel_11960\545205634.py:102: DtypeWarning: Columns (8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  batch6_case_comparison_df = pd.read_csv(BATCH3_CASE_COMPARISON_PATH)


Batch 6 inputs loaded.
Case comparison rows: 27,285
Summary rows: 10,127
Representative cases target: outputs/27_multi_model_comparison/analysis/multi_model_representative_cases.csv
Figure manifest target: outputs/27_multi_model_comparison/figures/multi_model_figure_manifest.csv


In [39]:
# Batch 6 / Cell 2 - Build deterministic representative-case selections

batch6_work_df = batch6_case_comparison_df.copy()

batch6_work_df["batch6_warning_flags"] = batch6_work_df["batch5_warning_flags_json"].map(batch6_safe_json_loads)
batch6_work_df["batch6_coverage_flags"] = batch6_work_df["batch5_coverage_flags_json"].map(batch6_safe_json_loads)
batch6_work_df["batch6_failure_flags"] = batch6_work_df["batch5_true_failure_flags_json"].map(batch6_safe_json_loads)

for column in [
    "batch5_rankable_metric",
    "batch5_coverage_flag_count",
    "batch5_warning_flag_count",
    "batch5_true_failure_flag_count",
    "batch5_failure_flag_count",
]:
    batch6_work_df[column] = pd.to_numeric(batch6_work_df[column], errors="coerce").fillna(0)

batch6_work_df["batch6_has_top_rank_tie"] = batch6_work_df["batch6_warning_flags"].map(lambda flags: "top_rank_tie" in flags)
batch6_work_df["batch6_has_high_disagreement"] = batch6_work_df["batch5_metric_disagreement_flag"].astype(str).eq("high")
batch6_work_df["batch6_is_zero_control"] = batch6_work_df["case_id"].astype(str).str.contains("zero_control", case=False, na=False)

disagreement_priority = {
    "high": 5,
    "moderate": 4,
    "low": 3,
    "none": 2,
    "insufficient_models": 1,
}

stability_priority = {
    "unstable": 5,
    "mixed": 4,
    "stable": 3,
    "no_rankable_metrics": 1,
}

batch6_work_df["batch6_disagreement_priority"] = (
    batch6_work_df["batch5_metric_disagreement_flag"].astype(str).map(disagreement_priority).fillna(0)
)

batch6_work_df["batch6_stability_priority"] = (
    batch6_work_df["batch5_ranking_stability_label"].astype(str).map(stability_priority).fillna(0)
)

batch6_work_df["batch6_selection_score"] = (
    batch6_work_df["batch6_disagreement_priority"] * 100
    + batch6_work_df["batch6_stability_priority"] * 10
    + batch6_work_df["batch5_warning_flag_count"].astype(float)
    + batch6_work_df["batch5_rankable_metric"].astype(float)
)

batch6_selection_rules = [
    {
        "selection_reason": "high_disagreement_unstable",
        "max_rows": 8,
        "mask": (
            batch6_work_df["batch5_metric_disagreement_flag"].astype(str).eq("high")
            & batch6_work_df["batch5_ranking_stability_label"].astype(str).eq("unstable")
        ),
    },
    {
        "selection_reason": "high_disagreement_mixed_or_stable",
        "max_rows": 6,
        "mask": (
            batch6_work_df["batch5_metric_disagreement_flag"].astype(str).eq("high")
            & batch6_work_df["batch5_ranking_stability_label"].astype(str).isin(["mixed", "stable"])
        ),
    },
    {
        "selection_reason": "moderate_or_low_unstable",
        "max_rows": 8,
        "mask": (
            batch6_work_df["batch5_metric_disagreement_flag"].astype(str).isin(["moderate", "low"])
            & batch6_work_df["batch5_ranking_stability_label"].astype(str).eq("unstable")
        ),
    },
    {
        "selection_reason": "top_rank_tie",
        "max_rows": 8,
        "mask": batch6_work_df["batch6_has_top_rank_tie"],
    },
    {
        "selection_reason": "insufficient_models_coverage_limit",
        "max_rows": 8,
        "mask": batch6_work_df["batch5_metric_disagreement_flag"].astype(str).eq("insufficient_models"),
    },
    {
        "selection_reason": "zero_control_sanity_check",
        "max_rows": 6,
        "mask": batch6_work_df["batch6_is_zero_control"],
    },
    {
        "selection_reason": "stable_low_or_none_baseline",
        "max_rows": 8,
        "mask": (
            batch6_work_df["batch5_metric_disagreement_flag"].astype(str).isin(["none", "low"])
            & batch6_work_df["batch5_ranking_stability_label"].astype(str).eq("stable")
            & batch6_work_df["batch5_warning_flag_count"].eq(0)
        ),
    },
]

selected_batches = []

for rule_index, rule in enumerate(batch6_selection_rules, start=1):
    rule_df = batch6_work_df.loc[rule["mask"]].copy()
    if rule_df.empty:
        continue

    rule_df["batch6_selection_reason"] = rule["selection_reason"]
    rule_df["batch6_selection_rule_order"] = rule_index

    sort_columns = [
        "batch6_selection_score",
        "case_id",
        "evaluation_region",
        "metric_name",
    ]

    rule_df = (
        rule_df.sort_values(sort_columns, ascending=[False, True, True, True], kind="mergesort")
        .drop_duplicates(["case_id", "evaluation_region", "metric_name"])
        .head(rule["max_rows"])
    )

    selected_batches.append(rule_df)

if not selected_batches:
    raise RuntimeError("Batch 6 could not select any representative rows. Check Batch 5 outputs.")

batch6_representative_cases_df = (
    pd.concat(selected_batches, ignore_index=True, sort=False)
    .drop_duplicates(["case_id", "evaluation_region", "metric_name", "batch6_selection_reason"])
    .sort_values(
        ["batch6_selection_rule_order", "batch6_selection_score", "case_id", "evaluation_region", "metric_name"],
        ascending=[True, False, True, True, True],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

batch6_representative_cases_df["batch6_representative_case_id"] = [
    f"rep_{idx + 1:03d}_{batch6_clean_text(row.case_id)}"
    for idx, row in batch6_representative_cases_df.iterrows()
]

batch6_output_columns = [
    "batch6_representative_case_id",
    "batch6_selection_reason",
    "batch6_selection_rule_order",
    "case_id",
    "evaluation_region",
    "metric_name",
    "batch5_rankable_metric",
    "batch5_metric_disagreement_flag",
    "batch5_ranking_stability_label",
    "batch5_coverage_flags_json",
    "batch5_warning_flags_json",
    "batch5_true_failure_flags_json",
    "batch5_failure_flags_json",
    "batch5_coverage_flag_count",
    "batch5_warning_flag_count",
    "batch5_true_failure_flag_count",
    "batch5_failure_flag_count",
]

batch6_output_columns = [
    column for column in batch6_output_columns
    if column in batch6_representative_cases_df.columns
]

batch6_representative_cases_df[batch6_output_columns].to_csv(BATCH6_REPRESENTATIVE_CASES_PATH, index=False)

print(f"Selected representative rows: {len(batch6_representative_cases_df):,}")
print(f"Saved: {batch6_rel(BATCH6_REPRESENTATIVE_CASES_PATH)}")

display(
    batch6_representative_cases_df[
        [
            "batch6_representative_case_id",
            "batch6_selection_reason",
            "case_id",
            "evaluation_region",
            "metric_name",
            "batch5_metric_disagreement_flag",
            "batch5_ranking_stability_label",
            "batch5_warning_flags_json",
        ]
    ].head(30)
)

Selected representative rows: 52
Saved: outputs/27_multi_model_comparison/analysis/multi_model_representative_cases.csv


,batch6_representative_case_id,batch6_selection_reason,case_id,evaluation_region,metric_name,batch5_metric_disagreement_flag,batch5_ranking_stability_label,batch5_warning_flags_json
0,rep_001_canonical__p001_loss_large,high_disagreement_unstable,canonical__p001_loss_large,content_region,clip_similarity_improvement,high,unstable,"[""high_metric_disagreement""]"
1,rep_002_canonical__p001_loss_large,high_disagreement_unstable,canonical__p001_loss_large,content_region,dinov2_similarity_improvement,high,unstable,"[""high_metric_disagreement""]"
2,rep_003_canonical__p001_loss_large,high_disagreement_unstable,canonical__p001_loss_large,content_region,mae_improvement,high,unstable,"[""high_metric_disagreement""]"
3,rep_004_canonical__p001_loss_large,high_disagreement_unstable,canonical__p001_loss_large,content_region,mean_similarity_improvement,high,unstable,"[""high_metric_disagreement""]"
4,rep_005_canonical__p001_loss_large,high_disagreement_unstable,canonical__p001_loss_large,full_image,clip_similarity_improvement,high,unstable,"[""high_metric_disagreement""]"
5,rep_006_canonical__p001_loss_large,high_disagreement_unstable,canonical__p001_loss_large,full_image,dinov2_similarity_improvement,high,unstable,"[""high_metric_disagreement""]"
6,rep_007_canonical__p001_loss_large,high_disagreement_unstable,canonical__p001_loss_large,full_image,mae_improvement,high,unstable,"[""high_metric_disagreement""]"
7,rep_008_canonical__p001_loss_large,high_disagreement_unstable,canonical__p001_loss_large,full_image,mean_similarity_improvement,high,unstable,"[""high_metric_disagreement""]"
8,rep_009_canonical__p001_loss_large,high_disagreement_mixed_or_stable,canonical__p001_loss_large,boundary_region,mae_improvement,high,mixed,"[""high_metric_disagreement""]"
9,rep_010_canonical__p001_loss_large,high_disagreement_mixed_or_stable,canonical__p001_loss_large,mask_bbox_crop,clip_similarity_improvement,high,mixed,"[""high_metric_disagreement""]"


In [42]:
# Batch 6 / Cell 3 - Discover available image/path columns for panels
# Fixed: snapshot globals before iteration so notebook variable creation does not mutate the dict mid-loop.

batch6_path_column_patterns = [
    "path",
    "image",
    "clean",
    "damaged",
    "restored",
    "inpainted",
    "mask",
]

batch6_candidate_frames = {}

# Important in notebooks: assignments at top level mutate globals().
# So we iterate over a snapshot, not globals().items() directly.
batch6_global_items_snapshot = list(globals().items())

for name, value in batch6_global_items_snapshot:
    if isinstance(value, pd.DataFrame):
        columns_lower = [str(column).lower() for column in value.columns]
        has_case_id = "case_id" in value.columns
        has_path_like = any(
            any(pattern in column for pattern in batch6_path_column_patterns)
            for column in columns_lower
        )

        if has_case_id and has_path_like:
            batch6_candidate_frames[name] = value

print("Candidate dataframes with case_id and path-like columns:")
for frame_name, frame in batch6_candidate_frames.items():
    path_like_columns = [
        column for column in frame.columns
        if any(pattern in str(column).lower() for pattern in batch6_path_column_patterns)
    ]
    print(f"- {frame_name}: rows={len(frame):,}, path-like columns={path_like_columns[:20]}")

batch6_path_lookup_rows = []

for source_name, source_frame in batch6_candidate_frames.items():
    frame_copy = source_frame.copy()

    if "case_id" not in frame_copy.columns:
        continue

    path_like_columns = [
        column for column in frame_copy.columns
        if any(pattern in str(column).lower() for pattern in batch6_path_column_patterns)
    ]

    for _, row in frame_copy.iterrows():
        case_id = batch6_clean_text(row.get("case_id"))

        if not case_id:
            continue

        model_slug = batch6_clean_text(
            row.get(
                "model_slug",
                row.get("restoration_method", row.get("method", "")),
            ),
            default="",
        )

        candidate_label = model_slug if model_slug else source_name

        for column in path_like_columns:
            raw_path = row.get(column)

            if pd.isna(raw_path):
                continue

            raw_path = str(raw_path).strip()

            if not raw_path:
                continue

            image_path = Path(raw_path)

            if not image_path.is_absolute():
                image_path = Path.cwd() / image_path

            if image_path.exists() and image_path.suffix.lower() in [".png", ".jpg", ".jpeg", ".webp", ".bmp"]:
                batch6_path_lookup_rows.append(
                    {
                        "case_id": case_id,
                        "source_dataframe": source_name,
                        "source_column": str(column),
                        "image_role": str(column),
                        "model_slug": candidate_label,
                        "image_path": str(image_path),
                    }
                )

batch6_image_lookup_df = pd.DataFrame(batch6_path_lookup_rows)

if batch6_image_lookup_df.empty:
    print("No existing image paths were found in loaded dataframes.")
    print(
        "Batch 6 will still write the representative cases CSV, "
        "but panel rendering will be skipped unless image paths are available."
    )
else:
    batch6_image_lookup_df = (
        batch6_image_lookup_df
        .drop_duplicates(["case_id", "image_role", "model_slug", "image_path"])
        .sort_values(["case_id", "image_role", "model_slug", "image_path"], kind="mergesort")
        .reset_index(drop=True)
    )

    print(f"Discovered image references: {len(batch6_image_lookup_df):,}")
    display(batch6_image_lookup_df.head(40))

Candidate dataframes with case_id and path-like columns:
- source_df: rows=945, path-like columns=['mask_seed', 'effect_mask_seed', 'difference_map__damaged_error_map_path', 'difference_map__damaged_error_map_filename', 'difference_map__damaged_error_filename_length', 'difference_map__damaged_error_file_size_bytes', 'difference_map__restored_error_map_path', 'difference_map__restored_error_map_filename', 'difference_map__restored_error_filename_length', 'difference_map__restored_error_file_size_bytes', 'difference_map__damaged_error_mean_full', 'difference_map__damaged_error_std_full', 'difference_map__damaged_error_max_full', 'difference_map__restored_error_mean_full', 'difference_map__restored_error_std_full', 'difference_map__restored_error_max_full', 'difference_map__damaged_error_masked_mean', 'difference_map__damaged_error_masked_std', 'difference_map__restored_error_masked_mean', 'difference_map__restored_error_masked_std']
- working_df: rows=2,724, path-like columns=['source_re

In [43]:
# Batch 6 / Cell 4 - Render comparison panels where image paths are available

def batch6_load_panel_image(path, target_size=(256, 256)):
    image = Image.open(path).convert("RGB")
    image.thumbnail(target_size, Image.Resampling.LANCZOS)

    canvas = Image.new("RGB", target_size, "white")
    x = (target_size[0] - image.width) // 2
    y = (target_size[1] - image.height) // 2
    canvas.paste(image, (x, y))
    return canvas

def batch6_wrap_text(text, width=28):
    text = batch6_clean_text(text)
    if not text:
        return ""
    return "\n".join(textwrap.wrap(text, width=width))

def batch6_render_panel(rep_row, image_rows, output_path):
    tile_size = (256, 256)
    label_height = 74
    title_height = 120
    padding = 18
    max_images = 8

    image_rows = image_rows.head(max_images).copy()
    tile_count = len(image_rows)

    if tile_count == 0:
        return False

    columns = min(4, tile_count)
    rows = int(math.ceil(tile_count / columns))

    panel_width = padding * 2 + columns * tile_size[0]
    panel_height = title_height + rows * (tile_size[1] + label_height) + padding

    panel = Image.new("RGB", (panel_width, panel_height), "white")
    draw = ImageDraw.Draw(panel)

    title = (
        f"{rep_row['batch6_representative_case_id']}\n"
        f"{rep_row['case_id']} | {rep_row['evaluation_region']} | {rep_row['metric_name']}\n"
        f"{rep_row['batch6_selection_reason']} | disagreement={rep_row['batch5_metric_disagreement_flag']} | "
        f"stability={rep_row['batch5_ranking_stability_label']}"
    )

    draw.multiline_text((padding, padding), batch6_wrap_text(title, width=110), fill=(20, 20, 20), spacing=4)

    start_y = title_height

    for idx, (_, image_row) in enumerate(image_rows.iterrows()):
        row_idx = idx // columns
        col_idx = idx % columns

        x = padding + col_idx * tile_size[0]
        y = start_y + row_idx * (tile_size[1] + label_height)

        try:
            tile = batch6_load_panel_image(image_row["image_path"], target_size=tile_size)
        except Exception:
            continue

        panel.paste(tile, (x, y))

        label = (
            f"{image_row.get('model_slug', '')}\n"
            f"{image_row.get('image_role', '')}"
        )

        draw.multiline_text(
            (x + 4, y + tile_size[1] + 6),
            batch6_wrap_text(label, width=30),
            fill=(30, 30, 30),
            spacing=3,
        )

    output_path.parent.mkdir(parents=True, exist_ok=True)
    panel.save(output_path)
    return True

batch6_figure_manifest_rows = []

if "batch6_image_lookup_df" not in globals() or batch6_image_lookup_df.empty:
    print("No image lookup table available. Skipping panel rendering.")
else:
    for _, rep_row in batch6_representative_cases_df.iterrows():
        case_id = batch6_clean_text(rep_row["case_id"])
        rep_id = batch6_clean_text(rep_row["batch6_representative_case_id"])

        image_rows = batch6_image_lookup_df.loc[
            batch6_image_lookup_df["case_id"].astype(str).eq(case_id)
        ].copy()

        if image_rows.empty:
            batch6_figure_manifest_rows.append(
                {
                    "figure_id": f"batch6_panel_{rep_id}",
                    "case_id": case_id,
                    "evaluation_region": rep_row["evaluation_region"],
                    "metric_name": rep_row["metric_name"],
                    "selection_reason": rep_row["batch6_selection_reason"],
                    "figure_path": "",
                    "figure_type": "multi_model_comparison_panel",
                    "render_status": "skipped_no_image_paths",
                    "image_count": 0,
                    "notes": "No discoverable image paths were available for this case in loaded notebook dataframes.",
                }
            )
            continue

        output_path = BATCH6_PANEL_DIR / f"{rep_id}.png"
        rendered = batch6_render_panel(rep_row, image_rows, output_path)

        batch6_figure_manifest_rows.append(
            {
                "figure_id": f"batch6_panel_{rep_id}",
                "case_id": case_id,
                "evaluation_region": rep_row["evaluation_region"],
                "metric_name": rep_row["metric_name"],
                "selection_reason": rep_row["batch6_selection_reason"],
                "figure_path": batch6_rel(output_path) if rendered else "",
                "figure_type": "multi_model_comparison_panel",
                "render_status": "rendered" if rendered else "failed_render",
                "image_count": int(len(image_rows)),
                "notes": (
                    "Panel uses discoverable image paths from loaded notebook dataframes. "
                    "Image order is deterministic but depends on available path metadata."
                ),
            }
        )

batch6_figure_manifest_df = pd.DataFrame(batch6_figure_manifest_rows)

if batch6_figure_manifest_df.empty:
    batch6_figure_manifest_df = pd.DataFrame(
        columns=[
            "figure_id",
            "case_id",
            "evaluation_region",
            "metric_name",
            "selection_reason",
            "figure_path",
            "figure_type",
            "render_status",
            "image_count",
            "notes",
        ]
    )

batch6_figure_manifest_df.to_csv(BATCH6_FIGURE_MANIFEST_PATH, index=False)

print(f"Figure manifest rows: {len(batch6_figure_manifest_df):,}")
print(f"Rendered panels: {int(batch6_figure_manifest_df['render_status'].eq('rendered').sum()) if len(batch6_figure_manifest_df) else 0:,}")
print(f"Saved: {batch6_rel(BATCH6_FIGURE_MANIFEST_PATH)}")

display(batch6_figure_manifest_df.head(30))

No image lookup table available. Skipping panel rendering.
Figure manifest rows: 0
Rendered panels: 0
Saved: outputs/27_multi_model_comparison/figures/multi_model_figure_manifest.csv


,figure_id,case_id,evaluation_region,metric_name,selection_reason,figure_path,figure_type,render_status,image_count,notes


In [44]:
# Batch 6 / Cell 5 - Validate outputs and update manifest

batch6_representative_cases_df = pd.read_csv(BATCH6_REPRESENTATIVE_CASES_PATH)
batch6_figure_manifest_df = pd.read_csv(BATCH6_FIGURE_MANIFEST_PATH)

batch6_rendered_panel_count = (
    int(batch6_figure_manifest_df["render_status"].astype(str).eq("rendered").sum())
    if "render_status" in batch6_figure_manifest_df.columns
    else 0
)

batch6_selection_reason_count = (
    int(batch6_representative_cases_df["batch6_selection_reason"].nunique())
    if "batch6_selection_reason" in batch6_representative_cases_df.columns
    else 0
)

batch6_validation_rows = [
    batch6_validation_row(
        "batch6_representative_cases_written",
        len(batch6_representative_cases_df),
        "> 0",
        len(batch6_representative_cases_df) > 0,
        "Representative cases CSV is empty.",
    ),
    batch6_validation_row(
        "batch6_multiple_selection_reasons_present",
        batch6_selection_reason_count,
        ">= 3",
        batch6_selection_reason_count >= 3,
        "Representative selection is too narrow; expected at least three selection reasons.",
    ),
    batch6_validation_row(
        "batch6_figure_manifest_written",
        len(batch6_figure_manifest_df),
        ">= 0",
        len(batch6_figure_manifest_df) >= 0,
        "Figure manifest could not be written.",
    ),
    batch6_validation_row(
        "batch6_panel_rendering_status_recorded",
        sorted(batch6_figure_manifest_df["render_status"].dropna().astype(str).unique().tolist())
        if "render_status" in batch6_figure_manifest_df.columns
        else [],
        "rendered or skipped/failed status",
        "render_status" in batch6_figure_manifest_df.columns,
        "Figure manifest is missing render_status.",
    ),
]

batch6_validation_df = pd.DataFrame(batch6_validation_rows)
batch6_passed = bool(batch6_validation_df["passed"].all())

batch6_validation_df.to_csv(BATCH6_VALIDATION_PATH, index=False)

stage_manifest = batch6_read_json_if_exists(STAGE_MANIFEST_PATH)
stage_manifest.update(
    {
        "notebook_id": NOTEBOOK_ID,
        "notebook_title": NOTEBOOK_TITLE,
        "stage": "batch6_representative_case_selection_and_comparison_panels",
        "stage_status": "passed" if batch6_passed else "failed",
        "updated_at_utc": batch6_utc_now_iso(),
        "outputs": {
            **stage_manifest.get("outputs", {}),
            "multi_model_representative_cases_csv": batch6_rel(BATCH6_REPRESENTATIVE_CASES_PATH),
            "multi_model_figure_manifest_csv": batch6_rel(BATCH6_FIGURE_MANIFEST_PATH),
            "batch6_validation_csv": batch6_rel(BATCH6_VALIDATION_PATH),
        },
        "batch6": {
            "representative_case_rows": int(len(batch6_representative_cases_df)),
            "selection_reason_count": batch6_selection_reason_count,
            "figure_manifest_rows": int(len(batch6_figure_manifest_df)),
            "rendered_panel_count": batch6_rendered_panel_count,
            "panel_directory": batch6_rel(BATCH6_PANEL_DIR),
            "note": (
                "Representative cases are selected deterministically from Batch 5 disagreement, "
                "ranking stability, warning, coverage-limit, and zero-control signals. "
                "Panel rendering depends on discoverable image paths in loaded notebook dataframes."
            ),
        },
    }
)

batch6_write_json(STAGE_MANIFEST_PATH, stage_manifest)

print(f"Batch 6 checks passed: {int(batch6_validation_df['passed'].sum())} / {len(batch6_validation_df)}")
display(batch6_validation_df)

print(f"Saved representative cases: {batch6_rel(BATCH6_REPRESENTATIVE_CASES_PATH)}")
print(f"Saved figure manifest: {batch6_rel(BATCH6_FIGURE_MANIFEST_PATH)}")
print(f"Saved validation: {batch6_rel(BATCH6_VALIDATION_PATH)}")

if not batch6_passed:
    display(batch6_validation_df.loc[~batch6_validation_df["passed"], ["check_name", "actual", "expected", "failure_message"]])
    raise RuntimeError("Batch 6 validation failed. Fix representative selection/panel manifest before continuing.")

print("Batch 6 passed. Representative cases and comparison-panel manifest are ready.")

Batch 6 checks passed: 4 / 4


,check_name,actual,expected,passed,failure_message
0,batch6_representative_cases_written,52,> 0,True,
1,batch6_multiple_selection_reasons_present,7,>= 3,True,
2,batch6_figure_manifest_written,0,>= 0,True,
3,batch6_panel_rendering_status_recorded,[],rendered or skipped/failed status,True,


Saved representative cases: outputs/27_multi_model_comparison/analysis/multi_model_representative_cases.csv
Saved figure manifest: outputs/27_multi_model_comparison/figures/multi_model_figure_manifest.csv
Saved validation: outputs/27_multi_model_comparison/validation/batch6_representative_cases_validation.csv
Batch 6 passed. Representative cases and comparison-panel manifest are ready.


In [45]:
# Batch 7 / Cell 1 - Load outputs and define report paths

from pathlib import Path
import json
import html
import math
import os
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK_ID = globals().get("NOTEBOOK_ID", "27_multi_model_comparison")
NOTEBOOK_TITLE = globals().get("NOTEBOOK_TITLE", "Multi-model comparison")

NOTEBOOK_OUTPUT_DIR = Path(globals().get("NOTEBOOK_OUTPUT_DIR", Path("outputs") / NOTEBOOK_ID))
METRICS_DIR = Path(globals().get("METRICS_DIR", NOTEBOOK_OUTPUT_DIR / "metrics"))
ANALYSIS_DIR = Path(globals().get("ANALYSIS_DIR", NOTEBOOK_OUTPUT_DIR / "analysis"))
FIGURES_DIR = Path(globals().get("FIGURES_DIR", NOTEBOOK_OUTPUT_DIR / "figures"))
VALIDATION_DIR = Path(globals().get("VALIDATION_DIR", NOTEBOOK_OUTPUT_DIR / "validation"))
REPORTS_DIR = Path(globals().get("REPORTS_DIR", NOTEBOOK_OUTPUT_DIR / "reports"))
MANIFESTS_DIR = Path(globals().get("MANIFESTS_DIR", NOTEBOOK_OUTPUT_DIR / "manifests"))

BATCH7_PLOTS_DIR = FIGURES_DIR / "multi_model_report_plots"

for directory in [METRICS_DIR, ANALYSIS_DIR, FIGURES_DIR, VALIDATION_DIR, REPORTS_DIR, MANIFESTS_DIR, BATCH7_PLOTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

if "BATCH3_CASE_COMPARISON_PATH" not in globals():
    BATCH3_CASE_COMPARISON_PATH = METRICS_DIR / "multi_model_case_comparison.csv"

if "BATCH4_SUMMARY_TABLES_PATH" not in globals():
    BATCH4_SUMMARY_TABLES_PATH = ANALYSIS_DIR / "multi_model_summary_tables.csv"

if "BATCH6_REPRESENTATIVE_CASES_PATH" not in globals():
    BATCH6_REPRESENTATIVE_CASES_PATH = ANALYSIS_DIR / "multi_model_representative_cases.csv"

if "BATCH6_FIGURE_MANIFEST_PATH" not in globals():
    BATCH6_FIGURE_MANIFEST_PATH = FIGURES_DIR / "multi_model_figure_manifest.csv"

if "STAGE_MANIFEST_PATH" not in globals():
    STAGE_MANIFEST_PATH = NOTEBOOK_OUTPUT_DIR / "stage_manifest.json"

BATCH7_HTML_REPORT_PATH = REPORTS_DIR / "multi_model_comparison_report.html"
BATCH7_ARTIFACT_MANIFEST_PATH = MANIFESTS_DIR / "multi_model_comparison_manifest.json"
BATCH7_FINAL_VALIDATION_PATH = VALIDATION_DIR / "multi_model_final_validation.csv"

BATCH7_MAX_REPORT_IMAGES = globals().get("BATCH7_MAX_REPORT_IMAGES", 200)
BATCH7_MAX_TABLE_ROWS = globals().get("BATCH7_MAX_TABLE_ROWS", 80)

def batch7_clean_text(value, default=""):
    if value is None:
        return default
    try:
        if pd.isna(value):
            return default
    except Exception:
        pass
    return str(value).strip()

def batch7_safe_json_loads(value):
    if isinstance(value, list):
        return value
    if value is None:
        return []
    try:
        if pd.isna(value):
            return []
    except Exception:
        pass
    text_value = str(value).strip()
    if not text_value:
        return []
    try:
        loaded = json.loads(text_value)
        return loaded if isinstance(loaded, list) else []
    except Exception:
        return []

def batch7_safe_json_dumps(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True)

def batch7_rel(path):
    if "rel" in globals():
        return rel(path)
    try:
        return str(Path(path).relative_to(Path.cwd()))
    except Exception:
        return str(path)

def batch7_html_rel(path, from_path=BATCH7_HTML_REPORT_PATH):
    path = Path(path)
    if not path.is_absolute():
        path = Path.cwd() / path
    try:
        return os.path.relpath(path, start=from_path.parent)
    except Exception:
        return str(path)

def batch7_read_json_if_exists(path):
    if "read_json_if_exists" in globals():
        return read_json_if_exists(path)
    path = Path(path)
    if not path.exists():
        return {}
    return json.loads(path.read_text(encoding="utf-8"))

def batch7_write_json(path, payload):
    if "write_json" in globals():
        write_json(path, payload)
    else:
        Path(path).write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")

def batch7_utc_now_iso():
    if "utc_now_iso" in globals():
        return utc_now_iso()
    return pd.Timestamp.utcnow().isoformat()

def batch7_validation_row(check_name, actual, expected, passed, failure_message):
    if "validation_row" in globals():
        return validation_row(check_name, actual, expected, passed, failure_message)
    return {
        "check_name": check_name,
        "actual": actual,
        "expected": expected,
        "passed": bool(passed),
        "failure_message": failure_message,
    }

batch7_case_comparison_df = pd.read_csv(BATCH3_CASE_COMPARISON_PATH)
batch7_summary_tables_df = pd.read_csv(BATCH4_SUMMARY_TABLES_PATH)
batch7_representative_cases_df = pd.read_csv(BATCH6_REPRESENTATIVE_CASES_PATH)
batch7_figure_manifest_df = pd.read_csv(BATCH6_FIGURE_MANIFEST_PATH)

print("Batch 7 inputs loaded.")
print(f"Case comparison rows: {len(batch7_case_comparison_df):,}")
print(f"Summary rows: {len(batch7_summary_tables_df):,}")
print(f"Representative case rows: {len(batch7_representative_cases_df):,}")
print(f"Existing figure manifest rows: {len(batch7_figure_manifest_df):,}")
print(f"HTML report target: {batch7_rel(BATCH7_HTML_REPORT_PATH)}")

E:\HFCache\tmp\ipykernel_11960\1907993496.py:131: DtypeWarning: Columns (8,9,10) have mixed types. Specify dtype option on import or set low_memory=False.
  batch7_case_comparison_df = pd.read_csv(BATCH3_CASE_COMPARISON_PATH)


Batch 7 inputs loaded.
Case comparison rows: 27,285
Summary rows: 10,127
Representative case rows: 52
Existing figure manifest rows: 0
HTML report target: outputs/27_multi_model_comparison/reports/multi_model_comparison_report.html


In [52]:
# Batch 7 / Cell 2 hotfix - robust inventory discovery from project root

import os
import sys
import subprocess
from pathlib import Path

def batch7_find_project_root():
    if "PROJECT_ROOT" in globals():
        root = Path(PROJECT_ROOT).resolve()
        if (root / "outputs").exists() or (root / "tools").exists():
            return root

    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (
            (candidate / "tools" / "build_project_inventory.py").is_file()
            or (candidate / "src" / "restoration_eval").is_dir()
            or (candidate / "outputs").is_dir()
        ):
            return candidate

    return start

PROJECT_ROOT = batch7_find_project_root()
print("Batch 7 project root:", PROJECT_ROOT)

def batch7_resolve_project_path(path_value):
    path = Path(str(path_value))
    if path.is_absolute():
        return path
    return PROJECT_ROOT / path

def batch7_existing_path(path_value):
    if not batch7_clean_text(path_value):
        return None

    path = Path(str(path_value))
    candidates = []

    if path.is_absolute():
        candidates.append(path)
    else:
        candidates.extend([
            PROJECT_ROOT / path,
            Path.cwd().resolve() / path,
            path,
        ])

    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()

    return None

# Prefer the canonical generated inventory, then notebook snapshots, then uploaded inventories.
BATCH7_INVENTORY_CANDIDATES = [
    PROJECT_ROOT / "outputs" / "inventory" / "project_file_inventory.csv",
    NOTEBOOK_OUTPUT_DIR / "inventory" / "batch0_project_inventory_snapshot.csv",
    PROJECT_ROOT / "project_file_inventory.csv",
]

BATCH7_INVENTORY_CANDIDATES.extend(
    sorted((PROJECT_ROOT / "outputs").glob("*/inventory/*inventory*.csv"))
)

BATCH7_INVENTORY_CANDIDATES.extend(
    sorted((PROJECT_ROOT / "upload").glob("project_file_inventory*.csv"))
)

BATCH7_INVENTORY_CANDIDATES.extend(
    sorted((PROJECT_ROOT / "upload").glob("outputs_inventory*.csv"))
)

BATCH7_INVENTORY_CANDIDATES = list(dict.fromkeys(
    path.resolve() for path in BATCH7_INVENTORY_CANDIDATES
))

# If the canonical inventory is missing but the builder exists, regenerate it once.
canonical_inventory_path = PROJECT_ROOT / "outputs" / "inventory" / "project_file_inventory.csv"
inventory_builder_path = PROJECT_ROOT / "tools" / "build_project_inventory.py"

if not canonical_inventory_path.exists() and inventory_builder_path.exists():
    print("Canonical inventory missing. Regenerating project inventory...")
    subprocess.run(
        [
            sys.executable,
            str(inventory_builder_path),
            "--root",
            str(PROJECT_ROOT),
            "--out-dir",
            str(PROJECT_ROOT / "outputs" / "inventory"),
        ],
        cwd=PROJECT_ROOT,
        check=True,
    )

inventory_frames = []
for inventory_path in BATCH7_INVENTORY_CANDIDATES:
    if inventory_path.exists():
        frame = pd.read_csv(inventory_path)
        frame["inventory_source_path"] = str(inventory_path)
        inventory_frames.append(frame)

print("Inventory candidates checked:")
for path in BATCH7_INVENTORY_CANDIDATES[:30]:
    print(" -", path, "FOUND" if path.exists() else "missing")

if not inventory_frames:
    raise RuntimeError(
        "Batch 7 could not find any project inventory CSV even after project-root search/regeneration. "
        f"Resolved PROJECT_ROOT={PROJECT_ROOT}"
    )

batch7_inventory_df = pd.concat(inventory_frames, ignore_index=True, sort=False)
batch7_inventory_df = batch7_inventory_df.drop_duplicates("relative_path", keep="first").reset_index(drop=True)

metric_name_patterns = [
    "mse", "mae", "psnr", "ssim", "lpips", "clip", "dinov2",
    "similarity", "improvement", "error",
]

metric_source_mask = (
    batch7_inventory_df.get("extension", "").astype(str).str.lower().eq(".csv")
    & batch7_inventory_df.get("csv_columns", "").astype(str).str.contains("case_id", case=False, na=False)
    & batch7_inventory_df.get("csv_columns", "").astype(str).str.contains(
        "|".join(metric_name_patterns), case=False, na=False
    )
)

exclude_path_terms = [
    "summary", "selected", "ranked", "smoke", "validation",
    "figure_manifest", "input_cases", "correlation", "diagnostic",
]

batch7_metric_source_inventory_df = batch7_inventory_df.loc[metric_source_mask].copy()
batch7_metric_source_inventory_df = batch7_metric_source_inventory_df.loc[
    ~batch7_metric_source_inventory_df["relative_path"].astype(str).str.lower().apply(
        lambda value: any(term in value for term in exclude_path_terms)
    )
].copy()

batch7_source_frames = []

if "batch7_case_comparison_df" in globals() and isinstance(batch7_case_comparison_df, pd.DataFrame):
    existing_frame = batch7_case_comparison_df.copy()
    existing_frame["batch7_source_table"] = "batch7_case_comparison_df"
    existing_frame["batch7_source_path"] = batch7_rel(BATCH3_CASE_COMPARISON_PATH)
    batch7_source_frames.append(existing_frame)

for _, inventory_row in batch7_metric_source_inventory_df.iterrows():
    source_path = batch7_existing_path(inventory_row["relative_path"])
    if source_path is None:
        continue

    try:
        source_frame = pd.read_csv(source_path)
    except Exception as exc:
        print(f"Skipped unreadable metric source: {inventory_row['relative_path']} ({exc})")
        continue

    if source_frame.empty:
        continue

    source_frame["batch7_source_table"] = Path(str(inventory_row["relative_path"])).name
    source_frame["batch7_source_path"] = str(inventory_row["relative_path"])
    source_frame["batch7_inventory_model_guess"] = batch7_clean_text(inventory_row.get("model_guess"))
    source_frame["batch7_inventory_metric_family_guess"] = batch7_clean_text(inventory_row.get("metric_family_guess"))
    batch7_source_frames.append(source_frame)

print(f"Loaded inventory rows: {len(batch7_inventory_df):,}")
print(f"Inventory metric CSV candidates: {len(batch7_metric_source_inventory_df):,}")
print(f"Metric source frames loaded: {len(batch7_source_frames):,}")

if not batch7_source_frames:
    display(batch7_metric_source_inventory_df[["relative_path", "csv_columns"]].head(30))
    raise RuntimeError("No usable metric source frames were loaded from the inventory.")

def batch7_normalize_model(value, source_path=""):
    raw = batch7_clean_text(value).lower()
    source = batch7_clean_text(source_path).lower()
    combined = f"{raw} {source}"

    if "sdxl" in combined:
        return "SDXL"
    if "stable_diffusion" in combined or "stable diffusion" in combined:
        return "Stable Diffusion"
    if "lama" in combined:
        return "LaMa"
    if "opencv" in combined or "telea" in combined:
        return "OpenCV Telea"
    if raw:
        return batch7_clean_text(value)
    return "Unknown"

def batch7_model_family(model_name):
    model = batch7_clean_text(model_name).lower()
    if "stable diffusion" in model or "sdxl" in model:
        return "generative"
    if "opencv" in model or "telea" in model or "lama" in model:
        return "deterministic_or_non_generative"
    return "unknown"

def batch7_metric_family(metric_name):
    metric = batch7_clean_text(metric_name).lower()
    if "lpips" in metric:
        return "lpips"
    if "clip" in metric or "dinov2" in metric or "similarity" in metric:
        return "feature_similarity"
    if any(token in metric for token in ["mse", "mae", "psnr", "ssim"]):
        return "classical"
    if "runtime" in metric or "seconds" in metric:
        return "runtime"
    return "other"

def batch7_metric_direction(metric_name):
    metric = batch7_clean_text(metric_name).lower()
    if "improvement" in metric:
        return "higher"
    if "psnr" in metric or "ssim" in metric or "clip" in metric or "dinov2" in metric or "similarity" in metric:
        return "higher"
    if "mse" in metric or "mae" in metric or "lpips" in metric or "error" in metric:
        return "lower"
    return "higher"

def batch7_pick(row, columns, default=""):
    for column in columns:
        if column in row.index:
            value = batch7_clean_text(row.get(column))
            if value:
                return value
    return default

def batch7_damage_percent(row):
    percent_columns = [
        "damage_percentage", "damage_percent", "mask_area_percent",
        "damage_area_percent", "mask_percentage",
    ]
    fraction_columns = [
        "mask_area_fraction", "damage_area_fraction", "mask_area_ratio",
        "damage_ratio",
    ]

    for column in percent_columns:
        if column in row.index:
            value = pd.to_numeric(row.get(column), errors="coerce")
            if pd.notna(value):
                return float(value)

    for column in fraction_columns:
        if column in row.index:
            value = pd.to_numeric(row.get(column), errors="coerce")
            if pd.notna(value):
                return float(value * 100 if value <= 1 else value)

    return np.nan

def batch7_damage_bin(value):
    value = pd.to_numeric(value, errors="coerce")
    if pd.isna(value):
        return ""
    if value <= 5:
        return "0-5%"
    if value <= 10:
        return "5-10%"
    if value <= 20:
        return "10-20%"
    if value <= 35:
        return "20-35%"
    return "35%+"

metric_column_blocklist = [
    "id", "index", "seed", "rank", "position", "count", "rows", "cases",
    "width", "height", "pixel", "threshold", "version", "year",
    "exists", "passed", "valid", "flag", "priority",
]

metric_rows = []

for source_frame in batch7_source_frames:
    frame = source_frame.copy()
    source_path = batch7_clean_text(frame["batch7_source_path"].iloc[0])
    source_guess = batch7_clean_text(frame.get("batch7_inventory_model_guess", pd.Series([""])).iloc[0])

    model_column = next(
        (column for column in ["model_name", "report_model_name", "restoration_method", "model_slug"] if column in frame.columns),
        None,
    )

    if {"metric_name", "metric_value"}.issubset(frame.columns):
        long_metric_columns = [("metric_value", "metric_name")]
    else:
        long_metric_columns = []

    if long_metric_columns:
        metric_value_columns = ["metric_value"]
    else:
        metric_value_columns = []
        for column in frame.columns:
            lower = str(column).lower()
            if not any(pattern in lower for pattern in metric_name_patterns):
                continue
            if any(blocked in lower for blocked in metric_column_blocklist):
                continue
            values = pd.to_numeric(frame[column], errors="coerce")
            if values.notna().sum() > 0:
                metric_value_columns.append(column)

    for _, row in frame.iterrows():
        raw_model = row.get(model_column, source_guess) if model_column else source_guess
        model_name = batch7_normalize_model(raw_model, source_path)

        case_id = batch7_pick(row, ["case_id", "report_case_id", "source_case_id", "restoration_case_id"])
        if not case_id:
            continue

        context = {
            "source_path": source_path,
            "dataset_name": batch7_pick(row, ["dataset_name", "report_dataset_name"]),
            "case_id": case_id,
            "painting_id": batch7_pick(row, ["painting_id", "report_painting_id"]),
            "category": batch7_pick(row, ["category", "style_or_period"]),
            "style": batch7_pick(row, ["style", "style_or_period"]),
            "mask_type": batch7_pick(row, ["mask_type", "report_mask_type", "metric_mask_type"]),
            "damage_type": batch7_pick(row, ["damage_type", "damage_fill_strategy", "mask_shape"]),
            "degradation_type": batch7_pick(row, ["degradation_type", "synthetic_degradation_type", "generator_name"]),
            "evaluation_region": batch7_pick(row, ["evaluation_region", "rank_evaluation_region"], default="full_or_unspecified"),
            "candidate_id": batch7_pick(row, ["candidate_id", "restoration_case_id"]),
            "restoration_case_id": batch7_pick(row, ["restoration_case_id", "report_case_id"]),
            "model_name": model_name,
            "model_family": batch7_model_family(model_name),
        }

        damage_percent = batch7_damage_percent(row)
        context["damage_percentage"] = damage_percent
        context["damage_percentage_bin"] = batch7_damage_bin(damage_percent)

        for metric_column in metric_value_columns:
            metric_value = pd.to_numeric(row.get(metric_column), errors="coerce")
            if pd.isna(metric_value) or not np.isfinite(metric_value):
                continue

            if metric_column == "metric_value" and "metric_name" in row.index:
                metric_name = batch7_clean_text(row.get("metric_name"))
            else:
                metric_name = str(metric_column)

            metric_rows.append(
                {
                    **context,
                    "metric_name": metric_name,
                    "metric_family": batch7_metric_family(metric_name),
                    "metric_direction": batch7_metric_direction(metric_name),
                    "score": float(metric_value),
                }
            )

batch7_model_score_df = pd.DataFrame(metric_rows)

if batch7_model_score_df.empty:
    raise RuntimeError("Inventory-driven Batch 7 found metric CSVs, but extracted zero finite model scores.")

batch7_model_score_df = batch7_model_score_df.loc[
    batch7_model_score_df["model_name"].ne("Unknown")
].copy()

if batch7_model_score_df["model_name"].nunique() < 2:
    display(batch7_model_score_df[["source_path", "model_name", "metric_name"]].drop_duplicates().head(50))
    raise RuntimeError("Batch 7 needs at least two models with finite scores. Inventory extraction found fewer than two.")

comparison_context_columns = [
    "dataset_name", "case_id", "mask_type", "evaluation_region", "metric_name", "model_name"
]

best_rows = []
for _, group in batch7_model_score_df.groupby(comparison_context_columns, dropna=False):
    direction = group["metric_direction"].iloc[0]
    sorted_group = group.sort_values("score", ascending=(direction == "lower"), kind="mergesort")
    best_row = sorted_group.iloc[0].copy()
    best_row["candidate_rows_collapsed"] = int(len(group))
    best_row["mean_candidate_score"] = float(group["score"].mean())
    best_rows.append(best_row)

batch7_model_case_best_df = pd.DataFrame(best_rows).reset_index(drop=True)

batch7_model_score_df.to_csv(BATCH7_MODEL_SCORE_PATH, index=False)
batch7_model_case_best_df.to_csv(BATCH7_MODEL_CASE_BEST_PATH, index=False)

print(f"Extracted finite model-score rows: {len(batch7_model_score_df):,}")
print(f"Collapsed best model-case-metric rows: {len(batch7_model_case_best_df):,}")
print(f"Models: {sorted(batch7_model_case_best_df['model_name'].unique())}")
print(f"Metric families: {sorted(batch7_model_case_best_df['metric_family'].unique())}")
print(f"Saved long scores: {batch7_rel(BATCH7_MODEL_SCORE_PATH)}")
print(f"Saved best scores: {batch7_rel(BATCH7_MODEL_CASE_BEST_PATH)}")

display(batch7_model_case_best_df.head(20))

Batch 7 project root: D:\Masters\FH\Thesis\painting-restoration-eval
Inventory candidates checked:
 - D:\Masters\FH\Thesis\painting-restoration-eval\outputs\inventory\project_file_inventory.csv FOUND
 - D:\Masters\FH\Thesis\painting-restoration-eval\notebooks\outputs\27_multi_model_comparison\inventory\batch0_project_inventory_snapshot.csv missing
 - D:\Masters\FH\Thesis\painting-restoration-eval\project_file_inventory.csv missing
 - D:\Masters\FH\Thesis\painting-restoration-eval\outputs\21_stable_diffusion_restoration\inventory\batch0_project_inventory_snapshot.csv FOUND
 - D:\Masters\FH\Thesis\painting-restoration-eval\outputs\22_stable_diffusion_classical_metrics\inventory\batch0_project_inventory_snapshot.csv FOUND
 - D:\Masters\FH\Thesis\painting-restoration-eval\outputs\23_stable_diffusion_difference_maps\inventory\batch0_project_inventory_snapshot.csv FOUND
 - D:\Masters\FH\Thesis\painting-restoration-eval\outputs\24_stable_diffusion_lpips_metrics\inventory\batch0_project_invent

,source_path,dataset_name,case_id,painting_id,category,style,mask_type,damage_type,degradation_type,evaluation_region,...,model_name,model_family,damage_percentage,damage_percentage_bin,metric_name,metric_family,metric_direction,score,candidate_rows_collapsed,mean_candidate_score
0,outputs/23_stable_diffusion_difference_maps/me...,,p001__blur__mild,p001,portrait_figure,,blur,,,full_or_unspecified,...,Stable Diffusion,generative,NaN,,boundary_signed_improvement_file_size_bytes,other,higher,3848.000000,1,3848.000000
1,outputs/23_stable_diffusion_difference_maps/me...,,p001__blur__mild,p001,portrait_figure,,blur,,,full_or_unspecified,...,Stable Diffusion,generative,NaN,,boundary_signed_improvement_filename_length,other,higher,17.000000,1,17.000000
2,outputs/23_stable_diffusion_difference_maps/me...,,p001__blur__mild,p001,portrait_figure,,blur,,,full_or_unspecified,...,Stable Diffusion,generative,NaN,,damaged_error_boundary_mean,other,lower,0.185346,1,0.185346
3,outputs/23_stable_diffusion_difference_maps/me...,,p001__blur__mild,p001,portrait_figure,,blur,,,full_or_unspecified,...,Stable Diffusion,generative,NaN,,damaged_error_boundary_std,other,lower,0.533024,1,0.533024
4,outputs/23_stable_diffusion_difference_maps/me...,,p001__blur__mild,p001,portrait_figure,,blur,,,full_or_unspecified,...,Stable Diffusion,generative,NaN,,damaged_error_file_size_bytes,other,lower,18474.000000,1,18474.000000
5,outputs/23_stable_diffusion_difference_maps/me...,,p001__blur__mild,p001,portrait_figure,,blur,,,full_or_unspecified,...,Stable Diffusion,generative,NaN,,damaged_error_filename_length,other,lower,17.000000,1,17.000000
6,outputs/23_stable_diffusion_difference_maps/me...,,p001__blur__mild,p001,portrait_figure,,blur,,,full_or_unspecified,...,Stable Diffusion,generative,NaN,,damaged_error_masked_mean,other,lower,0.234896,1,0.234896
7,outputs/23_stable_diffusion_difference_maps/me...,,p001__blur__mild,p001,portrait_figure,,blur,,,full_or_unspecified,...,Stable Diffusion,generative,NaN,,damaged_error_masked_std,other,lower,0.469754,1,0.469754
8,outputs/23_stable_diffusion_difference_maps/me...,,p001__blur__mild,p001,portrait_figure,,blur,,,full_or_unspecified,...,Stable Diffusion,generative,NaN,,damaged_error_max_full,other,lower,7.333333,1,7.333333
9,outputs/23_stable_diffusion_difference_maps/me...,,p001__blur__mild,p001,portrait_figure,,blur,,,full_or_unspecified,...,Stable Diffusion,generative,NaN,,damaged_error_mean_full,other,lower,0.024262,1,0.024262


In [53]:
# Batch 7 / Replacement Cell 3 - Leaderboards, paired wins, stratified comparisons, plots

BATCH7_MODEL_LEADERBOARD_PATH = ANALYSIS_DIR / "multi_model_model_leaderboard.csv"
BATCH7_PAIRWISE_PATH = ANALYSIS_DIR / "multi_model_pairwise_wins.csv"
BATCH7_BY_METRIC_PATH = ANALYSIS_DIR / "multi_model_by_metric.csv"
BATCH7_BY_FAMILY_PATH = ANALYSIS_DIR / "multi_model_by_metric_family.csv"
BATCH7_BY_CATEGORY_PATH = ANALYSIS_DIR / "multi_model_by_category.csv"
BATCH7_BY_STYLE_PATH = ANALYSIS_DIR / "multi_model_by_style.csv"
BATCH7_BY_MASK_PATH = ANALYSIS_DIR / "multi_model_by_mask_type.csv"
BATCH7_BY_DAMAGE_BIN_PATH = ANALYSIS_DIR / "multi_model_by_damage_percentage_bin.csv"
BATCH7_BY_DEGRADATION_PATH = ANALYSIS_DIR / "multi_model_by_degradation_type.csv"
BATCH7_RUNTIME_PATH = ANALYSIS_DIR / "multi_model_runtime_compute_summary.csv"

ranking_group_columns = ["dataset_name", "case_id", "mask_type", "evaluation_region", "metric_name"]

ranked_groups = []
for _, group in batch7_model_case_best_df.groupby(ranking_group_columns, dropna=False):
    if group["model_name"].nunique() < 2:
        continue

    direction = group["metric_direction"].iloc[0]
    ranked = group.copy()
    ranked["rank"] = ranked["score"].rank(method="min", ascending=(direction == "lower"))
    ranked["is_winner"] = ranked["rank"].eq(ranked["rank"].min())
    ranked["comparable_model_count"] = ranked["model_name"].nunique()
    ranked_groups.append(ranked)

if not ranked_groups:
    raise RuntimeError("No paired model comparisons were possible. Check that case_id/mask/evaluation_region/metric_name align across models.")

batch7_ranked_df = pd.concat(ranked_groups, ignore_index=True, sort=False)
batch7_ranked_df["is_top3"] = batch7_ranked_df["rank"].le(3)

comparison_count = batch7_ranked_df[ranking_group_columns].drop_duplicates().shape[0]

batch7_model_leaderboard_df = (
    batch7_ranked_df
    .groupby(["model_name", "model_family"], dropna=False)
    .agg(
        comparable_rows=("score", "size"),
        unique_comparisons=("case_id", lambda values: len(values)),
        win_count=("is_winner", "sum"),
        top3_count=("is_top3", "sum"),
        mean_rank=("rank", "mean"),
        median_rank=("rank", "median"),
        mean_score=("score", "mean"),
        median_score=("score", "median"),
        collapsed_candidate_rows=("candidate_rows_collapsed", "sum"),
    )
    .reset_index()
)
batch7_model_leaderboard_df["win_rate"] = batch7_model_leaderboard_df["win_count"] / batch7_model_leaderboard_df["comparable_rows"]
batch7_model_leaderboard_df["top3_rate"] = batch7_model_leaderboard_df["top3_count"] / batch7_model_leaderboard_df["comparable_rows"]
batch7_model_leaderboard_df = batch7_model_leaderboard_df.sort_values(
    ["win_rate", "mean_rank", "comparable_rows"],
    ascending=[False, True, False],
    kind="mergesort",
)

def batch7_group_leaderboard(group_columns):
    valid_columns = [column for column in group_columns if column in batch7_ranked_df.columns]
    if not valid_columns:
        return pd.DataFrame()

    frame = batch7_ranked_df.copy()
    for column in valid_columns:
        frame = frame.loc[frame[column].astype(str).str.strip().ne("")].copy()

    if frame.empty:
        return pd.DataFrame()

    output = (
        frame
        .groupby(valid_columns + ["model_name", "model_family"], dropna=False)
        .agg(
            comparable_rows=("score", "size"),
            win_count=("is_winner", "sum"),
            mean_rank=("rank", "mean"),
            median_rank=("rank", "median"),
            mean_score=("score", "mean"),
            median_score=("score", "median"),
        )
        .reset_index()
    )
    output["win_rate"] = output["win_count"] / output["comparable_rows"]
    return output.sort_values(valid_columns + ["win_rate", "mean_rank"], ascending=[True] * len(valid_columns) + [False, True], kind="mergesort")

batch7_by_metric_df = batch7_group_leaderboard(["metric_name", "evaluation_region"])
batch7_by_family_df = batch7_group_leaderboard(["metric_family", "evaluation_region"])
batch7_by_category_df = batch7_group_leaderboard(["category", "metric_family"])
batch7_by_style_df = batch7_group_leaderboard(["style", "metric_family"])
batch7_by_mask_df = batch7_group_leaderboard(["mask_type", "metric_family"])
batch7_by_damage_bin_df = batch7_group_leaderboard(["damage_percentage_bin", "metric_family"])
batch7_by_degradation_df = batch7_group_leaderboard(["degradation_type", "metric_family"])

pairwise_rows = []
for comparison_key, group in batch7_ranked_df.groupby(ranking_group_columns, dropna=False):
    direction = group["metric_direction"].iloc[0]
    rows = list(group.to_dict(orient="records"))

    for left, right in combinations(rows, 2):
        left_model = left["model_name"]
        right_model = right["model_name"]
        left_score = left["score"]
        right_score = right["score"]

        if left_score == right_score:
            winner = "tie"
        elif direction == "lower":
            winner = left_model if left_score < right_score else right_model
        else:
            winner = left_model if left_score > right_score else right_model

        for model_a, model_b in [(left_model, right_model), (right_model, left_model)]:
            if winner == "tie":
                result = "tie"
            elif winner == model_a:
                result = "win"
            else:
                result = "loss"

            pairwise_rows.append(
                {
                    "model_a": model_a,
                    "model_b": model_b,
                    "metric_name": left["metric_name"],
                    "metric_family": left["metric_family"],
                    "evaluation_region": left["evaluation_region"],
                    "result": result,
                }
            )

batch7_pairwise_raw_df = pd.DataFrame(pairwise_rows)

batch7_pairwise_df = (
    batch7_pairwise_raw_df
    .groupby(["model_a", "model_b"], dropna=False)
    .agg(
        paired_rows=("result", "size"),
        wins=("result", lambda values: int((pd.Series(values) == "win").sum())),
        losses=("result", lambda values: int((pd.Series(values) == "loss").sum())),
        ties=("result", lambda values: int((pd.Series(values) == "tie").sum())),
    )
    .reset_index()
)
batch7_pairwise_df["win_rate_excluding_ties"] = batch7_pairwise_df["wins"] / (batch7_pairwise_df["wins"] + batch7_pairwise_df["losses"]).replace(0, np.nan)
batch7_pairwise_df["net_wins"] = batch7_pairwise_df["wins"] - batch7_pairwise_df["losses"]
batch7_pairwise_df = batch7_pairwise_df.sort_values(["net_wins", "wins"], ascending=[False, False], kind="mergesort")

runtime_columns = [
    column for column in batch7_model_score_df.columns
    if "runtime" in str(column).lower() or "seconds" in str(column).lower()
]

runtime_source_rows = []
for source_frame in batch7_source_frames:
    frame = source_frame.copy()
    source_path = batch7_clean_text(frame["batch7_source_path"].iloc[0])
    runtime_value_columns = [
        column for column in frame.columns
        if ("runtime" in str(column).lower() or "seconds" in str(column).lower())
        and pd.to_numeric(frame[column], errors="coerce").notna().sum() > 0
    ]
    model_column = next(
        (column for column in ["model_name", "report_model_name", "restoration_method", "model_slug"] if column in frame.columns),
        None,
    )

    for runtime_column in runtime_value_columns:
        for _, row in frame.iterrows():
            value = pd.to_numeric(row.get(runtime_column), errors="coerce")
            if pd.isna(value) or not np.isfinite(value):
                continue
            raw_model = row.get(model_column, "") if model_column else ""
            runtime_source_rows.append(
                {
                    "model_name": batch7_normalize_model(raw_model, source_path),
                    "runtime_column": runtime_column,
                    "runtime_seconds": float(value),
                    "source_path": source_path,
                }
            )

batch7_runtime_raw_df = pd.DataFrame(runtime_source_rows)
if batch7_runtime_raw_df.empty:
    batch7_runtime_df = pd.DataFrame(columns=["model_name", "runtime_rows", "mean_runtime_seconds", "median_runtime_seconds", "total_runtime_seconds"])
else:
    batch7_runtime_df = (
        batch7_runtime_raw_df
        .groupby("model_name", dropna=False)
        .agg(
            runtime_rows=("runtime_seconds", "size"),
            mean_runtime_seconds=("runtime_seconds", "mean"),
            median_runtime_seconds=("runtime_seconds", "median"),
            total_runtime_seconds=("runtime_seconds", "sum"),
        )
        .reset_index()
        .sort_values("median_runtime_seconds", kind="mergesort")
    )

batch7_model_leaderboard_df.to_csv(BATCH7_MODEL_LEADERBOARD_PATH, index=False)
batch7_pairwise_df.to_csv(BATCH7_PAIRWISE_PATH, index=False)
batch7_by_metric_df.to_csv(BATCH7_BY_METRIC_PATH, index=False)
batch7_by_family_df.to_csv(BATCH7_BY_FAMILY_PATH, index=False)
batch7_by_category_df.to_csv(BATCH7_BY_CATEGORY_PATH, index=False)
batch7_by_style_df.to_csv(BATCH7_BY_STYLE_PATH, index=False)
batch7_by_mask_df.to_csv(BATCH7_BY_MASK_PATH, index=False)
batch7_by_damage_bin_df.to_csv(BATCH7_BY_DAMAGE_BIN_PATH, index=False)
batch7_by_degradation_df.to_csv(BATCH7_BY_DEGRADATION_PATH, index=False)
batch7_runtime_df.to_csv(BATCH7_RUNTIME_PATH, index=False)

batch7_plot_paths = []

def batch7_save_plot(path, title):
    fig = plt.gcf()
    fig.tight_layout()
    fig.savefig(path, dpi=180)
    plt.close(fig)
    batch7_plot_paths.append(path)

if not batch7_model_leaderboard_df.empty:
    plot_df = batch7_model_leaderboard_df.sort_values("win_rate", ascending=True)
    plt.figure(figsize=(9, 5))
    plt.barh(plot_df["model_name"], plot_df["win_rate"])
    plt.xlabel("Win rate across paired metric rows")
    plt.title("Overall Model Win Rate")
    plt.grid(axis="x", alpha=0.25)
    batch7_save_plot(BATCH7_PLOTS_DIR / "overall_model_win_rate.png", "Overall Model Win Rate")

    plot_df = batch7_model_leaderboard_df.sort_values("mean_rank", ascending=False)
    plt.figure(figsize=(9, 5))
    plt.barh(plot_df["model_name"], plot_df["mean_rank"])
    plt.xlabel("Mean rank; lower is better")
    plt.title("Overall Mean Rank by Model")
    plt.grid(axis="x", alpha=0.25)
    batch7_save_plot(BATCH7_PLOTS_DIR / "overall_model_mean_rank.png", "Overall Mean Rank by Model")

if not batch7_by_family_df.empty:
    pivot = batch7_by_family_df.pivot_table(index="metric_family", columns="model_name", values="win_rate", aggfunc="mean")
    plt.figure(figsize=(max(8, 1.4 * len(pivot.columns)), max(4.5, 0.6 * len(pivot.index))))
    plt.imshow(pivot.fillna(0).values, aspect="auto")
    plt.xticks(range(len(pivot.columns)), pivot.columns, rotation=35, ha="right")
    plt.yticks(range(len(pivot.index)), pivot.index)
    plt.colorbar(label="Win rate")
    plt.title("Win Rate by Metric Family")
    for row_index in range(pivot.shape[0]):
        for col_index in range(pivot.shape[1]):
            value = pivot.iloc[row_index, col_index]
            label = "" if pd.isna(value) else f"{value:.2f}"
            plt.text(col_index, row_index, label, ha="center", va="center", color="white")
    batch7_save_plot(BATCH7_PLOTS_DIR / "win_rate_by_metric_family.png", "Win Rate by Metric Family")

if not batch7_pairwise_df.empty:
    pivot = batch7_pairwise_df.pivot_table(index="model_a", columns="model_b", values="net_wins", aggfunc="sum")
    plt.figure(figsize=(max(7, 1.4 * len(pivot.columns)), max(5, 1.0 * len(pivot.index))))
    plt.imshow(pivot.fillna(0).values, aspect="auto")
    plt.xticks(range(len(pivot.columns)), pivot.columns, rotation=35, ha="right")
    plt.yticks(range(len(pivot.index)), pivot.index)
    plt.colorbar(label="Net wins")
    plt.title("Pairwise Net Wins: Row Model Beats Column Model")
    for row_index in range(pivot.shape[0]):
        for col_index in range(pivot.shape[1]):
            value = pivot.iloc[row_index, col_index]
            label = "" if pd.isna(value) else f"{int(value)}"
            plt.text(col_index, row_index, label, ha="center", va="center", color="white")
    batch7_save_plot(BATCH7_PLOTS_DIR / "pairwise_net_wins_heatmap.png", "Pairwise Net Wins")

if not batch7_runtime_df.empty:
    plot_df = batch7_runtime_df.sort_values("median_runtime_seconds", ascending=True)
    plt.figure(figsize=(9, 5))
    plt.barh(plot_df["model_name"], plot_df["median_runtime_seconds"])
    plt.xlabel("Median runtime seconds")
    plt.title("Runtime / Compute Comparison")
    plt.grid(axis="x", alpha=0.25)
    batch7_save_plot(BATCH7_PLOTS_DIR / "runtime_compute_comparison.png", "Runtime / Compute Comparison")

if not batch7_by_damage_bin_df.empty:
    pivot = batch7_by_damage_bin_df.pivot_table(index="damage_percentage_bin", columns="model_name", values="win_rate", aggfunc="mean")
    pivot = pivot.reindex(["0-5%", "5-10%", "10-20%", "20-35%", "35%+"]).dropna(how="all")
    if not pivot.empty:
        pivot.plot(kind="bar", figsize=(11, 5))
        plt.ylabel("Win rate")
        plt.title("Win Rate by Damage Percentage Bin")
        plt.grid(axis="y", alpha=0.25)
        batch7_save_plot(BATCH7_PLOTS_DIR / "win_rate_by_damage_percentage_bin.png", "Win Rate by Damage Percentage Bin")

print(f"Paired comparison groups: {comparison_count:,}")
print(f"Rendered Batch 7 plots: {len(batch7_plot_paths):,}")
print(f"Saved leaderboard: {batch7_rel(BATCH7_MODEL_LEADERBOARD_PATH)}")
display(batch7_model_leaderboard_df)
display(batch7_pairwise_df.head(30))

Paired comparison groups: 35,941
Rendered Batch 7 plots: 5
Saved leaderboard: outputs/27_multi_model_comparison/analysis/multi_model_model_leaderboard.csv


,model_name,model_family,comparable_rows,unique_comparisons,win_count,top3_count,mean_rank,median_rank,mean_score,median_score,collapsed_candidate_rows,win_rate,top3_rate
0,LaMa,deterministic_or_non_generative,35941,35941,25409,35941,1.293036,1.0,893.897329,0.992929,49306,0.706964,1.0
1,OpenCV Telea,deterministic_or_non_generative,35941,35941,20085,35941,1.441167,1.0,20.168813,0.966649,35941,0.558833,1.0


,model_a,model_b,paired_rows,wins,losses,ties,win_rate_excluding_ties,net_wins
0,LaMa,OpenCV Telea,35941,15856,10532,9553,0.600879,5324
1,OpenCV Telea,LaMa,35941,10532,15856,9553,0.399121,-5324


In [54]:
# Batch 7 / Replacement Cell 4 - Final HTML report, manifest, validation

def batch7_df_html(df, max_rows=BATCH7_MAX_TABLE_ROWS):
    if df is None or df.empty:
        return "<p class='muted'>No rows available.</p>"
    table = df.head(max_rows).to_html(index=False, escape=True, classes="data-table")
    if len(df) > max_rows:
        table += f"<p class='muted'>Showing {max_rows:,} of {len(df):,} rows.</p>"
    return table

def batch7_img_html(path, caption):
    path = Path(path)
    if not path.exists():
        return ""
    return f"""
    <figure>
      <img src="{html.escape(batch7_html_rel(path))}" alt="{html.escape(caption)}">
      <figcaption>{html.escape(caption)}</figcaption>
    </figure>
    """

batch7_plot_html = "\n".join(
    batch7_img_html(path, Path(path).stem.replace("_", " ").title())
    for path in batch7_plot_paths
)

batch7_panel_paths = []
if BATCH6_FIGURE_MANIFEST_PATH.exists():
    panel_manifest = pd.read_csv(BATCH6_FIGURE_MANIFEST_PATH)
    for column in ["figure_path", "path", "path_project_relative"]:
        if column in panel_manifest.columns:
            for value in panel_manifest[column].dropna().head(BATCH7_MAX_REPORT_IMAGES):
                path = batch7_existing_path(value)
                if path is not None and path.suffix.lower() in [".png", ".jpg", ".jpeg", ".webp"]:
                    batch7_panel_paths.append(path)

batch7_panel_html = "\n".join(
    batch7_img_html(path, Path(path).stem.replace("_", " ").title())
    for path in batch7_panel_paths[:BATCH7_MAX_REPORT_IMAGES]
)

best_model = batch7_model_leaderboard_df.iloc[0]["model_name"] if not batch7_model_leaderboard_df.empty else "No model"
best_win_rate = batch7_model_leaderboard_df.iloc[0]["win_rate"] if not batch7_model_leaderboard_df.empty else np.nan

cross_family_df = (
    batch7_ranked_df
    .groupby(ranking_group_columns, dropna=False)
    .agg(model_families=("model_family", lambda values: sorted(set(values))))
    .reset_index()
)
cross_family_rows = int(cross_family_df["model_families"].apply(lambda values: "generative" in values and "deterministic_or_non_generative" in values).sum())

batch7_html = f"""<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>{html.escape(NOTEBOOK_TITLE)} - Multi-Model Comparison Report</title>
<style>
body {{ font-family: Arial, Helvetica, sans-serif; margin: 0; color: #1f2933; line-height: 1.45; }}
main {{ max-width: 1320px; margin: 0 auto; padding: 32px 28px 60px; }}
h1 {{ font-size: 30px; margin-bottom: 4px; }}
h2 {{ font-size: 22px; margin-top: 34px; padding-top: 18px; border-top: 1px solid #d0d5dd; }}
h3 {{ font-size: 17px; margin-top: 24px; }}
p, li {{ font-size: 14px; }}
.muted {{ color: #667085; }}
.callout {{ background: #f8fafc; border: 1px solid #d0d5dd; border-left: 5px solid #1d4ed8; padding: 14px 16px; margin: 18px 0; }}
.grid {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(230px, 1fr)); gap: 12px; margin: 18px 0; }}
.metric-card {{ border: 1px solid #d0d5dd; background: #f8fafc; padding: 12px; }}
.metric-card .label {{ color: #667085; font-size: 12px; }}
.metric-card .value {{ font-size: 22px; font-weight: 700; margin-top: 4px; }}
.data-table {{ border-collapse: collapse; width: 100%; font-size: 12px; margin: 12px 0 20px; }}
.data-table th, .data-table td {{ border: 1px solid #d0d5dd; padding: 6px 8px; vertical-align: top; }}
.data-table th {{ background: #eef2f7; text-align: left; }}
.figure-grid {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(380px, 1fr)); gap: 18px; }}
figure {{ margin: 0; border: 1px solid #d0d5dd; background: #fff; padding: 10px; }}
figure img {{ width: 100%; height: auto; display: block; }}
figcaption {{ color: #667085; font-size: 12px; margin-top: 8px; }}
code {{ background: #eef2f7; padding: 1px 4px; }}
</style>
</head>
<body>
<main>
<h1>{html.escape(NOTEBOOK_TITLE)} - Multi-Model Comparison Report</h1>
<p class="muted">Generated at {html.escape(batch7_utc_now_iso())}. Notebook ID: <code>{html.escape(NOTEBOOK_ID)}</code>.</p>

<div class="callout">
<strong>Main result:</strong>
The best overall model by paired metric-row win rate is <strong>{html.escape(str(best_model))}</strong>
with win rate <strong>{best_win_rate:.3f}</strong>.
This is not a single universal winner claim; it is a paired, metric-aware summary across cases, regions, and metric families.
</div>

<h2>Executive Summary</h2>
<ul>
<li>Inventory-discovered finite score rows: {len(batch7_model_score_df):,}</li>
<li>Best-score rows after collapsing repeated candidates per model/case/metric: {len(batch7_model_case_best_df):,}</li>
<li>Paired comparison rows used for ranking: {len(batch7_ranked_df):,}</li>
<li>Unique paired comparison groups: {comparison_count:,}</li>
<li>Cross-family deterministic/non-generative vs generative comparable groups: {cross_family_rows:,}</li>
<li>Rendered report plots: {len(batch7_plot_paths):,}</li>
<li>Rendered restoration/example panels found from Batch 6 manifest: {len(batch7_panel_paths):,}</li>
</ul>

<h2>What This Report Actually Compares</h2>
<p>
The report uses the project inventory to discover case-level metric CSVs from previous notebooks,
extracts finite model scores, normalizes model names, infers metric direction, and ranks models only
inside paired comparison groups where at least two models have evidence for the same case, mask type,
evaluation region, and metric.
</p>
<p>
When Stable Diffusion or SDXL have multiple candidates for the same case and metric, this report collapses
those rows to the best available candidate for that model/case/metric and records the number of collapsed
candidate rows. That makes the comparison explicit instead of silently mixing candidate-level and model-level rows.
</p>

<h2>Report Plots</h2>
<div class="figure-grid">
{batch7_plot_html}
</div>

<h2>Overall Model Leaderboard</h2>
<p>This is the core outperformance table. Higher win rate and lower mean rank are better.</p>
{batch7_df_html(batch7_model_leaderboard_df)}

<h2>Pairwise Model Wins</h2>
<p>Each row asks whether model A beats model B on paired metric rows. Net wins above zero means model A beats model B more often than it loses.</p>
{batch7_df_html(batch7_pairwise_df)}

<h2>Comparison by Metric Family</h2>
{batch7_df_html(batch7_by_family_df)}

<h2>Comparison by Metric and Region</h2>
{batch7_df_html(batch7_by_metric_df)}

<h2>Comparison by Style / Category</h2>
<h3>Category</h3>
{batch7_df_html(batch7_by_category_df)}
<h3>Style</h3>
{batch7_df_html(batch7_by_style_df)}

<h2>Comparison by Damage Type and Damage Percentage</h2>
<h3>Mask / Damage Type</h3>
{batch7_df_html(batch7_by_mask_df)}
<h3>Damage Percentage Bin</h3>
{batch7_df_html(batch7_by_damage_bin_df)}

<h2>Comparison by Degradation Type</h2>
{batch7_df_html(batch7_by_degradation_df)}

<h2>Runtime and Compute Comparison</h2>
{batch7_df_html(batch7_runtime_df)}

<h2>Metric Disagreement and Ranking Stability</h2>
<p>
The previous disagreement/stability outputs remain relevant, but they are now supporting evidence rather than the whole report.
They should be read alongside the leaderboard and pairwise tables above.
</p>
{batch7_df_html(batch7_disagreement_stability_df if "batch7_disagreement_stability_df" in globals() else pd.DataFrame())}

<h2>Restoration Examples and Panels</h2>
<p>
These are pulled from the Batch 6 figure manifest when available. If this section is empty, Batch 6 did not expose usable panel paths.
</p>
<div class="figure-grid">
{batch7_panel_html}
</div>

<h2>Artifact Index</h2>
<ul>
<li>Model score long CSV: <code>{html.escape(batch7_rel(BATCH7_MODEL_SCORE_PATH))}</code></li>
<li>Best model-case-metric CSV: <code>{html.escape(batch7_rel(BATCH7_MODEL_CASE_BEST_PATH))}</code></li>
<li>Overall model leaderboard CSV: <code>{html.escape(batch7_rel(BATCH7_MODEL_LEADERBOARD_PATH))}</code></li>
<li>Pairwise wins CSV: <code>{html.escape(batch7_rel(BATCH7_PAIRWISE_PATH))}</code></li>
<li>Metric-family comparison CSV: <code>{html.escape(batch7_rel(BATCH7_BY_FAMILY_PATH))}</code></li>
<li>Runtime comparison CSV: <code>{html.escape(batch7_rel(BATCH7_RUNTIME_PATH))}</code></li>
<li>HTML report: <code>{html.escape(batch7_rel(BATCH7_HTML_REPORT_PATH))}</code></li>
<li>Artifact manifest JSON: <code>{html.escape(batch7_rel(BATCH7_ARTIFACT_MANIFEST_PATH))}</code></li>
<li>Final validation CSV: <code>{html.escape(batch7_rel(BATCH7_FINAL_VALIDATION_PATH))}</code></li>
</ul>
</main>
</body>
</html>
"""

BATCH7_HTML_REPORT_PATH.write_text(batch7_html, encoding="utf-8")

batch7_artifact_manifest = {
    "notebook_id": NOTEBOOK_ID,
    "notebook_title": NOTEBOOK_TITLE,
    "stage": "batch7_inventory_driven_multi_model_comparison_report",
    "created_at_utc": batch7_utc_now_iso(),
    "inventory_sources": [str(path) for path in BATCH7_INVENTORY_CANDIDATES if path.exists()],
    "outputs": {
        "html_report": batch7_rel(BATCH7_HTML_REPORT_PATH),
        "artifact_manifest_json": batch7_rel(BATCH7_ARTIFACT_MANIFEST_PATH),
        "final_validation_csv": batch7_rel(BATCH7_FINAL_VALIDATION_PATH),
        "model_score_long_csv": batch7_rel(BATCH7_MODEL_SCORE_PATH),
        "model_case_best_csv": batch7_rel(BATCH7_MODEL_CASE_BEST_PATH),
        "model_leaderboard_csv": batch7_rel(BATCH7_MODEL_LEADERBOARD_PATH),
        "pairwise_wins_csv": batch7_rel(BATCH7_PAIRWISE_PATH),
        "by_metric_csv": batch7_rel(BATCH7_BY_METRIC_PATH),
        "by_metric_family_csv": batch7_rel(BATCH7_BY_FAMILY_PATH),
        "by_category_csv": batch7_rel(BATCH7_BY_CATEGORY_PATH),
        "by_style_csv": batch7_rel(BATCH7_BY_STYLE_PATH),
        "by_mask_type_csv": batch7_rel(BATCH7_BY_MASK_PATH),
        "by_damage_percentage_bin_csv": batch7_rel(BATCH7_BY_DAMAGE_BIN_PATH),
        "by_degradation_type_csv": batch7_rel(BATCH7_BY_DEGRADATION_PATH),
        "runtime_compute_csv": batch7_rel(BATCH7_RUNTIME_PATH),
        "plots": [batch7_rel(path) for path in batch7_plot_paths],
    },
    "summary": {
        "finite_score_rows": int(len(batch7_model_score_df)),
        "best_score_rows": int(len(batch7_model_case_best_df)),
        "paired_ranked_rows": int(len(batch7_ranked_df)),
        "unique_comparison_groups": int(comparison_count),
        "model_count": int(batch7_model_case_best_df["model_name"].nunique()),
        "metric_family_count": int(batch7_model_case_best_df["metric_family"].nunique()),
        "cross_family_comparable_groups": int(cross_family_rows),
        "plot_count": int(len(batch7_plot_paths)),
        "panel_count": int(len(batch7_panel_paths)),
        "best_overall_model": str(best_model),
        "best_overall_win_rate": None if pd.isna(best_win_rate) else float(best_win_rate),
    },
    "claim_boundary": (
        "This report supports paired, metric-aware comparison claims. It does not support a universal model winner "
        "without qualification by metric family, case type, region, and available evidence."
    ),
}

batch7_write_json(BATCH7_ARTIFACT_MANIFEST_PATH, batch7_artifact_manifest)

batch7_html_text = BATCH7_HTML_REPORT_PATH.read_text(encoding="utf-8")

batch7_final_validation_rows = [
    batch7_validation_row("inventory_loaded", len(batch7_inventory_df), "> 0", len(batch7_inventory_df) > 0, "Inventory was not loaded."),
    batch7_validation_row("metric_sources_loaded", len(batch7_source_frames), "> 0", len(batch7_source_frames) > 0, "No metric source frames loaded from inventory."),
    batch7_validation_row("finite_scores_extracted", len(batch7_model_score_df), "> 0", len(batch7_model_score_df) > 0, "No finite model scores extracted."),
    batch7_validation_row("multiple_models_present", batch7_model_case_best_df["model_name"].nunique(), ">= 2", batch7_model_case_best_df["model_name"].nunique() >= 2, "Fewer than two models found."),
    batch7_validation_row("paired_comparisons_present", comparison_count, "> 0", comparison_count > 0, "No paired model comparisons produced."),
    batch7_validation_row("leaderboard_present", len(batch7_model_leaderboard_df), "> 0", len(batch7_model_leaderboard_df) > 0, "Model leaderboard is empty."),
    batch7_validation_row("pairwise_table_present", len(batch7_pairwise_df), "> 0", len(batch7_pairwise_df) > 0, "Pairwise model table is empty."),
    batch7_validation_row("plots_rendered", len(batch7_plot_paths), "> 0", len(batch7_plot_paths) > 0, "No Batch 7 plots rendered."),
    batch7_validation_row("html_report_exists", BATCH7_HTML_REPORT_PATH.exists(), True, BATCH7_HTML_REPORT_PATH.exists(), "HTML report missing."),
    batch7_validation_row("html_contains_model_leaderboard", "Overall Model Leaderboard" in batch7_html_text, True, "Overall Model Leaderboard" in batch7_html_text, "HTML missing leaderboard section."),
    batch7_validation_row("html_contains_pairwise_wins", "Pairwise Model Wins" in batch7_html_text, True, "Pairwise Model Wins" in batch7_html_text, "HTML missing pairwise wins section."),
    batch7_validation_row("html_contains_runtime", "Runtime and Compute Comparison" in batch7_html_text, True, "Runtime and Compute Comparison" in batch7_html_text, "HTML missing runtime/compute section."),
]

batch7_final_validation_df = pd.DataFrame(batch7_final_validation_rows)
batch7_passed = bool(batch7_final_validation_df["passed"].all())
batch7_final_validation_df.to_csv(BATCH7_FINAL_VALIDATION_PATH, index=False)

stage_manifest = batch7_read_json_if_exists(STAGE_MANIFEST_PATH)
stage_manifest.update(
    {
        "notebook_id": NOTEBOOK_ID,
        "notebook_title": NOTEBOOK_TITLE,
        "stage": "batch7_inventory_driven_multi_model_comparison_report",
        "stage_status": "passed" if batch7_passed else "failed",
        "updated_at_utc": batch7_utc_now_iso(),
        "outputs": {
            **stage_manifest.get("outputs", {}),
            "multi_model_comparison_html_report": batch7_rel(BATCH7_HTML_REPORT_PATH),
            "multi_model_comparison_manifest_json": batch7_rel(BATCH7_ARTIFACT_MANIFEST_PATH),
            "multi_model_final_validation_csv": batch7_rel(BATCH7_FINAL_VALIDATION_PATH),
            "multi_model_leaderboard_csv": batch7_rel(BATCH7_MODEL_LEADERBOARD_PATH),
            "multi_model_pairwise_wins_csv": batch7_rel(BATCH7_PAIRWISE_PATH),
        },
        "batch7": batch7_artifact_manifest["summary"],
    }
)
batch7_write_json(STAGE_MANIFEST_PATH, stage_manifest)

print(f"Saved HTML report: {batch7_rel(BATCH7_HTML_REPORT_PATH)}")
print(f"Saved artifact manifest: {batch7_rel(BATCH7_ARTIFACT_MANIFEST_PATH)}")
print(f"Saved final validation: {batch7_rel(BATCH7_FINAL_VALIDATION_PATH)}")
print(f"Batch 7 checks passed: {int(batch7_final_validation_df['passed'].sum())} / {len(batch7_final_validation_df)}")
display(batch7_final_validation_df)

if not batch7_passed:
    display(batch7_final_validation_df.loc[~batch7_final_validation_df["passed"], ["check_name", "actual", "expected", "failure_message"]])
    raise RuntimeError("Batch 7 final validation failed. The report is not final yet.")

print("Batch 7 passed. Inventory-driven model comparison report is complete.")

Saved HTML report: outputs/27_multi_model_comparison/reports/multi_model_comparison_report.html
Saved artifact manifest: outputs/27_multi_model_comparison/manifests/multi_model_comparison_manifest.json
Saved final validation: outputs/27_multi_model_comparison/validation/multi_model_final_validation.csv
Batch 7 checks passed: 12 / 12


,check_name,actual,expected,passed,failure_message
0,inventory_loaded,10947,> 0,True,
1,metric_sources_loaded,30,> 0,True,
2,finite_scores_extracted,328298,> 0,True,
3,multiple_models_present,3,>= 2,True,
4,paired_comparisons_present,35941,> 0,True,
5,leaderboard_present,2,> 0,True,
6,pairwise_table_present,2,> 0,True,
7,plots_rendered,5,> 0,True,
8,html_report_exists,True,True,True,
9,html_contains_model_leaderboard,True,True,True,


Batch 7 passed. Inventory-driven model comparison report is complete.
